# AI–Quantum Portfolio Optimization — Standalone Complete System

Notebook này chứa **toàn bộ mã nguồn hệ thống trong các cell**. Không `git clone`, không tải source từ GitHub/Drive và không phụ thuộc repository bên ngoài. Người dùng chỉ upload một CSV hoặc ZIP dữ liệu.

Pipeline nghiên cứu:

`Upload → schema/hash audit → point-in-time panel → feature engineering → XGBoost/EWMA → Adaptive Universe Reduction → cardinality QUBO → Exact/SA/Penalty-QAOA/XY-QAOA → constrained classical weights → walk-forward backtest → bootstrap/Holm tests → report`

EWMA là bộ ước lượng mean–covariance đa biến và tín hiệu đối chứng, không phải AI. XY-QAOA sử dụng Dicke state và ideal statevector simulator nội bộ; runtime simulator không chứng minh quantum speedup hay quantum advantage. Nếu gói dữ liệu không có đầy đủ disclosure binaries, PIT financial statements và lịch sử thành viên HOSE hoàn chỉnh, kết quả phải được diễn giải là **exploratory**.

## 1. Cấu hình tập trung

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile

EXECUTION_PROFILE = "FULL"  # Đổi thành "SMOKE" để kiểm tra 4 folds.
EXPECTED_CSV_SHA256 = "aea9644cfafc359ed04546deca62fea83509826864463669b219f370f1433eba"
STANDALONE_ROOT = Path("/content/ai_quantum_standalone")
WORKSPACE = STANDALONE_ROOT / "outputs" / "uploaded_research_data"
RESULTS_ROOT = Path("/content/ai_quantum_results")
VENV_DIR = Path("/content/ai_quantum_venv")
NOTEBOOK_CONFIG = {
    "random_seed": 42, "start_date": "2020-01-01", "end_date": "2025-12-31",
    "training_months": 24, "validation_months": 3, "testing_months": 1,
    "rebalance_frequency": "monthly", "maximum_universe_size": 300,
    "candidate_count": 8, "cardinality": 4, "minimum_history": 40,
    "maximum_weight": 0.40, "minimum_weight": 0.05, "risk_aversion": 1.25,
    "transaction_cost_bps": 10, "slippage_bps": 5, "turnover_penalty": 0.01,
    "qaoa_depth": 2, "shots": 1024, "optimizer_iterations": 45,
    "simulated_annealing_iterations": 800, "bootstrap_iterations": 500,
    "significance_level": 0.05, "execution_profile": EXECUTION_PROFILE,
}
assert EXECUTION_PROFILE in {"SMOKE", "FULL"}
print(json.dumps(NOTEBOOK_CONFIG, indent=2, ensure_ascii=False))

## 2. Tạo package tạm và nhúng toàn bộ source code

In [ ]:
if STANDALONE_ROOT.exists():
    shutil.rmtree(STANDALONE_ROOT)
for directory in [
    STANDALONE_ROOT / "src", STANDALONE_ROOT / "scripts",
    STANDALONE_ROOT / "configs", STANDALONE_ROOT / "tests",
    STANDALONE_ROOT / "ai_quantum_system", RESULTS_ROOT,
]:
    directory.mkdir(parents=True, exist_ok=True)
print("Standalone root:", STANDALONE_ROOT)

In [ ]:
%%writefile /content/ai_quantum_standalone/pyproject.toml
[project]
name = "ai-quantum-standalone-colab"
version = "1.0.0"
description = "Self-contained AI-Quantum portfolio research pipeline for Google Colab"
requires-python = ">=3.11"
dependencies = [
  "numpy==2.2.6", "pandas==2.3.1", "pyarrow==24.0.0",
  "scipy==1.16.1", "scikit-learn==1.8.0", "xgboost==3.3.0",
  "matplotlib==3.10.5", "PyYAML==6.0.2", "Pillow==11.3.0"
]

[project.optional-dependencies]
dev = ["pytest==9.1.1"]

[tool.pytest.ini_options]
testpaths = ["tests"]


In [ ]:
%%writefile /content/ai_quantum_standalone/src/__init__.py
"""AI-Quantum standalone research package embedded in the notebook."""
__version__ = "1.0.0"


In [ ]:
%%writefile /content/ai_quantum_standalone/src/cli.py
from __future__ import annotations

import argparse
import json
import os
import shutil
import sys
import tempfile
from pathlib import Path

import pandas as pd

from .data_pipeline import (
    Paths, apply_price_adjustment_contract, build_complete_case_workspace, build_universe,
    generate_fixture, import_csv, leakage_audit, quarantine_fixture_auxiliary, validate_data,
    sha256_file,
)
from .research import ResearchRunBlocked, build_features, load_config, run_experiment
from .sources import (
    audit_available_data_sources,
    crawl_historical_hose_price_gaps,
    crawl_ssi_stage1,
    crawl_hose_official_security_master,
    crawl_vietstock_stage1,
    crawl_fdr_hose,
    crawl_trading_economics_crosscheck,
    crawl_vnstock_hose,
    import_point_in_time_table,
    merge_historical_hose_checkpoints,
    merge_hose_checkpoints,
    crawl_world_bank_vietnam_snapshot,
    crawl_cafef_standalone_workspace,
)
from .corporate_actions import crawl_corporate_actions
from .price_adjustment import (
    build_price_adjustment_v2, build_return_only_adjustment_counterfactual,
    prepare_research_v2_runtime,
)
from .universe_pit import build_historical_universe_pit
from .data_17_8 import (
    audit_data_17_8,
    build_financial_statement_facts,
    build_historical_sector_pit,
    crawl_current_hose_sector_reference,
    crawl_data_17_8_cafef_price_crosscheck,
    crawl_data_17_8_prices,
    crawl_hose_disclosure_index,
    crawl_hose_documents,
    crawl_hose_tri_benchmark,
    crawl_vietstock_company_documents,
    data_17_8_workspace,
    extract_company_document_archives,
    extract_company_document_text,
    finalize_data_17_8_research_universe,
    initialize_data_17_8,
    stage_data_17_8_corporate_actions,
    write_data_17_8_source_report,
)


ROOT = Path(__file__).resolve().parents[1]


def configure_utf8_console() -> None:
    """Make Vietnamese output reliable on Windows and redirected shells."""
    for stream in (sys.stdout, sys.stderr):
        reconfigure = getattr(stream, "reconfigure", None)
        if reconfigure is not None:
            reconfigure(encoding="utf-8", errors="replace")


def parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(description="Point-in-time AI–Quantum portfolio pipeline")
    sub = p.add_subparsers(dest="command", required=True)
    crawl = sub.add_parser("crawl", help="Collect/import raw data through an explicit source adapter")
    crawl.add_argument("--stage", type=int, default=1, choices=[1, 2, 3])
    crawl.add_argument(
        "--source",
        choices=["fixture", "csv", "ssi", "vietstock", "vnstock", "fdr", "tradingeconomics"],
        required=True,
    )
    crawl.add_argument("--from", dest="start", default="2020-01-01")
    crawl.add_argument("--to", dest="end", default="2025-12-31")
    crawl.add_argument("--tickers", default="AAA,BBB,CCC,DDD,EEE,FFF,GGG,HHH")
    crawl.add_argument("--input", type=Path)
    crawl.add_argument("--source-name", default="user_authorized_csv")
    crawl.add_argument("--source-url", default="local://user-authorized")
    crawl.add_argument("--dry-run", action="store_true")
    crawl.add_argument(
        "--max-tickers", type=int, default=300,
        help="Maximum current HOSE equities for --source vnstock when --tickers=auto.",
    )
    crawl.add_argument(
        "--insecure",
        action="store_true",
        help="Disable TLS verification only for a diagnosed local certificate-chain problem.",
    )
    pit = sub.add_parser("import-pit-table", help="Import a point-in-time Stage 1/2/3 table")
    pit.add_argument("--table", required=True, choices=[
        "index_membership", "corporate_actions", "financial_statements", "macro",
        "foreign_flow", "benchmark", "security_master"
    ])
    pit.add_argument("--input", type=Path, required=True)
    adjustment = sub.add_parser(
        "apply-adjustment-contract",
        help="Certify the exact normalized price panel using a hash-bound JSON contract",
    )
    adjustment.add_argument("--input", type=Path, required=True)
    sub.add_parser("normalize", help="Normalization is performed by the selected adapter")
    merge = sub.add_parser(
        "merge-market-sources",
        help="Merge FinanceDataReader primary and vnstock fallback checkpoints.",
    )
    merge.add_argument("--target-tickers", type=int, default=300)
    val = sub.add_parser("validate", help="Validate schemas and data quality")
    val.add_argument("--stage", type=int, default=1)
    uni = sub.add_parser("build-universe", help="Build point-in-time rebalance universe")
    uni.add_argument("--rebalance", default="monthly")
    uni.add_argument(
        "--definition", choices=["hose_all_listed", "index_membership"],
        default="hose_all_listed",
    )
    uni.add_argument("--index-code")
    uni.add_argument("--max-assets", type=int)
    uni.add_argument("--liquidity-lookback-days", type=int, default=60)
    uni.add_argument("--minimum-observations", type=int, default=40)
    sub.add_parser("report-coverage", help="Generate coverage report")
    sub.add_parser("leakage-audit", help="Audit point-in-time contracts")
    feat = sub.add_parser("build-features", help="Build leakage-aware features")
    feat.add_argument("--stage", type=int, default=1)
    for name in ("make-folds", "train-ranker", "build-instances", "run-solvers",
                 "optimize-weights", "backtest", "evaluate"):
        sp = sub.add_parser(name, help=f"Stage command; use run-experiment for orchestrated execution")
        sp.add_argument("--config", type=Path, default=ROOT / "configs" / "quick.yaml")
    run = sub.add_parser("run-experiment", help="Run end-to-end reproducible experiment")
    run.add_argument("--config", type=Path, default=ROOT / "configs" / "quick.yaml")
    full = sub.add_parser(
        "run-full",
        help="Validate/build/run the complete fixture or pre-imported research pipeline",
    )
    full.add_argument("--config", type=Path, default=ROOT / "configs" / "full_demo.yaml")
    complete_case = sub.add_parser(
        "run-complete-case",
        help=(
            "Build an isolated complete-case real-data workspace and run the "
            "explicitly exploratory pipeline"
        ),
    )
    complete_case.add_argument(
        "--config", type=Path,
        default=ROOT / "configs" / "hose300_complete_case_exploratory.yaml",
    )
    complete_case.add_argument("--from", dest="start", default="2020-01-01")
    complete_case.add_argument("--to", dest="end", default="2025-12-31")
    complete_case.add_argument("--minimum-total-observations", type=int, default=40)
    complete_case.add_argument("--maximum-calendar-gap-days", type=int, default=5)
    data_b = sub.add_parser(
        "run-data-b",
        help=(
            "Reuse the immutable Data A cleaned panel, create a separate Data B workspace, "
            "and run the optimized leakage-aware exploratory pipeline"
        ),
    )
    data_b.add_argument(
        "--config", type=Path, default=ROOT / "configs" / "data_b.yaml",
    )
    data_b.add_argument(
        "--base-workspace", type=Path, default=ROOT / "outputs" / "Data A",
    )
    data_b.add_argument(
        "--output-workspace", type=Path, default=ROOT / "outputs" / "Data B",
    )
    cafef = sub.add_parser(
        "run-cafef",
        help="Crawl a separate CafeF-only panel, quality-gate it, and run only if accepted",
    )
    cafef.add_argument(
        "--config", type=Path,
        default=ROOT / "configs" / "cafef_standalone_exploratory.yaml",
    )
    cafef.add_argument("--from", dest="start", default="2020-01-01")
    cafef.add_argument("--to", dest="end", default="2025-12-31")
    cafef.add_argument(
        "--tickers",
        default="VCB,BID,CTG,MBB,HPG,FPT,VNM,VIC,GAS,MSN,MWG,SSI",
    )
    cafef.add_argument("--max-workers", type=int, default=3)
    cafef.add_argument(
        "--workspace-name",
        help="Stable output folder name under outputs, for example 'data CafeF'",
    )
    cafef.add_argument("--minimum-total-observations", type=int, default=40)
    cafef.add_argument("--maximum-calendar-gap-days", type=int, default=5)
    cafef.add_argument(
        "--existing-workspace", type=Path,
        help="Resume quality-gating an already collected CafeF workspace without recrawling",
    )
    prepare = sub.add_parser(
        "prepare-research-data",
        help="Inspect data contracts and build the PIT universe without fabricating missing tables",
    )
    prepare.add_argument("--config", type=Path, default=ROOT / "configs" / "hose300_real.yaml")
    hose_master = sub.add_parser(
        "crawl-hose-security-master",
        help="Collect official HOSE current listings and historical delisting events",
    )
    hose_master.add_argument("--from-year", type=int, default=2015)
    hose_master.add_argument("--to-year", type=int, default=2025)
    hose_master.add_argument("--pause-seconds", type=float, default=0.05)
    historical_prices = sub.add_parser(
        "crawl-historical-price-gaps",
        help="Checkpoint missing historical HOSE symbols using public adapters without promotion",
    )
    historical_prices.add_argument("--from", dest="start", default="2020-01-01")
    historical_prices.add_argument("--to", dest="end", default="2025-12-31")
    historical_prices.add_argument("--no-vnstock-fallback", action="store_true")
    historical_prices.add_argument("--pause-seconds", type=float, default=0.35)
    historical_merge = sub.add_parser(
        "merge-historical-price-checkpoints",
        help="Promote all available historical HOSE checkpoints using official security identities",
    )
    historical_merge.add_argument("--from", dest="start", default="2020-01-01")
    historical_merge.add_argument("--to", dest="end", default="2025-12-31")
    world_bank = sub.add_parser(
        "crawl-world-bank",
        help="Collect a keyless official World Bank Vietnam macro snapshot (non-PIT)",
    )
    world_bank.add_argument("--from-year", type=int, default=2015)
    world_bank.add_argument("--to-year", type=int, default=2025)
    actions = sub.add_parser(
        "crawl-corporate-actions",
        help="Collect VSDC official notices and CafeF ex-date corroboration into research_v2",
    )
    actions.add_argument("--from", dest="start", default="2020-01-01")
    actions.add_argument("--to", dest="end", default="2025-12-31")
    actions.add_argument("--tickers", default="auto")
    actions.add_argument("--max-workers", type=int, default=3)
    actions.add_argument("--pause-seconds", type=float, default=0.20)
    sub.add_parser(
        "build-price-adjustment-v2",
        help="Build raw/source-adjusted/research-total-return candidate and its fail-closed audit",
    )
    universe_v2 = sub.add_parser(
        "build-universe-pit-v2",
        help="Build the isolated monthly historical HOSE universe from prior-only data",
    )
    universe_v2.add_argument("--from", dest="start", default="2020-01-01")
    universe_v2.add_argument("--to", dest="end", default="2025-12-31")
    universe_v2.add_argument("--lookback-days", type=int, default=90)
    universe_v2.add_argument("--minimum-observations", type=int, default=40)
    sub.add_parser(
        "adjustment-counterfactual",
        help="Reprice frozen Data A holdings using raw, source-adjusted and research returns",
    )
    research_v2 = sub.add_parser(
        "run-research-v2",
        help=(
            "Stage and run the isolated confirmatory pipeline only after the "
            "adjustment and total-return benchmark gates pass"
        ),
    )
    research_v2.add_argument(
        "--config", type=Path, default=ROOT / "configs" / "hose_research_v2.yaml"
    )
    data_17_8_init = sub.add_parser(
        "init-data-17-8",
        help="Initialize the isolated Data 17/8 workspace from immutable Data A prices",
    )
    data_17_8_init.add_argument(
        "--base-workspace", type=Path, default=ROOT / "outputs" / "Data A"
    )
    data_17_8_crawl = sub.add_parser(
        "crawl-data-17-8",
        help="Crawl official HOSE disclosures, documents, TRI benchmark and sector reference",
    )
    data_17_8_crawl.add_argument(
        "--stage",
        choices=[
            "all", "disclosures", "documents", "benchmark", "sector",
            "corporate-actions", "prices", "cafef-prices",
        ],
        default="all",
    )
    data_17_8_crawl.add_argument("--from", dest="start", default="2020-01-01")
    data_17_8_crawl.add_argument("--to", dest="end", default="2025-12-31")
    data_17_8_crawl.add_argument("--max-workers", type=int, default=4)
    data_17_8_crawl.add_argument("--metadata-only", action="store_true")
    data_17_8_crawl.add_argument(
        "--pause-seconds", type=float, default=6.4,
        help="Minimum delay between uncached KBS price requests (at least 6.1 seconds)",
    )
    data_17_8_extract = sub.add_parser(
        "extract-data-17-8-documents",
        help="Extract native PDF text, OCR low-text pages and build PIT facts/sectors",
    )
    data_17_8_extract.add_argument("--max-workers", type=int, default=2)
    data_17_8_extract.add_argument("--max-documents", type=int)
    data_17_8_extract.add_argument("--maximum-pages", type=int, default=250)
    data_17_8_extract.add_argument("--no-ocr", action="store_true")
    sub.add_parser(
        "audit-data-17-8", help="Run the fail-closed Data 17/8 source and PIT audit"
    )
    data_17_8_run = sub.add_parser(
        "run-data-17-8",
        help="Run the locked Data 17/8 exploratory model after source auditing",
    )
    data_17_8_run.add_argument(
        "--config", type=Path, default=ROOT / "configs" / "data_17_8.yaml"
    )
    sub.add_parser("audit-data-sources", help="Write the current source and research-data gap inventory")
    return p


def _fmt(value, percent=False) -> str:
    try:
        number = float(value)
    except (TypeError, ValueError):
        return str(value)
    if pd.isna(number):
        return "NA"
    return f"{number * 100:.2f}%" if percent else f"{number:.6f}"


def print_experiment_summary(out: Path) -> None:
    manifest = json.loads((out / "manifest.json").read_text(encoding="utf-8"))
    quality = json.loads((out / "data_quality.json").read_text(encoding="utf-8"))
    leakage = json.loads((out / "leakage_audit.json").read_text(encoding="utf-8"))
    metrics = pd.read_csv(out / "metrics_long.csv")
    comparisons = pd.read_csv(out / "comparisons.csv")
    statistics = pd.read_csv(out / "statistical_tests.csv")
    ablations = pd.read_csv(out / "ablation_results.csv")
    sensitivity = pd.read_csv(out / "sensitivity_results.csv")
    rankings = pd.read_csv(out / "rankings.csv")
    trades = pd.read_csv(out / "trades.csv")
    weights = pd.read_csv(out / "weights.csv")
    latest_path = out / "latest_selected_portfolio.csv"
    latest = pd.read_csv(latest_path) if latest_path.exists() else pd.DataFrame()
    print("\n" + "=" * 100)
    print("BÁO CÁO CHẠY TOÀN BỘ HỆ THỐNG AI–QUANTUM PORTFOLIO")
    print("=" * 100)
    print(f"Experiment ID       : {manifest['experiment_id']}")
    print(f"Trạng thái          : {manifest['status']}")
    print(f"Nhãn dữ liệu         : {manifest['label']}")
    print(f"Config hash          : {manifest['config_hash']}")
    print(f"Dataset hash         : {manifest['dataset_hash']}")
    print(f"Walk-forward folds   : {manifest['folds_completed']}/{manifest['folds_requested']}")
    print(f"Thư mục kết quả      : {out}")
    print("\n[1] KIỂM TRA DỮ LIỆU VÀ POINT-IN-TIME")
    print("-" * 100)
    print(f"Data class           : {', '.join(quality['data_class'])}")
    print(f"Số bản ghi           : {quality['records']:,}")
    print(f"Số mã                : {quality['tickers']}")
    print(f"Khoảng thời gian     : {quality['start']} → {quality['end']}")
    print(f"Data quality         : {quality['status']}")
    print(f"Leakage audit        : {leakage['status']}")
    for check, passed in leakage["checks"].items():
        print(f"  - {check:<50}: {'PASS' if passed else 'FAIL'}")
    print("\n[2] CHẤT LƯỢNG XẾP HẠNG XGBOOST")
    print("-" * 100)
    fold_ic = rankings.groupby("fold")["fold_rank_ic"].first()
    print(f"Số ranking rows      : {len(rankings):,}")
    print(f"Mean rank IC         : {fold_ic.mean():.6f}")
    print(f"Median rank IC       : {fold_ic.median():.6f}")
    print(f"Min/Max rank IC      : {fold_ic.min():.6f} / {fold_ic.max():.6f}")
    print("\n[3] SO SÁNH SOLVER")
    print("-" * 100)
    solver_view = comparisons[[
        "method", "energy_mean", "feasibility_rate", "optimality_gap_mean",
        "runtime_seconds", "runs",
    ]].copy()
    solver_view["feasibility_rate"] = solver_view["feasibility_rate"].map(
        lambda x: _fmt(x, percent=True)
    )
    print(solver_view.to_string(index=False))
    print("\n[4] KẾT QUẢ DANH MỤC VÀ BACKTEST")
    print("-" * 100)
    metric_view = metrics[[
        "strategy", "cumulative_return", "annualized_return", "annualized_volatility",
        "sharpe", "sortino", "max_drawdown", "observations",
    ]].copy()
    for column in ["cumulative_return", "annualized_return", "annualized_volatility", "max_drawdown"]:
        metric_view[column] = metric_view[column].map(lambda x: _fmt(x, percent=True))
    for column in ["sharpe", "sortino"]:
        metric_view[column] = metric_view[column].map(_fmt)
    print(metric_view.to_string(index=False))
    print(f"\nSố trade rows        : {len(trades):,}")
    print(f"Số weight rows       : {len(weights):,}")
    print("\nRổ cổ phiếu của fold cuối cùng:")
    if latest.empty:
        print("  Không có fold nghiên cứu hoàn tất.")
    else:
        columns = [column for column in [
            "ticker", "company_name", "sector", "target_weight", "signal",
            "trade_weight", "estimated_cost", "adv_participation",
        ] if column in latest.columns]
        print(latest[columns].to_string(index=False))
    print("\n[5] ABLATION STUDY")
    print("-" * 100)
    ablation_view = ablations.groupby(["configuration", "selector", "solver"]).agg(
        folds=("fold", "nunique"),
        objective_mean=("objective", "mean"),
        feasibility_rate=("feasibility_rate", "mean"),
        optimality_gap=("optimality_gap", "mean"),
    ).reset_index()
    ablation_view["feasibility_rate"] = ablation_view["feasibility_rate"].map(
        lambda x: _fmt(x, percent=True)
    )
    print(ablation_view.to_string(index=False))
    print("\n[6] SENSITIVITY / ROBUSTNESS")
    print("-" * 100)
    print(f"Số sensitivity cases: {len(sensitivity):,}")
    sensitivity_view = sensitivity.groupby(
        ["depth_p", "shots", "cardinality", "uniform_probability_noise_proxy"]
    ).agg(
        feasibility_rate=("feasibility_rate", "mean"),
        optimality_gap=("optimality_gap", "mean"),
        runtime_seconds=("runtime_seconds", "mean"),
    ).reset_index()
    sensitivity_view["feasibility_rate"] = sensitivity_view["feasibility_rate"].map(
        lambda x: _fmt(x, percent=True)
    )
    print(sensitivity_view.to_string(index=False))
    print("\n[7] KIỂM ĐỊNH THỐNG KÊ — BLOCK BOOTSTRAP + HOLM")
    print("-" * 100)
    stats_view = statistics[[
        "test", "mean_difference", "ci_low", "ci_high",
        "p_value", "p_value_holm", "conclusion",
    ]]
    print(stats_view.to_string(index=False))
    print("\n[8] CÁC FILE OUTPUT")
    print("-" * 100)
    for name in manifest["artifacts"]:
        print(f"  - {out / name}")
    print("\nKẾT LUẬN THỰC THI")
    print("-" * 100)
    if manifest.get("mode") == "exploratory":
        print("Toàn bộ pipeline đã chạy thành công trên panel giá thật dạng complete-case.")
        print("Kết quả chỉ mang tính khám phá, không phải kiểm định confirmatory toàn HOSE,")
        print("khuyến nghị đầu tư hoặc bằng chứng quantum advantage.")
    elif "fixture" in str(manifest.get("data_class", "")).lower():
        print("Toàn bộ code path đã chạy thành công. Kết quả fixture chỉ dùng kiểm thử phần mềm;")
        print("không được diễn giải là kết quả nghiên cứu HOSE hoặc quantum advantage.")
    else:
        print("Toàn bộ pipeline nghiên cứu đã chạy thành công trên panel giá thị trường thực.")
        print("Kết quả phụ thuộc cấu hình và mẫu nghiên cứu; không phải khuyến nghị đầu tư hay bằng chứng quantum advantage.")
    print("=" * 100 + "\n")


def main(argv=None) -> int:
    configure_utf8_console()
    args = parser().parse_args(argv)
    paths = Paths(ROOT)
    if args.command == "crawl":
        if args.dry_run:
            print(json.dumps(vars(args), default=str, indent=2))
            return 0
        if args.stage != 1 and args.source != "csv":
            raise SystemExit("Stage 2/3 use import-pit-table or an explicitly configured adapter.")
        if args.source == "fixture":
            result = generate_fixture(paths, args.start, args.end, args.tickers.split(","))
        elif args.source == "csv":
            if not args.input:
                raise SystemExit("--input is required for --source csv")
            result = import_csv(paths, args.input, args.source_name, args.source_url)
        elif args.source == "ssi":
            result = crawl_ssi_stage1(paths, args.tickers.split(","), args.start, args.end)
        elif args.source == "vietstock":
            result = crawl_vietstock_stage1(
                paths,
                args.tickers.split(","),
                args.start,
                args.end,
                verify_tls=not args.insecure,
            )
        elif args.source == "vnstock":
            requested = None if args.tickers.strip().lower() == "auto" else args.tickers.split(",")
            result = crawl_vnstock_hose(
                paths, args.start, args.end,
                max_tickers=args.max_tickers, tickers=requested,
            )
        elif args.source == "fdr":
            requested = None if args.tickers.strip().lower() == "auto" else args.tickers.split(",")
            result = crawl_fdr_hose(
                paths, args.start, args.end,
                max_tickers=args.max_tickers, tickers=requested,
            )
        else:
            if args.tickers.strip().lower() == "auto":
                raise SystemExit(
                    "Trading Economics cross-check requires explicit tickers; its current-list API "
                    "must not define the historical HOSE universe."
                )
            result = crawl_trading_economics_crosscheck(
                paths, args.tickers.split(","), args.start, args.end
            )
        if args.source != "fixture":
            result["quarantined_fixture_auxiliary"] = quarantine_fixture_auxiliary(paths)
        print(json.dumps(result, indent=2))
    elif args.command == "import-pit-table":
        contracts = {
            "index_membership": {"ticker", "index_code", "effective_from", "effective_to", "available_at", "source", "source_url", "history_method"},
            "corporate_actions": {"ticker", "security_id", "event_type", "announcement_date", "effective_date", "available_at", "source", "source_url"},
            "financial_statements": {"ticker", "fiscal_period_end", "publication_date", "available_at", "source", "source_url"},
            "macro": {"series_id", "observation_date", "release_date", "available_at", "value", "source", "source_url"},
            "foreign_flow": {"date", "ticker", "available_at", "foreign_net_value", "source", "source_url"},
            "benchmark": {"date", "benchmark", "total_return_index", "index_type", "methodology_url", "available_at", "source", "source_url"},
            "security_master": {
                "security_id", "ticker", "exchange", "listing_date", "delisting_date", "effective_from",
                "effective_to", "available_at", "history_method", "source", "source_url",
            },
        }
        output = paths.normalized / f"{args.table}.parquet"
        result = import_point_in_time_table(args.input, output, contracts[args.table], args.table)
        print(json.dumps(result, indent=2))
    elif args.command == "apply-adjustment-contract":
        print(json.dumps(
            apply_price_adjustment_contract(paths, args.input),
            indent=2, ensure_ascii=False,
        ))
    elif args.command == "normalize":
        print("Normalization is idempotently performed during crawl/import.")
    elif args.command == "merge-market-sources":
        result = merge_hose_checkpoints(paths, args.target_tickers)
        result["quarantined_fixture_auxiliary"] = quarantine_fixture_auxiliary(paths)
        print(json.dumps(
            result,
            indent=2, ensure_ascii=False,
        ))
    elif args.command in {"validate", "report-coverage"}:
        report, coverage = validate_data(paths)
        print(json.dumps(report, indent=2))
        if args.command == "report-coverage":
            print(coverage.to_string(index=False))
    elif args.command == "build-universe":
        universe = build_universe(
            paths, args.rebalance, args.definition, args.index_code,
            args.max_assets, args.liquidity_lookback_days, args.minimum_observations,
        )
        print(f"universe_rows={len(universe)}")
    elif args.command == "leakage-audit":
        print(json.dumps(leakage_audit(paths), indent=2))
    elif args.command == "build-features":
        prices = pd.read_parquet(paths.normalized / "prices.parquet")
        from .research import attach_point_in_time_features
        features = attach_point_in_time_features(build_features(prices), paths)
        paths.curated.mkdir(parents=True, exist_ok=True)
        features.to_parquet(paths.curated / "features.parquet", index=False)
        print(f"feature_rows={len(features)}")
    elif args.command == "prepare-research-data":
        cfg = load_config(args.config.resolve())
        universe_cfg = cfg.get("universe", {})
        credential_status = {
            name: bool(os.getenv(name)) for name in [
                "SSI_CONSUMER_ID", "SSI_CONSUMER_SECRET", "VIETSTOCK_COOKIE_FILE",
                "VIETSTOCK_AUTH_HEADER_FILE", "TRADING_ECONOMICS_API_KEY", "FRED_API_KEY",
            ]
        }
        quality, _ = validate_data(paths)
        build_error = None
        try:
            universe = build_universe(
                paths, cfg["data"].get("rebalance", "monthly"),
                universe_cfg.get("definition", "hose_all_listed"),
                universe_cfg.get("index_code"), universe_cfg.get("max_assets"),
                universe_cfg.get("liquidity_lookback_days", 60),
                universe_cfg.get("minimum_observations", 40),
            )
            universe_rows = len(universe)
        except (ValueError, KeyError) as exc:
            build_error = str(exc)
            universe_rows = 0
        result = {
            "config": str(args.config.resolve()), "credentials_configured": credential_status,
            "data_quality": quality, "universe_rows": universe_rows,
            "universe_build_error": build_error, "leakage_audit": leakage_audit(paths),
            "note": "Credential booleans only; no secret values are printed or persisted.",
        }
        print(json.dumps(result, indent=2, ensure_ascii=False))
    elif args.command == "crawl-hose-security-master":
        result = crawl_hose_official_security_master(
            paths, args.from_year, args.to_year, args.pause_seconds
        )
        print(json.dumps(result, indent=2, ensure_ascii=False))
    elif args.command == "crawl-historical-price-gaps":
        result = crawl_historical_hose_price_gaps(
            paths, args.start, args.end,
            try_vnstock_fallback=not args.no_vnstock_fallback,
            pause_seconds=args.pause_seconds,
        )
        print(json.dumps(result, indent=2, ensure_ascii=False))
    elif args.command == "merge-historical-price-checkpoints":
        result = merge_historical_hose_checkpoints(paths, args.start, args.end)
        print(json.dumps(result, indent=2, ensure_ascii=False))
    elif args.command == "crawl-world-bank":
        result = crawl_world_bank_vietnam_snapshot(paths, args.from_year, args.to_year)
        print(json.dumps(result, indent=2, ensure_ascii=False))
    elif args.command == "crawl-corporate-actions":
        requested = None if args.tickers.strip().lower() == "auto" else args.tickers.split(",")
        result = crawl_corporate_actions(
            paths, args.start, args.end, requested, args.max_workers, args.pause_seconds,
        )
        print(json.dumps(result, indent=2, ensure_ascii=False))
    elif args.command == "build-price-adjustment-v2":
        result = build_price_adjustment_v2(paths)
        print(json.dumps(result, indent=2, ensure_ascii=False))
        if result["status"] == "blocked":
            raise SystemExit(2)
    elif args.command == "build-universe-pit-v2":
        result = build_historical_universe_pit(
            paths, args.start, args.end, args.lookback_days, args.minimum_observations,
        )
        print(json.dumps(result, indent=2, ensure_ascii=False))
    elif args.command == "adjustment-counterfactual":
        result = build_return_only_adjustment_counterfactual(paths)
        print(json.dumps(result, indent=2, ensure_ascii=False))
    elif args.command == "run-research-v2":
        try:
            runtime_root = prepare_research_v2_runtime(paths)
        except RuntimeError as exc:
            raise SystemExit(f"RESEARCH_V2_BLOCKED: {exc}") from exc
        temporary_out = run_experiment(runtime_root, args.config.resolve())
        experiments = ROOT / "outputs" / "research_v2" / "experiments"
        experiments.mkdir(parents=True, exist_ok=True)
        out = experiments / temporary_out.name
        if out.exists():
            raise RuntimeError(f"Research V2 artifact already exists: {out}")
        shutil.copytree(temporary_out, out)
        print_experiment_summary(out)
    elif args.command == "init-data-17-8":
        result = initialize_data_17_8(ROOT, args.base_workspace.resolve())
        print(json.dumps(result, indent=2, ensure_ascii=False))
    elif args.command == "crawl-data-17-8":
        initialize_data_17_8(ROOT)
        stages = (
            [
                "disclosures", "documents", "sector", "corporate-actions",
                "prices", "cafef-prices", "benchmark",
            ]
            if args.stage == "all" else [args.stage]
        )
        results = {}
        for stage in stages:
            if stage == "disclosures":
                results[stage] = crawl_hose_disclosure_index(
                    ROOT, args.start, args.end, args.max_workers
                )
            elif stage == "documents":
                results["hose_documents"] = crawl_hose_documents(
                    ROOT, args.max_workers, download=not args.metadata_only
                )
                results["vietstock_documents"] = crawl_vietstock_company_documents(
                    ROOT, args.max_workers, download=not args.metadata_only
                )
            elif stage == "benchmark":
                results[stage] = crawl_hose_tri_benchmark(ROOT, max_workers=args.max_workers)
            elif stage == "sector":
                results[stage] = crawl_current_hose_sector_reference(ROOT)
            elif stage == "corporate-actions":
                results[stage] = stage_data_17_8_corporate_actions(ROOT)
            elif stage == "prices":
                results[stage] = crawl_data_17_8_prices(
                    ROOT, args.start, args.end, args.pause_seconds
                )
            elif stage == "cafef-prices":
                results[stage] = crawl_data_17_8_cafef_price_crosscheck(
                    ROOT, args.start, args.end, args.max_workers
                )
                results["research_universe"] = finalize_data_17_8_research_universe(ROOT)
        print(json.dumps(results, indent=2, ensure_ascii=False, default=str))
    elif args.command == "extract-data-17-8-documents":
        archives = extract_company_document_archives(ROOT)
        extraction = extract_company_document_text(
            ROOT,
            max_workers=args.max_workers,
            maximum_documents=args.max_documents,
            maximum_pages=args.maximum_pages,
            use_ocr=not args.no_ocr,
        )
        financial = build_financial_statement_facts(ROOT)
        sector = build_historical_sector_pit(ROOT)
        print(json.dumps({
            "archives": archives,
            "extraction": extraction,
            "financial_statements": financial,
            "historical_sector": sector,
        }, indent=2, ensure_ascii=False, default=str))
    elif args.command == "audit-data-17-8":
        audit = audit_data_17_8(ROOT)
        report = write_data_17_8_source_report(ROOT)
        audit["report"] = str(report)
        print(json.dumps(audit, indent=2, ensure_ascii=False, default=str))
    elif args.command == "run-data-17-8":
        audit = audit_data_17_8(ROOT)
        if not audit.get("exploratory_run_permitted"):
            raise SystemExit("DATA_17_8_BLOCKED: price panel did not pass the exploratory gate")
        output = run_experiment(data_17_8_workspace(ROOT), args.config.resolve())
        write_data_17_8_source_report(ROOT)
        print_experiment_summary(output)
    elif args.command == "audit-data-sources":
        print(json.dumps(audit_available_data_sources(paths), indent=2, ensure_ascii=False))
    elif args.command == "run-experiment":
        out = run_experiment(ROOT, args.config.resolve())
        print_experiment_summary(out)
    elif args.command == "run-full":
        cfg = load_config(args.config.resolve())
        if cfg["data"]["source"] == "fixture":
            # A demo must never replace, certify, or otherwise mutate the real-data
            # workspace. Build and execute it in an isolated temporary project, then
            # copy only the immutable experiment artifact back to outputs/experiments.
            with tempfile.TemporaryDirectory(prefix="ai-quantum-fixture-") as temporary:
                demo_root = Path(temporary)
                demo_paths = Paths(demo_root)
                manifest = generate_fixture(
                    demo_paths, cfg["data"]["start"], cfg["data"]["end"],
                    cfg["data"]["tickers"], cfg["seed"],
                )
                print(json.dumps(manifest, indent=2, ensure_ascii=False))
                quality, _ = validate_data(demo_paths)
                print(json.dumps(quality, indent=2, ensure_ascii=False))
                universe_cfg = cfg.get("universe", {})
                universe = build_universe(
                    demo_paths, cfg["data"].get("rebalance", "monthly"),
                    universe_cfg.get("definition", "hose_all_listed"),
                    universe_cfg.get("index_code"), universe_cfg.get("max_assets"),
                    universe_cfg.get("liquidity_lookback_days", 60),
                    universe_cfg.get("minimum_observations", 40),
                )
                print(f"universe_rows={len(universe):,}")
                print(json.dumps(leakage_audit(demo_paths), indent=2, ensure_ascii=False))
                temporary_out = run_experiment(demo_root, args.config.resolve())
                experiments = ROOT / "outputs" / "experiments"
                experiments.mkdir(parents=True, exist_ok=True)
                out = experiments / temporary_out.name
                if out.exists():
                    raise RuntimeError(f"Experiment artifact already exists: {out}")
                shutil.copytree(temporary_out, out)
            print_experiment_summary(out)
            return 0
        if not (paths.normalized / "prices.parquet").exists():
            raise SystemExit(
                "Research run-full requires an existing normalized price panel. "
                "Run the authorized crawl/import command first."
            )
        quality, coverage = validate_data(paths)
        print(json.dumps(quality, indent=2, ensure_ascii=False))
        universe_cfg = cfg.get("universe", {})
        universe = build_universe(
            paths, cfg["data"].get("rebalance", "monthly"),
            universe_cfg.get("definition", "hose_all_listed"),
            universe_cfg.get("index_code"),
            universe_cfg.get("max_assets"),
            universe_cfg.get("liquidity_lookback_days", 60),
            universe_cfg.get("minimum_observations", 40),
        )
        print(f"universe_rows={len(universe):,}")
        print(json.dumps(leakage_audit(paths), indent=2, ensure_ascii=False))
        out = run_experiment(ROOT, args.config.resolve())
        print_experiment_summary(out)
    elif args.command == "run-complete-case":
        workspace, dataset_manifest = build_complete_case_workspace(
            paths, args.start, args.end, args.minimum_total_observations,
            args.maximum_calendar_gap_days,
        )
        print(json.dumps(dataset_manifest, indent=2, ensure_ascii=False))
        complete_paths = Paths(workspace)
        quality, _ = validate_data(complete_paths)
        print(json.dumps(quality, indent=2, ensure_ascii=False))
        temporary_out = run_experiment(workspace, args.config.resolve())
        (temporary_out / "complete_case_dataset_manifest.json").write_text(
            json.dumps(dataset_manifest, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        temporary_manifest_path = temporary_out / "manifest.json"
        temporary_manifest = json.loads(temporary_manifest_path.read_text(encoding="utf-8"))
        temporary_manifest["artifacts"] = sorted(
            path.relative_to(temporary_out).as_posix()
            for path in temporary_out.rglob("*") if path.is_file()
        )
        temporary_manifest["artifact_sha256"] = {
            path.relative_to(temporary_out).as_posix(): sha256_file(path)
            for path in temporary_out.rglob("*")
            if path.is_file() and path.name != "manifest.json"
        }
        temporary_manifest_path.write_text(
            json.dumps(temporary_manifest, indent=2), encoding="utf-8"
        )
        experiments = ROOT / "outputs" / "experiments"
        experiments.mkdir(parents=True, exist_ok=True)
        out = experiments / temporary_out.name
        if out.exists():
            raise RuntimeError(f"Experiment artifact already exists: {out}")
        shutil.copytree(temporary_out, out)
        print(f"Complete-case workspace: {workspace}")
        print_experiment_summary(out)
    elif args.command == "run-data-b":
        base_workspace = args.base_workspace.resolve()
        output_workspace = args.output_workspace.resolve()
        base_outputs = base_workspace / "outputs"
        required = [
            base_outputs / "normalized" / "prices.parquet",
            base_outputs / "normalized" / "security_master.parquet",
            base_outputs / "raw" / "manifest.json",
        ]
        missing = [str(path) for path in required if not path.exists()]
        if missing:
            raise SystemExit("Data B requires the existing Data A package: " + ", ".join(missing))
        if not output_workspace.exists():
            (output_workspace / "outputs").mkdir(parents=True, exist_ok=False)
            for folder in ["raw", "normalized", "reports"]:
                source = base_outputs / folder
                if source.exists():
                    shutil.copytree(source, output_workspace / "outputs" / folder)
        else:
            output_prices = output_workspace / "outputs" / "normalized" / "prices.parquet"
            if not output_prices.exists():
                raise RuntimeError(
                    f"Existing Data B folder is incomplete and was not overwritten: {output_workspace}"
                )
            if sha256_file(output_prices) != sha256_file(required[0]):
                raise RuntimeError(
                    "Existing Data B price panel no longer matches Data A; refusing implicit overwrite."
                )
        data_b_paths = Paths(output_workspace)
        quality, _ = validate_data(data_b_paths)
        print(json.dumps(quality, indent=2, ensure_ascii=False))
        out = run_experiment(output_workspace, args.config.resolve())
        base_manifest = base_workspace / "DATA_A_PACKAGE.json"
        package = {
            "package": "Data B",
            "status": "completed_exploratory" if (out / "strategy_metrics_summary.csv").exists()
            else "incomplete",
            "base_workspace": str(base_workspace),
            "base_data_a_manifest": str(base_manifest) if base_manifest.exists() else None,
            "base_price_sha256": sha256_file(required[0]),
            "config": str(args.config.resolve()),
            "config_sha256": sha256_file(args.config.resolve()),
            "experiment_id": out.name,
            "experiment_path": str(out),
            "method": (
                "Data A reuse + purged-validation XGBoost/technical blend + adaptive "
                "universe reduction + best-observed XY-QAOA + constrained weights + "
                "market-regime exposure"
            ),
            "interpretation": "exploratory_only_not_confirmatory_research",
        }
        (output_workspace / "DATA_B_PACKAGE.json").write_text(
            json.dumps(package, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        print(f"Data B workspace: {output_workspace}")
        print_experiment_summary(out)
    elif args.command == "run-cafef":
        if args.existing_workspace:
            collection_root = args.existing_workspace.resolve()
            collection_manifest_path = collection_root / "outputs" / "raw" / "manifest.json"
            if not collection_manifest_path.exists():
                raise SystemExit(
                    f"Existing CafeF workspace has no collection manifest: "
                    f"{collection_manifest_path}"
                )
            collection_manifest = json.loads(
                collection_manifest_path.read_text(encoding="utf-8")
            )
        else:
            requested_tickers = (
                None if args.tickers.strip().lower() == "auto" else args.tickers.split(",")
            )
            collection_root, collection_manifest = crawl_cafef_standalone_workspace(
                paths, args.start, args.end, requested_tickers, args.max_workers,
                args.workspace_name,
            )
        print(json.dumps(collection_manifest, indent=2, ensure_ascii=False))
        if collection_manifest.get("status") == "rejected":
            raise SystemExit(
                f"CafeF dataset rejected before quality gate. Audit: "
                f"{collection_root / 'outputs' / 'raw' / 'manifest.json'}"
            )
        complete_root, complete_manifest = build_complete_case_workspace(
            Paths(collection_root), args.start, args.end,
            args.minimum_total_observations, args.maximum_calendar_gap_days,
        )
        cfg = load_config(args.config.resolve())
        retained = int(complete_manifest["tickers_retained"])
        required = int(cfg.get("reduction", {}).get("candidate_size", 8))
        complete_paths = Paths(complete_root)
        quality, _ = validate_data(complete_paths)
        initial_quality = quality
        quality_excluded_tickers: list[str] = []
        if quality["status"] != "pass":
            review_path = complete_paths.reports / "return_outlier_review.csv"
            if review_path.exists():
                review = pd.read_csv(review_path)
                quality_excluded_tickers = sorted(
                    review.loc[
                        review.get("resolution", pd.Series(dtype=str)).astype(str).eq("unresolved"),
                        "ticker",
                    ].dropna().astype(str).unique().tolist()
                )
            non_outlier_errors = [
                issue for issue in quality.get("issues", [])
                if issue.get("severity") == "error"
                and issue.get("check") != "unresolved_return_outlier"
            ]
            if (
                quality_excluded_tickers
                and not non_outlier_errors
                and retained - len(quality_excluded_tickers) >= required
            ):
                complete_root, complete_manifest = build_complete_case_workspace(
                    Paths(collection_root), args.start, args.end,
                    args.minimum_total_observations, args.maximum_calendar_gap_days,
                    forced_excluded_tickers=quality_excluded_tickers,
                )
                complete_paths = Paths(complete_root)
                quality, _ = validate_data(complete_paths)
                retained = int(complete_manifest["tickers_retained"])
        acceptance = {
            "status": "accepted" if quality["status"] == "pass" and retained >= required else "rejected",
            "quality": quality,
            "initial_quality": initial_quality,
            "quality_excluded_tickers": quality_excluded_tickers,
            "quality_exclusion_policy": (
                "drop_entire_ticker_with_unresolved_return_outlier; never alter source price"
            ),
            "retained_tickers": retained,
            "minimum_required_tickers": required,
            "collection_workspace": str(collection_root),
            "complete_case_workspace": str(complete_root),
            "created_at": pd.Timestamp.utcnow().isoformat(),
        }
        (complete_paths.reports / "cafef_acceptance_gate.json").write_text(
            json.dumps(acceptance, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        print(json.dumps(acceptance, indent=2, ensure_ascii=False))
        if acceptance["status"] != "accepted":
            raise SystemExit(
                "CafeF panel did not meet the declared system quality gate; no training or "
                f"backtest was run. Audit: {complete_paths.reports / 'cafef_acceptance_gate.json'}"
            )
        temporary_out = run_experiment(complete_root, args.config.resolve())
        for name, payload in (
            ("cafef_collection_manifest.json", collection_manifest),
            ("complete_case_dataset_manifest.json", complete_manifest),
            ("cafef_acceptance_gate.json", acceptance),
        ):
            (temporary_out / name).write_text(
                json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8"
            )
        temporary_manifest_path = temporary_out / "manifest.json"
        temporary_manifest = json.loads(temporary_manifest_path.read_text(encoding="utf-8"))
        temporary_manifest["artifacts"] = sorted(
            path.relative_to(temporary_out).as_posix()
            for path in temporary_out.rglob("*") if path.is_file()
        )
        temporary_manifest["artifact_sha256"] = {
            path.relative_to(temporary_out).as_posix(): sha256_file(path)
            for path in temporary_out.rglob("*")
            if path.is_file() and path.name != "manifest.json"
        }
        temporary_manifest_path.write_text(
            json.dumps(temporary_manifest, indent=2), encoding="utf-8"
        )
        experiments = ROOT / "outputs" / "experiments"
        experiments.mkdir(parents=True, exist_ok=True)
        out = experiments / temporary_out.name
        if out.exists():
            raise RuntimeError(f"Experiment artifact already exists: {out}")
        shutil.copytree(temporary_out, out)
        print(f"CafeF collection workspace: {collection_root}")
        print(f"CafeF accepted workspace: {complete_root}")
        print_experiment_summary(out)
    else:
        out = run_experiment(ROOT, args.config.resolve())
        artifact_map = {
            "make-folds": "fold_manifest.csv",
            "train-ranker": "rankings.csv",
            "build-instances": "optimization_instances.json",
            "run-solvers": "solver_runs.csv",
            "optimize-weights": "weights.csv",
            "backtest": "portfolio_returns.csv",
            "evaluate": "RESEARCH_REPORT.md",
        }
        print(out.relative_to(ROOT) / artifact_map[args.command])
    return 0


if __name__ == "__main__":
    try:
        raise SystemExit(main())
    except ResearchRunBlocked as exc:
        print(f"RESEARCH RUN BLOCKED: {exc}", file=sys.stderr)
        print(f"Audit artifact: {exc.output_dir}", file=sys.stderr)
        raise SystemExit(2)


In [ ]:
%%writefile /content/ai_quantum_standalone/src/data_pipeline.py
from __future__ import annotations

import hashlib
import json
import shutil
import uuid
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


PRICE_COLUMNS = [
    "date", "ticker", "security_id", "open", "high", "low", "close", "adjusted_close", "volume",
    "trading_value", "source", "source_url", "fetched_at", "available_at",
    "raw_checksum", "parser_version", "data_class", "adjustment_policy",
]

PROVENANCE_COLUMNS = {"source", "source_url", "fetched_at", "raw_checksum"}
HISTORICAL_UNIVERSE_METHODS = {
    "official_event_history",
    "exchange_listing_history",
    "verified_membership_history",
    "verified_provider_history",
    "fixture",
}
VERIFIED_ADJUSTMENT_POLICIES = {
    "verified_corporate_action_adjusted",
    "unadjusted_with_verified_actions_join",
    "verified_vendor_total_return_adjusted",
    "fixture",
}
UNIVERSE_DEFINITIONS = {"hose_all_listed", "index_membership"}


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


@dataclass
class Paths:
    root: Path

    @property
    def raw(self) -> Path:
        return self.root / "outputs" / "raw"

    @property
    def normalized(self) -> Path:
        return self.root / "outputs" / "normalized"

    @property
    def curated(self) -> Path:
        return self.root / "outputs" / "curated"

    @property
    def reports(self) -> Path:
        return self.root / "outputs" / "reports"

    @property
    def staging(self) -> Path:
        return self.root / "outputs" / "staging"

    def ensure(self) -> None:
        for p in (self.raw, self.normalized, self.curated, self.reports, self.staging):
            p.mkdir(parents=True, exist_ok=True)


def create_staging_run(paths: Paths, source: str) -> Path:
    """Create a versioned collection directory without touching normalized data."""
    paths.ensure()
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run = paths.staging / f"{stamp}-{source}-{uuid.uuid4().hex[:8]}"
    run.mkdir(parents=True, exist_ok=False)
    return run


def build_complete_case_workspace(
    source_paths: Paths,
    start: str,
    end: str,
    minimum_total_observations: int = 40,
    maximum_calendar_gap_days: int = 5,
    forced_excluded_tickers: list[str] | None = None,
) -> tuple[Path, dict]:
    """Create an isolated, analysis-ready real-data workspace.

    The function never changes the canonical normalized panel. It retains only rows
    with complete OHLCV/provenance fields and securities with enough observations in
    the requested interval. The resulting security master is restricted to those
    observed securities so that downstream universe construction cannot reintroduce
    symbols with no usable price history.

    This is intentionally an *exploratory complete-case* dataset. Selecting securities
    using full-period data availability can introduce coverage/survivorship selection
    bias, and an unverified corporate-action adjustment policy remains unverified.
    """
    if minimum_total_observations < 1:
        raise ValueError("minimum_total_observations must be positive")
    if maximum_calendar_gap_days < 0:
        raise ValueError("maximum_calendar_gap_days must not be negative")
    prices_path = source_paths.normalized / "prices.parquet"
    master_path = source_paths.normalized / "security_master.parquet"
    if not prices_path.exists() or not master_path.exists():
        raise FileNotFoundError(
            "Complete-case construction requires normalized prices.parquet and "
            "security_master.parquet."
        )

    prices = pd.read_parquet(prices_path).copy()
    master = pd.read_parquet(master_path).copy()
    prices["date"] = pd.to_datetime(prices["date"], errors="coerce")
    prices["available_at"] = pd.to_datetime(prices["available_at"], errors="coerce")
    start_ts, end_ts = pd.Timestamp(start), pd.Timestamp(end)
    forced_excluded = {
        str(ticker).upper().strip() for ticker in (forced_excluded_tickers or [])
    }
    if start_ts > end_ts:
        raise ValueError("start must not be after end")
    prices = prices[prices["date"].between(start_ts, end_ts)].copy()
    if prices.get("data_class", pd.Series(dtype=str)).astype(str).eq("fixture").any():
        raise ValueError("Complete-case real-data construction refuses fixture observations.")

    required_non_null = [
        "date", "ticker", "security_id", "open", "high", "low", "close",
        "adjusted_close", "volume", "trading_value", "source", "source_url",
        "fetched_at", "available_at", "raw_checksum", "parser_version", "data_class",
    ]
    missing_columns = sorted(set(required_non_null) - set(prices.columns))
    if missing_columns:
        raise ValueError(f"Price panel is missing complete-case fields: {missing_columns}")
    numeric = ["open", "high", "low", "close", "adjusted_close", "volume", "trading_value"]
    complete = prices[required_non_null].notna().all(axis=1)
    complete &= np.isfinite(prices[numeric].astype(float)).all(axis=1)
    complete &= prices[["open", "high", "low", "close", "adjusted_close"]].gt(0).all(axis=1)
    complete &= prices[["volume", "trading_value"]].ge(0).all(axis=1)
    complete &= prices["high"].ge(prices[["open", "close", "low"]].max(axis=1))
    complete &= prices["low"].le(prices[["open", "close", "high"]].min(axis=1))
    complete &= prices["available_at"].ge(prices["date"])
    complete &= ~prices.duplicated(["ticker", "date"], keep=False)
    valid_prices = prices.loc[complete].sort_values(["ticker", "date"]).copy()

    master["ticker"] = master["ticker"].astype(str)
    master["listing_date"] = pd.to_datetime(master["listing_date"], errors="coerce")
    master["delisting_date"] = pd.to_datetime(master["delisting_date"], errors="coerce")
    relevant = master[
        master["listing_date"].le(end_ts)
        & (master["delisting_date"].isna() | master["delisting_date"].ge(start_ts))
    ].copy()
    calendar = pd.DatetimeIndex(sorted(valid_prices["date"].dropna().unique()))
    calendar_position = {date: position for position, date in enumerate(calendar)}
    counts = valid_prices.groupby("ticker")["date"].nunique()
    coverage_diagnostics: dict[str, dict] = {}
    eligible_tickers: set[str] = set()
    for row in relevant.itertuples():
        ticker = str(row.ticker)
        ticker_dates = pd.DatetimeIndex(sorted(
            valid_prices.loc[valid_prices["ticker"].astype(str).eq(ticker), "date"].unique()
        ))
        listing_start = max(start_ts, row.listing_date)
        listing_end = (
            min(end_ts, row.delisting_date) if pd.notna(row.delisting_date) else end_ts
        )
        expected = calendar[(calendar >= listing_start) & (calendar <= listing_end)]
        if len(ticker_dates):
            leading_gap = int((expected < ticker_dates.min()).sum())
            trailing_gap = int((expected > ticker_dates.max()).sum())
            positions = np.asarray([
                calendar_position[date] for date in ticker_dates if date in calendar_position
            ])
            maximum_internal_gap = int(np.diff(positions).max() - 1) if len(positions) > 1 else 0
        else:
            leading_gap = trailing_gap = maximum_internal_gap = int(len(expected))
        usable_rows = int(counts.get(ticker, 0))
        coverage_diagnostics[ticker] = {
            "complete_rows": usable_rows,
            "leading_calendar_gap_days": leading_gap,
            "trailing_calendar_gap_days": trailing_gap,
            "maximum_internal_calendar_gap_days": maximum_internal_gap,
        }
        if (
            ticker not in forced_excluded
            and usable_rows >= minimum_total_observations
            and leading_gap <= maximum_calendar_gap_days
            and trailing_gap <= maximum_calendar_gap_days
            and maximum_internal_gap <= maximum_calendar_gap_days
        ):
            eligible_tickers.add(ticker)

    valid_prices = valid_prices[valid_prices["ticker"].astype(str).isin(eligible_tickers)].copy()
    if valid_prices.empty:
        raise ValueError("No securities satisfy the complete-case eligibility criteria.")
    restricted_master = relevant[relevant["ticker"].isin(eligible_tickers)].copy()
    price_only = eligible_tickers - set(restricted_master["ticker"])
    if price_only:
        raise ValueError(
            "Usable prices have no verified security-master identity: "
            + ", ".join(sorted(price_only))
        )

    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    digest = hashlib.sha256(
        (sha256_file(prices_path) + str(start_ts.date()) + str(end_ts.date())
         + str(minimum_total_observations) + str(maximum_calendar_gap_days)
         + "|".join(sorted(forced_excluded))).encode("utf-8")
    ).hexdigest()[:10]
    workspace = source_paths.root / "outputs" / "complete_case_workspaces" / f"{stamp}-{digest}"
    target = Paths(workspace)
    target.ensure()
    valid_prices.to_parquet(target.normalized / "prices.parquet", index=False)
    restricted_master.to_parquet(target.normalized / "security_master.parquet", index=False)

    actions_path = source_paths.normalized / "corporate_actions.parquet"
    if actions_path.exists():
        shutil.copy2(actions_path, target.normalized / "corporate_actions.parquet")

    excluded_rows = []
    observed_counts = prices.groupby("ticker")["date"].nunique()
    for row in relevant.itertuples():
        ticker = str(row.ticker)
        if ticker in eligible_tickers:
            continue
        observed = int(observed_counts.get(ticker, 0))
        diagnostic = coverage_diagnostics.get(ticker, {})
        usable = int(diagnostic.get("complete_rows", 0))
        if ticker in forced_excluded:
            reason = "forced_quality_exclusion"
        elif observed == 0:
            reason = "no_price_observations_in_requested_period"
        elif usable < minimum_total_observations:
            reason = "fewer_than_minimum_complete_observations"
        elif diagnostic.get("leading_calendar_gap_days", 0) > maximum_calendar_gap_days:
            reason = "unexplained_leading_calendar_gap"
        elif diagnostic.get("trailing_calendar_gap_days", 0) > maximum_calendar_gap_days:
            reason = "unexplained_trailing_calendar_gap"
        elif diagnostic.get("maximum_internal_calendar_gap_days", 0) > maximum_calendar_gap_days:
            reason = "unexplained_internal_calendar_gap"
        else:
            reason = "failed_complete_case_contract"
        excluded_rows.append({
            "ticker": ticker,
            "observed_rows": observed,
            "complete_rows": usable,
            "minimum_required": minimum_total_observations,
            "leading_calendar_gap_days": diagnostic.get("leading_calendar_gap_days"),
            "trailing_calendar_gap_days": diagnostic.get("trailing_calendar_gap_days"),
            "maximum_internal_calendar_gap_days": diagnostic.get(
                "maximum_internal_calendar_gap_days"
            ),
            "maximum_allowed_calendar_gap_days": maximum_calendar_gap_days,
            "reason": reason,
        })
    exclusions = pd.DataFrame(excluded_rows, columns=[
        "ticker", "observed_rows", "complete_rows", "minimum_required",
        "leading_calendar_gap_days", "trailing_calendar_gap_days",
        "maximum_internal_calendar_gap_days", "maximum_allowed_calendar_gap_days", "reason",
    ]).sort_values("ticker")
    exclusions.to_csv(target.reports / "complete_case_exclusions.csv", index=False)

    manifest = {
        "dataset_kind": "exploratory_complete_case_real_prices",
        "label": "EXPLORATORY ONLY - RESTRICTED OBSERVED HOSE PANEL",
        "created_at": datetime.now(timezone.utc).isoformat(),
        "requested_start": str(start_ts.date()),
        "requested_end": str(end_ts.date()),
        "minimum_total_observations": int(minimum_total_observations),
        "maximum_calendar_gap_days": int(maximum_calendar_gap_days),
        "forced_quality_exclusions": sorted(forced_excluded),
        "source_price_dataset": str(prices_path),
        "source_price_dataset_sha256": sha256_file(prices_path),
        "source_security_master": str(master_path),
        "source_security_master_sha256": sha256_file(master_path),
        "records_before_row_filter": int(len(prices)),
        "records_failing_complete_row_contract": int((~complete).sum()),
        "records_retained": int(len(valid_prices)),
        "tickers_observed": int(prices["ticker"].nunique()),
        "tickers_retained": int(valid_prices["ticker"].nunique()),
        "relevant_master_tickers": int(relevant["ticker"].nunique()),
        "relevant_tickers_excluded": int(len(exclusions)),
        "excluded_tickers": exclusions["ticker"].tolist(),
        "price_dataset_sha256": sha256_file(target.normalized / "prices.parquet"),
        "security_master_sha256": sha256_file(target.normalized / "security_master.parquet"),
        "selection_rule": (
            "complete OHLCV/provenance row contract and at least "
            f"{minimum_total_observations} observations with no unexplained leading, "
            f"trailing or internal calendar gap above {maximum_calendar_gap_days} sessions"
        ),
        "limitations": [
            "full_period_availability_filter_can_create_coverage_or_survivorship_selection_bias",
            "corporate_action_adjustment_policy_is_not_verified",
            "no_verified_total_return_benchmark",
            "optional_fundamental_macro_and_foreign_flow_features_are_not_used",
            "results_must_not_be_labeled_confirmatory_or_full_hose_research",
        ],
    }
    (target.raw / "manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    (target.reports / "COMPLETE_CASE_DATASET.md").write_text(
        "# Exploratory complete-case HOSE dataset\n\n"
        f"- Period: `{manifest['requested_start']}` to `{manifest['requested_end']}`\n"
        f"- Retained: **{manifest['tickers_retained']} tickers / "
        f"{manifest['records_retained']:,} rows**\n"
        f"- Excluded relevant tickers: **{manifest['relevant_tickers_excluded']}**\n"
        f"- Minimum observations: **{minimum_total_observations}**\n\n"
        f"- Maximum unexplained calendar gap: **{maximum_calendar_gap_days} sessions**\n\n"
        "This dataset is suitable for an exploratory end-to-end run. It is not a "
        "replacement for a verified point-in-time, corporate-action-adjusted, "
        "full-HOSE research panel. See `complete_case_exclusions.csv` and the "
        "limitations in `outputs/raw/manifest.json`.\n",
        encoding="utf-8",
    )
    return workspace, manifest


def promote_staged_file(paths: Paths, staged_file: Path, target_name: str) -> dict:
    """Atomically promote one validated file and retain a recoverable previous copy."""
    if not staged_file.is_file():
        raise FileNotFoundError(staged_file)
    paths.normalized.mkdir(parents=True, exist_ok=True)
    target = paths.normalized / target_name
    archive = paths.root / "outputs" / "archive" / datetime.now().strftime("%Y%m%dT%H%M%S")
    backup = None
    if target.exists():
        archive.mkdir(parents=True, exist_ok=True)
        backup = archive / target.name
        shutil.copy2(target, backup)
    temporary = target.with_name(f".{target.name}.promoting")
    shutil.copy2(staged_file, temporary)
    temporary.replace(target)
    return {
        "target": str(target), "sha256": sha256_file(target),
        "backup": str(backup) if backup else None,
    }


def quarantine_fixture_auxiliary(paths: Paths) -> list[str]:
    """Move stale fixture-only auxiliary tables out of a real-data workspace."""
    names = ["index_membership", "corporate_actions", "financial_statements", "macro", "foreign_flow"]
    fixture_paths: list[Path] = []
    for name in names:
        path = paths.normalized / f"{name}.parquet"
        if not path.exists():
            continue
        table = pd.read_parquet(path)
        if table.empty:
            continue
        fixture = bool(
            ("data_class" in table and table["data_class"].astype(str).eq("fixture").all())
            or ("source" in table and table["source"].astype(str).str.contains(
                "fixture", case=False, na=False
            ).all())
        )
        if fixture:
            fixture_paths.append(path)
    if not fixture_paths:
        return []
    stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
    destination = paths.root / "outputs" / "quarantine" / "fixture_auxiliary" / stamp
    destination.mkdir(parents=True, exist_ok=False)
    moved = []
    for path in fixture_paths:
        target = destination / path.name
        path.replace(target)
        moved.append(str(target))
    return moved


def apply_price_adjustment_contract(paths: Paths, contract_path: Path) -> dict:
    """Apply a documented adjustment policy only to the exact certified price panel."""
    contract = json.loads(contract_path.read_text(encoding="utf-8"))
    required = {
        "price_dataset_sha256", "adjustment_policy", "source", "source_url",
        "methodology", "certified_by", "certified_at",
    }
    missing = sorted(required - set(contract))
    if missing:
        raise ValueError(f"Price adjustment contract missing fields: {missing}")
    if contract["adjustment_policy"] not in VERIFIED_ADJUSTMENT_POLICIES - {"fixture"}:
        raise ValueError("The adjustment policy is not accepted for a real research run.")
    prices_path = paths.normalized / "prices.parquet"
    before_hash = sha256_file(prices_path)
    if contract["price_dataset_sha256"] != before_hash:
        raise ValueError(
            "Price dataset hash does not match the contract; refusing to certify another panel."
        )
    prices = pd.read_parquet(prices_path)
    if prices["data_class"].astype(str).eq("fixture").any():
        raise ValueError("A real adjustment contract cannot certify fixture prices.")
    prices["adjustment_policy"] = contract["adjustment_policy"]
    prices.to_parquet(prices_path, index=False)
    stored = {
        **contract,
        "contract_file_sha256": sha256_file(contract_path),
        "input_price_dataset_sha256": before_hash,
        "output_price_dataset_sha256": sha256_file(prices_path),
        "applied_at": datetime.now(timezone.utc).isoformat(),
    }
    output = paths.normalized / "price_adjustment_contract.json"
    output.write_text(json.dumps(stored, indent=2), encoding="utf-8")
    return stored


def generate_fixture(
    paths: Paths,
    start: str,
    end: str,
    tickers: list[str],
    seed: int = 42,
) -> dict:
    """Create deterministic synthetic market data, explicitly marked as fixture."""
    paths.ensure()
    dates = pd.bdate_range(start, end)
    rng = np.random.default_rng(seed)
    market = rng.normal(0.00025, 0.009, len(dates))
    frames = []
    master = []
    for i, ticker in enumerate(tickers):
        beta = 0.7 + 0.12 * i
        alpha = (i - len(tickers) / 2) * 0.000015
        ret = alpha + beta * market + rng.normal(0, 0.006 + i * 0.00035, len(dates))
        close = (30 + 4 * i) * np.exp(np.cumsum(ret))
        open_ = close * (1 + rng.normal(0, 0.002, len(dates)))
        spread = np.abs(rng.normal(0.005, 0.002, len(dates)))
        high = np.maximum(open_, close) * (1 + spread)
        low = np.minimum(open_, close) * (1 - spread)
        volume = rng.integers(50_000, 2_000_000, len(dates))
        frame = pd.DataFrame({
            "date": dates,
            "ticker": ticker,
            "security_id": f"FIXTURE:{ticker}",
            "open": open_,
            "high": high,
            "low": low,
            "close": close,
            "adjusted_close": close,
            "volume": volume,
            "trading_value": volume * close,
        })
        frames.append(frame)
        master.append({
            "security_id": f"FIXTURE:{ticker}", "ticker": ticker,
            "company_name": f"Fixture Company {ticker}",
            "exchange": "HOSE_FIXTURE", "industry": f"Industry {i % 4}",
            "sector": f"Sector {i % 3}", "listing_date": dates[0],
            "delisting_date": pd.NaT, "effective_from": dates[0],
            "effective_to": pd.NaT, "available_at": dates[0],
            "source": "deterministic_fixture", "data_class": "fixture",
            "source_url": "local://tests/fixture", "fetched_at": pd.Timestamp.utcnow(),
            "raw_checksum": "pending", "history_method": "fixture",
        })
    prices = pd.concat(frames, ignore_index=True)
    raw_csv = paths.raw / "fixture_prices.csv"
    prices.to_csv(raw_csv, index=False)
    checksum = sha256_file(raw_csv)
    now = datetime.now(timezone.utc).isoformat()
    prices["source"] = "deterministic_fixture"
    prices["source_url"] = "local://tests/fixture"
    prices["fetched_at"] = now
    prices["available_at"] = prices["date"]
    prices["raw_checksum"] = checksum
    prices["parser_version"] = "fixture-v1"
    prices["data_class"] = "fixture"
    prices["adjustment_policy"] = "fixture"
    prices.to_parquet(paths.normalized / "prices.parquet", index=False)
    (paths.normalized / "price_adjustment_contract.json").write_text(json.dumps({
        "adjustment_policy": "fixture",
        "source": "deterministic_fixture", "source_url": "local://tests/fixture",
        "methodology": "fixture prices require no corporate-action adjustment",
        "certified_by": "fixture_generator", "certified_at": now,
        "output_price_dataset_sha256": sha256_file(paths.normalized / "prices.parquet"),
    }, indent=2), encoding="utf-8")
    master_df = pd.DataFrame(master)
    master_df["raw_checksum"] = checksum
    master_df.to_parquet(paths.normalized / "security_master.parquet", index=False)
    pd.DataFrame(columns=[
        "ticker", "event_type", "event_date", "announcement_date", "effective_date",
        "available_at", "cash_dividend", "split_ratio", "adjustment_factor", "source",
    ]).to_parquet(paths.normalized / "corporate_actions.parquet", index=False)
    pd.DataFrame([
        {
            "ticker": ticker, "index_code": "VN30_FIXTURE",
            "effective_from": dates[0], "effective_to": dates[-1],
            "announcement_date": dates[0], "available_at": dates[0],
            "source": "deterministic_fixture", "data_class": "fixture",
            "source_url": "local://tests/fixture", "fetched_at": now,
            "raw_checksum": checksum, "history_method": "fixture",
        }
        for ticker in tickers[: min(4, len(tickers))]
    ]).to_parquet(paths.normalized / "index_membership.parquet", index=False)
    quarter_ends = pd.date_range(dates[0], dates[-1], freq="QE")
    statements = []
    for i, ticker in enumerate(tickers):
        for quarter in quarter_ends:
            publication = quarter + pd.Timedelta(days=35)
            statements.append({
                "ticker": ticker, "fiscal_period_end": quarter,
                "publication_date": publication, "available_at": publication,
                "revenue": float(1000 + i * 50), "net_income": float(80 + i * 5),
                "total_assets": float(2000 + i * 100), "equity": float(1000 + i * 60),
                "source": "deterministic_fixture", "data_class": "fixture",
            })
    pd.DataFrame(statements).to_parquet(
        paths.normalized / "financial_statements.parquet", index=False
    )
    month_ends = pd.date_range(dates[0], dates[-1], freq="ME")
    pd.DataFrame({
        "series_id": "FIXTURE_POLICY_RATE", "observation_date": month_ends,
        "release_date": month_ends + pd.Timedelta(days=5),
        "available_at": month_ends + pd.Timedelta(days=5), "value": 0.04,
        "source": "deterministic_fixture", "data_class": "fixture",
    }).to_parquet(paths.normalized / "macro.parquet", index=False)
    pd.DataFrame({
        "date": dates, "ticker": "MARKET", "available_at": dates,
        "foreign_net_value": rng.normal(0, 1e9, len(dates)),
        "source": "deterministic_fixture", "data_class": "fixture",
    }).to_parquet(paths.normalized / "foreign_flow.parquet", index=False)
    manifest = {
        "status": "success", "data_class": "fixture", "label": "NOT RESEARCH RESULT",
        "records": len(prices), "tickers": len(tickers), "start": str(dates.min().date()),
        "end": str(dates.max().date()), "source": "deterministic_fixture",
        "raw_checksum": checksum, "generated_at": now,
    }
    (paths.raw / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return manifest


def import_csv(paths: Paths, input_path: Path, source: str, source_url: str) -> dict:
    """Import a user-authorized CSV source without guessing or scraping endpoints."""
    paths.ensure()
    df = pd.read_csv(input_path)
    required = {"date", "ticker", "open", "high", "low", "close", "volume"}
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    checksum = sha256_file(input_path)
    df["date"] = pd.to_datetime(df["date"])
    df["security_id"] = df.get("security_id", df["ticker"].map(lambda x: f"TICKER:{x}"))
    df["adjusted_close"] = df.get("adjusted_close", df["close"])
    df["trading_value"] = df.get("trading_value", df["volume"] * df["close"])
    df["source"] = source
    df["source_url"] = source_url
    df["fetched_at"] = datetime.now(timezone.utc).isoformat()
    df["available_at"] = pd.to_datetime(df.get("available_at", df["date"]))
    df["raw_checksum"] = checksum
    df["parser_version"] = "csv-v1"
    df["data_class"] = "real"
    # Research mode accepts only an explicitly documented adjustment contract.
    # A missing declaration remains usable for inspection but fails the research audit.
    df["adjustment_policy"] = df.get("adjustment_policy", "unverified")
    df[PRICE_COLUMNS].to_parquet(paths.normalized / "prices.parquet", index=False)
    return {"records": len(df), "data_class": "real", "raw_checksum": checksum}


def validate_data(paths: Paths) -> tuple[dict, pd.DataFrame]:
    prices = pd.read_parquet(paths.normalized / "prices.parquet")
    issues: list[dict] = []
    required_price_columns = set(PRICE_COLUMNS)
    missing_price_columns = sorted(required_price_columns - set(prices.columns))
    if missing_price_columns:
        issues.append({
            "severity": "error", "check": "required_price_schema",
            "count": len(missing_price_columns), "columns": missing_price_columns,
        })
    if "security_id" not in prices:
        # Continue the diagnostic pass, but do not silently manufacture research identity.
        prices["security_id"] = pd.NA
    duplicated = prices.duplicated(["date", "ticker"], keep=False)
    if duplicated.any():
        issues.append({"severity": "error", "check": "unique_date_ticker", "count": int(duplicated.sum())})
    bad_ohlc = (
        (prices["high"] < prices[["open", "close", "low"]].max(axis=1))
        | (prices["low"] > prices[["open", "close", "high"]].min(axis=1))
        | (prices[["open", "high", "low", "close"]] <= 0).any(axis=1)
    )
    if bad_ohlc.any():
        issues.append({"severity": "error", "check": "ohlc_logic", "count": int(bad_ohlc.sum())})
    negative = (prices[["volume", "trading_value"]] < 0).any(axis=1)
    if negative.any():
        issues.append({"severity": "error", "check": "nonnegative_volume_value", "count": int(negative.sum())})
    future = pd.to_datetime(prices["available_at"]) < pd.to_datetime(prices["date"])
    if future.any():
        issues.append({"severity": "error", "check": "available_before_observation", "count": int(future.sum())})
    ordered = prices.sort_values(["ticker", "date"]).copy()
    ordered["return_1d"] = ordered.groupby("ticker")["adjusted_close"].pct_change()
    outlier_rows = ordered[ordered["return_1d"].abs() > 0.30].copy()
    if not outlier_rows.empty:
        actions_path = paths.normalized / "corporate_actions.parquet"
        actions = pd.read_parquet(actions_path) if actions_path.exists() else pd.DataFrame()
        if not actions.empty and {"ticker", "effective_date"} <= set(actions.columns):
            actions["effective_date"] = pd.to_datetime(actions["effective_date"], errors="coerce")
            action_dates = {
                ticker: list(group["effective_date"].dropna())
                for ticker, group in actions.groupby("ticker")
            }
            outlier_rows["corporate_action_match"] = [
                any(abs((pd.Timestamp(date) - action_date).days) <= 3
                    for action_date in action_dates.get(ticker, []))
                for ticker, date in zip(outlier_rows["ticker"], outlier_rows["date"])
            ]
        else:
            outlier_rows["corporate_action_match"] = False
        contract_path = paths.normalized / "price_adjustment_contract.json"
        contract = json.loads(contract_path.read_text(encoding="utf-8")) if contract_path.exists() else {}
        contract_matches_dataset = bool(
            contract.get("output_price_dataset_sha256") == sha256_file(
                paths.normalized / "prices.parquet"
            )
        )
        outlier_rows["adjustment_contract_verified"] = contract_matches_dataset & outlier_rows.get(
            "adjustment_policy", pd.Series("unverified", index=outlier_rows.index)
        ).astype(str).isin(VERIFIED_ADJUSTMENT_POLICIES)
        outlier_rows["resolution"] = np.select(
            [outlier_rows["corporate_action_match"], outlier_rows["adjustment_contract_verified"]],
            ["verified_corporate_action", "verified_vendor_adjustment"],
            default="unresolved",
        )
        ledger_path = paths.reports / "outlier_resolution_ledger.csv"
        if ledger_path.exists():
            ledger = pd.read_csv(ledger_path)
            ledger["date"] = pd.to_datetime(ledger.get("date"), errors="coerce")
            allowed = {
                "verified_corporate_action", "verified_cross_source_correction",
                "verified_vendor_adjustment", "unresolved", "genuine_market_move",
            }
            required_ledger = {"ticker", "date", "resolution", "reviewer_status", "source_url"}
            if required_ledger <= set(ledger.columns):
                ledger = ledger[
                    ledger["resolution"].isin(allowed)
                    & ledger["reviewer_status"].astype(str).isin({"verified", "rejected", "pending"})
                ].drop_duplicates(["ticker", "date"], keep="last")
                outlier_rows = outlier_rows.merge(
                    ledger[["ticker", "date", "resolution", "reviewer_status", "source_url"]]
                    .rename(columns={
                        "resolution": "ledger_resolution", "source_url": "ledger_source_url",
                    }), on=["ticker", "date"], how="left",
                )
                verified_ledger = (
                    outlier_rows["reviewer_status"].eq("verified")
                    & outlier_rows["ledger_resolution"].isin(allowed - {"unresolved"})
                    & outlier_rows["ledger_source_url"].astype(str).str.startswith(("http://", "https://"))
                )
                outlier_rows.loc[verified_ledger, "resolution"] = outlier_rows.loc[
                    verified_ledger, "ledger_resolution"
                ]
        unresolved = int(outlier_rows["resolution"].eq("unresolved").sum())
        issues.append({
            "severity": "error" if unresolved else "warning",
            "check": "unresolved_return_outlier" if unresolved else "return_outlier_reviewed",
            "count": unresolved if unresolved else len(outlier_rows),
        })
    outlier_columns = [
        "date", "ticker", "close", "adjusted_close", "return_1d",
        "adjustment_policy", "corporate_action_match",
        "adjustment_contract_verified", "resolution", "reviewer_status",
        "ledger_source_url", "source", "source_url",
    ]
    paths.reports.mkdir(parents=True, exist_ok=True)
    outlier_rows.reindex(columns=outlier_columns).to_csv(
        paths.reports / "return_outlier_review.csv", index=False
    )
    coverage = (
        prices.assign(year=pd.to_datetime(prices["date"]).dt.year)
        .groupby(["ticker", "year", "source", "data_class"], dropna=False)
        .agg(records=("date", "size"), start=("date", "min"), end=("date", "max"),
             missing_close=("close", lambda s: int(s.isna().sum())))
        .reset_index()
    )
    report = {
        "status": "pass" if not any(i["severity"] == "error" for i in issues) else "fail",
        "data_class": sorted(prices["data_class"].astype(str).unique().tolist()),
        "records": len(prices), "tickers": prices["ticker"].nunique(),
        "start": str(pd.to_datetime(prices["date"]).min().date()),
        "end": str(pd.to_datetime(prices["date"]).max().date()),
        "issues": issues,
    }
    (paths.reports / "data_quality.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    coverage.to_csv(paths.reports / "coverage.csv", index=False)
    return report, coverage


def build_universe(
    paths: Paths,
    rebalance: str = "monthly",
    definition: str = "hose_all_listed",
    index_code: str | None = None,
    max_assets: int | None = None,
    liquidity_lookback_days: int = 60,
    minimum_observations: int = 40,
) -> pd.DataFrame:
    """Build auditable point-in-time snapshots for one declared universe definition.

    ``hose_all_listed`` uses exchange listing/delisting event history from the security
    master. ``index_membership`` uses effective membership intervals and therefore must
    never be substituted for an all-HOSE study (or vice versa).
    """
    if definition not in UNIVERSE_DEFINITIONS:
        raise ValueError(f"Unsupported universe definition: {definition}")
    prices = pd.read_parquet(paths.normalized / "prices.parquet")
    master = pd.read_parquet(paths.normalized / "security_master.parquet")
    prices["date"] = pd.to_datetime(prices["date"])
    prices["available_at"] = pd.to_datetime(prices["available_at"], errors="coerce")
    if "security_id" not in prices:
        prices["security_id"] = prices["ticker"]
    if "security_id" not in master:
        master["security_id"] = master["ticker"]
    master["listing_date"] = pd.to_datetime(master["listing_date"])
    master["delisting_date"] = pd.to_datetime(master["delisting_date"])
    master["available_at"] = pd.to_datetime(master["available_at"])
    master["effective_from"] = pd.to_datetime(master.get("effective_from", master["listing_date"]))
    master["effective_to"] = pd.to_datetime(master.get("effective_to", master["delisting_date"]))
    anchors = (
        prices.set_index("date").groupby("ticker")["close"].resample("ME").last().dropna()
        .reset_index()["date"].drop_duplicates().sort_values()
    )
    if definition == "index_membership":
        membership_path = paths.normalized / "index_membership.parquet"
        if not membership_path.exists():
            raise ValueError("index_membership universe requires index_membership.parquet")
        records = pd.read_parquet(membership_path).copy()
        if index_code:
            records = records[records["index_code"].astype(str) == str(index_code)]
        if records.empty:
            raise ValueError("No membership records match the requested index universe.")
        for column in ["effective_from", "effective_to", "available_at"]:
            records[column] = pd.to_datetime(records[column], errors="coerce")
        reason = "index_member_and_available_point_in_time"
    else:
        records = master.copy()
        reason = "listed_on_hose_and_available_point_in_time"
    rows = []
    audit_rows = []
    for date in anchors:
        eligible_records = []
        for row in records.itertuples():
            start = row.effective_from
            end = row.effective_to
            available = row.available_at
            if definition == "hose_all_listed":
                start = max(row.listing_date, start)
                ends = [value for value in (row.delisting_date, end) if pd.notna(value)]
                end = min(ends) if ends else pd.NaT
            eligible = (
                pd.notna(start) and pd.notna(available)
                and start <= date and available <= date
                and (pd.isna(end) or end >= date)
            )
            if eligible:
                eligible_records.append({
                    "decision_time": date, "ticker": row.ticker, "eligible": True,
                    "security_id": getattr(row, "security_id", row.ticker),
                    "reason": reason, "universe_definition": definition,
                    "index_code": getattr(row, "index_code", None),
                    "data_class": getattr(row, "data_class", "real"),
                    "effective_from": start, "effective_to": end,
                    "record_available_at": available, "source": row.source,
                    "source_url": getattr(row, "source_url", None),
                    "fetched_at": getattr(row, "fetched_at", None),
                    "raw_checksum": getattr(row, "raw_checksum", None),
                    "history_method": getattr(row, "history_method", None),
                })
            else:
                audit_rows.append({
                    "decision_time": date, "ticker": row.ticker,
                    "security_id": getattr(row, "security_id", row.ticker),
                    "included": False, "reason": "outside_verified_listing_interval_or_not_yet_available",
                    "trailing_observations": 0, "trailing_liquidity": np.nan,
                })
        eligible_frame = pd.DataFrame(eligible_records)
        if eligible_frame.empty:
            continue
        start_lookback = date - pd.offsets.BDay(max(1, liquidity_lookback_days))
        history = prices[
            (prices["date"] <= date)
            & (prices["date"] >= start_lookback)
            & (prices["available_at"] <= date)
            & prices["ticker"].isin(eligible_frame["ticker"])
        ]
        liquidity = history.groupby("ticker").agg(
            trailing_observations=("date", "nunique"),
            trailing_liquidity=("trading_value", "mean"),
        )
        eligible_frame = eligible_frame.merge(liquidity, on="ticker", how="left")
        eligible_frame["trailing_observations"] = eligible_frame["trailing_observations"].fillna(0).astype(int)
        enough_history = eligible_frame["trailing_observations"] >= minimum_observations
        ranked = eligible_frame[enough_history].sort_values(
            ["trailing_liquidity", "ticker"], ascending=[False, True], na_position="last"
        )
        if max_assets is not None:
            ranked = ranked.head(int(max_assets))
        selected = set(ranked["ticker"])
        for item in eligible_frame.itertuples():
            included = item.ticker in selected
            if included:
                exclusion_reason = "selected_by_trailing_liquidity_point_in_time"
            elif item.trailing_observations < minimum_observations:
                exclusion_reason = "insufficient_trailing_observations"
            else:
                exclusion_reason = "outside_dynamic_top_n_liquidity"
            audit_rows.append({
                "decision_time": date, "ticker": item.ticker,
                "security_id": item.security_id, "included": included,
                "reason": exclusion_reason,
                "trailing_observations": item.trailing_observations,
                "trailing_liquidity": item.trailing_liquidity,
            })
        rows.extend(ranked.to_dict("records"))
    universe = pd.DataFrame(rows)
    if universe.empty:
        raise ValueError("The declared point-in-time universe produced no eligible snapshots.")
    paths.curated.mkdir(parents=True, exist_ok=True)
    universe.to_parquet(paths.curated / "universe_monthly.parquet", index=False)
    eligibility_audit = pd.DataFrame(audit_rows)
    eligibility_audit.to_parquet(paths.curated / "universe_eligibility_audit.parquet", index=False)
    eligibility_audit.to_csv(paths.curated / "universe_eligibility_audit.csv", index=False)
    (paths.curated / "universe_contract.json").write_text(json.dumps({
        "definition": definition, "index_code": index_code, "rebalance": rebalance,
        "max_assets": max_assets, "liquidity_lookback_days": liquidity_lookback_days,
        "minimum_observations": minimum_observations,
        "selection_information_cutoff": "price.available_at <= decision_time",
        "snapshot_rows": len(universe), "tickers": int(universe["ticker"].nunique()),
        "start": str(pd.to_datetime(universe["decision_time"]).min().date()),
        "end": str(pd.to_datetime(universe["decision_time"]).max().date()),
        "built_at": datetime.now(timezone.utc).isoformat(),
    }, indent=2), encoding="utf-8")
    return universe


def leakage_audit(paths: Paths) -> dict:
    prices = pd.read_parquet(paths.normalized / "prices.parquet")
    master = pd.read_parquet(paths.normalized / "security_master.parquet")
    actions_path = paths.normalized / "corporate_actions.parquet"
    actions = pd.read_parquet(actions_path) if actions_path.exists() else pd.DataFrame()
    membership_path = paths.normalized / "index_membership.parquet"
    membership = pd.read_parquet(membership_path) if membership_path.exists() else pd.DataFrame()
    universe_metadata_path = paths.curated / "universe_contract.json"
    universe_metadata = (
        json.loads(universe_metadata_path.read_text(encoding="utf-8"))
        if universe_metadata_path.exists() else {}
    )
    universe_definition = universe_metadata.get("definition")
    auxiliary_names = [
        "index_membership", "corporate_actions", "financial_statements",
        "macro", "foreign_flow",
    ]
    auxiliary = {}
    fixture_auxiliary = []
    for name in auxiliary_names:
        path = paths.normalized / f"{name}.parquet"
        if not path.exists():
            auxiliary[name] = "missing"
            continue
        table = pd.read_parquet(path)
        is_fixture = bool(
            ("data_class" in table and table["data_class"].astype(str).eq("fixture").any())
            or ("source" in table and table["source"].astype(str).str.contains("fixture", case=False).any())
        )
        auxiliary[name] = "fixture" if is_fixture else "real_or_empty"
        if is_fixture:
            fixture_auxiliary.append(name)
    master_has_contract = bool(
        {"security_id", "listing_date", "delisting_date", "effective_from", "effective_to", "available_at",
         "history_method"} | PROVENANCE_COLUMNS <= set(master.columns)
    )
    history_methods = set(master.get("history_method", pd.Series(dtype=str)).dropna().astype(str))
    trusted_history = bool(history_methods) and history_methods <= HISTORICAL_UNIVERSE_METHODS
    master_times_valid = False
    if {"listing_date", "effective_from", "available_at"} <= set(master.columns):
        listing = pd.to_datetime(master["listing_date"], errors="coerce")
        effective = pd.to_datetime(master["effective_from"], errors="coerce")
        available = pd.to_datetime(master["available_at"], errors="coerce")
        master_times_valid = bool(
            listing.notna().all() and effective.notna().all() and available.notna().all()
        )
    universe_path = paths.curated / "universe_monthly.parquet"
    universe = pd.read_parquet(universe_path) if universe_path.exists() else pd.DataFrame()
    universe_contract = bool(
        not universe.empty
        and {"decision_time", "ticker", "eligible", "effective_from", "record_available_at",
             "source", "source_url", "fetched_at", "raw_checksum", "history_method"}
        <= set(universe.columns)
    )
    universe_times_valid = False
    universe_methods: set[str] = set()
    universe_fixture_free = False
    if universe_contract:
        decision = pd.to_datetime(universe["decision_time"], errors="coerce")
        effective = pd.to_datetime(universe["effective_from"], errors="coerce")
        available = pd.to_datetime(universe["record_available_at"], errors="coerce")
        universe_times_valid = bool(
            decision.notna().all() and effective.notna().all() and available.notna().all()
            and (effective <= decision).all() and (available <= decision).all()
        )
        universe_methods = set(universe["history_method"].dropna().astype(str))
        universe_fixture_free = not universe.get(
            "data_class", pd.Series(dtype=str)
        ).astype(str).eq("fixture").any()
    membership_contract = bool(
        not membership.empty
        and {"ticker", "effective_from", "effective_to", "available_at",
             "history_method"} | PROVENANCE_COLUMNS <= set(membership.columns)
        and not membership.get("data_class", pd.Series(dtype=str)).astype(str).eq("fixture").any()
        and set(membership["history_method"].dropna().astype(str)) <= HISTORICAL_UNIVERSE_METHODS
    )
    actions_contract = bool(
        not actions.empty
        and {"ticker", "security_id", "announcement_date", "effective_date", "available_at"}
        | PROVENANCE_COLUMNS <= set(actions.columns)
    )
    price_times_valid = bool(
        (pd.to_datetime(prices["available_at"], errors="coerce")
         >= pd.to_datetime(prices["date"], errors="coerce")).all()
    )
    adjustment_policies = set(
        prices.get("adjustment_policy", pd.Series(dtype=str)).dropna().astype(str)
    )
    adjustment_contract_path = paths.normalized / "price_adjustment_contract.json"
    adjustment_contract = (
        json.loads(adjustment_contract_path.read_text(encoding="utf-8"))
        if adjustment_contract_path.exists() else {}
    )
    adjustment_contract_required = {
        "adjustment_policy", "source", "source_url", "methodology", "certified_by",
        "certified_at", "output_price_dataset_sha256",
    }
    adjustment_contract_valid = bool(
        adjustment_contract
        and adjustment_contract_required <= set(adjustment_contract)
        and adjustment_contract.get("adjustment_policy") in adjustment_policies
        and adjustment_contract.get("output_price_dataset_sha256")
        == sha256_file(paths.normalized / "prices.parquet")
    )
    adjustment_verified = bool(
        adjustment_policies
        and adjustment_policies <= VERIFIED_ADJUSTMENT_POLICIES
        and adjustment_contract_valid
    )
    actions_required = "unadjusted_with_verified_actions_join" in adjustment_policies
    declared_universe_valid = universe_definition in UNIVERSE_DEFINITIONS
    membership_required = universe_definition == "index_membership"
    price_start = pd.to_datetime(prices["date"], errors="coerce").min()
    price_end = pd.to_datetime(prices["date"], errors="coerce").max()
    master_listing = pd.to_datetime(master.get("listing_date"), errors="coerce")
    master_delisting = pd.to_datetime(master.get("delisting_date"), errors="coerce")
    relevant_master = master[
        master_listing.le(price_end)
        & (master_delisting.isna() | master_delisting.ge(price_start))
    ]
    observed_tickers = set(prices["ticker"].dropna().astype(str))
    required_tickers = set(relevant_master["ticker"].dropna().astype(str))
    missing_universe_prices = sorted(required_tickers - observed_tickers)
    master_security_ids = set(master.get("security_id", pd.Series(dtype=str)).dropna().astype(str))
    price_security_ids = set(prices.get("security_id", pd.Series(dtype=str)).dropna().astype(str))
    unmatched_price_security_ids = sorted(price_security_ids - master_security_ids)
    checks = {
        "universe_definition_declared": declared_universe_valid,
        "historical_universe_contract": master_has_contract,
        "historical_universe_source_trusted": trusted_history,
        "historical_universe_fixture_free": not master.get(
            "data_class", pd.Series(dtype=str)
        ).astype(str).eq("fixture").any(),
        "historical_universe_times_valid": master_times_valid,
        "historical_universe_price_coverage_complete": not missing_universe_prices,
        "price_security_identity_matches_master": not unmatched_price_security_ids,
        "universe_snapshots_built_with_provenance": universe_contract,
        "universe_snapshot_times_valid": universe_times_valid,
        "universe_snapshot_source_trusted": bool(universe_methods)
        and universe_methods <= HISTORICAL_UNIVERSE_METHODS,
        "universe_snapshot_fixture_free": universe_fixture_free,
        "historical_membership_events_available_when_required": (
            membership_contract if membership_required else True
        ),
        "real_prices_only": not prices["data_class"].astype(str).eq("fixture").any(),
        "auxiliary_tables_fixture_free": not fixture_auxiliary,
        "corporate_actions_point_in_time_available_when_required": (
            actions_contract if actions_required else True
        ),
        "price_adjustment_policy_verified": adjustment_verified,
        "price_adjustment_contract_matches_dataset": adjustment_contract_valid,
        "availability_not_before_observation": price_times_valid,
    }
    fixture_only = prices["data_class"].eq("fixture").all()
    blockers = [] if fixture_only else [k for k, v in checks.items() if not v]
    limitations = [
        f"{name}_not_configured"
        for name, status in auxiliary.items()
        if status == "missing"
    ]
    report = {
        "status": (
            "pass_for_fixture_demo" if fixture_only
            else ("blocked" if blockers else ("pass_with_limitations" if limitations else "pass"))
        ),
        "checks": checks, "blockers": blockers,
        "limitations": limitations,
        "auxiliary_tables": auxiliary,
        "universe_definition": universe_definition,
        "index_code": universe_metadata.get("index_code"),
        "adjustment_policies": sorted(adjustment_policies),
        "price_adjustment_contract": adjustment_contract,
        "corporate_actions_required": actions_required,
        "historical_universe_price_coverage": {
            "required_tickers": len(required_tickers),
            "observed_tickers": len(required_tickers & observed_tickers),
            "missing_count": len(missing_universe_prices),
            "missing_tickers": missing_universe_prices,
        },
        "unmatched_price_security_ids": unmatched_price_security_ids,
        "note": (
            "Historical universe/membership, corporate actions and adjustment policy are core "
            "research contracts. Optional PIT features are excluded when unavailable. Fixture "
            "auxiliary tables block a real-data run."
        ),
    }
    (paths.reports / "leakage_audit.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    return report


In [ ]:
%%writefile /content/ai_quantum_standalone/src/research.py
from __future__ import annotations

import hashlib
import itertools
import json
import math
import platform
import shutil
import subprocess
import sys
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from scipy.optimize import minimize
from scipy.stats import spearmanr
from sklearn.covariance import LedoitWolf
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

from .data_pipeline import Paths, build_universe, leakage_audit, sha256_file, validate_data


FEATURES = [
    "return_5d", "return_20d", "return_60d", "return_120d", "sma_ratio_20",
    "ema_ratio_20", "rsi_14", "macd", "atr_14", "volatility_20d",
    "downside_volatility_20d", "drawdown_60d", "liquidity_20d", "beta_60d",
    "roe_pit", "revenue_growth_yoy_pit", "policy_rate_pit",
]


# Data B uses a deliberately small, interpretable technical composite alongside
# XGBoost.  Every component is converted to a same-date cross-sectional percentile
# before blending, so no future observation or arbitrary feature scale can dominate.
TECHNICAL_FACTOR_WEIGHTS = {
    "return_20d": 0.25,
    "return_60d": 0.25,
    "return_120d": 0.15,
    "sma_ratio_20": 0.10,
    "ema_ratio_20": 0.10,
    "volatility_20d": -0.05,
    "downside_volatility_20d": -0.05,
    "drawdown_60d": 0.05,
}


class ResearchRunBlocked(RuntimeError):
    """Raised after an auditable blocked-run artifact has been written."""

    def __init__(self, message: str, output_dir: Path):
        super().__init__(message)
        self.output_dir = output_dir


def load_config(path: Path) -> dict:
    return yaml.safe_load(path.read_text(encoding="utf-8"))


def _rsi(s: pd.Series, n: int = 14) -> pd.Series:
    d = s.diff()
    gain = d.clip(lower=0).rolling(n).mean()
    loss = -d.clip(upper=0).rolling(n).mean()
    return 100 - 100 / (1 + gain / loss.replace(0, np.nan))


def build_features(prices: pd.DataFrame, target_horizon_days: int = 20) -> pd.DataFrame:
    base = prices.sort_values(["ticker", "date"]).copy()
    base["_ret1"] = base.groupby("ticker")["adjusted_close"].pct_change()
    market_return = base.groupby("date")["_ret1"].mean()
    frames = []
    for ticker, g in base.groupby("ticker"):
        x = g.copy().sort_values("date")
        p = x["adjusted_close"].astype(float)
        r = p.pct_change()
        for n in (5, 20, 60, 120):
            x[f"return_{n}d"] = p.pct_change(n)
        x["sma_ratio_20"] = p / p.rolling(20).mean() - 1
        ema12, ema26 = p.ewm(span=12, adjust=False).mean(), p.ewm(span=26, adjust=False).mean()
        x["ema_ratio_20"] = p / p.ewm(span=20, adjust=False).mean() - 1
        x["rsi_14"] = _rsi(p)
        x["macd"] = ema12 - ema26
        # Use an adjusted OHLC scale consistently. Mixing adjusted close with raw
        # high/low creates artificial ATR spikes around splits and rights issues.
        adjustment_factor = p / x["close"].astype(float).replace(0, np.nan)
        adjusted_high = x["high"].astype(float) * adjustment_factor
        adjusted_low = x["low"].astype(float) * adjustment_factor
        tr = pd.concat([
            adjusted_high - adjusted_low,
            (adjusted_high - p.shift()).abs(),
            (adjusted_low - p.shift()).abs(),
        ], axis=1).max(axis=1)
        x["atr_14"] = tr.rolling(14).mean() / p
        x["volatility_20d"] = r.rolling(20).std()
        x["downside_volatility_20d"] = r.where(r < 0, 0).rolling(20).std()
        x["drawdown_60d"] = p / p.rolling(60).max() - 1
        x["adv_20d"] = x["trading_value"].rolling(20).mean()
        x["liquidity_20d"] = np.log1p(x["adv_20d"])
        market = x["date"].map(market_return)
        x["beta_60d"] = r.rolling(60).cov(market) / market.rolling(60).var()
        x["target_return_20d"] = p.shift(-target_horizon_days) / p - 1
        x["label_end_time"] = pd.to_datetime(x["date"]).shift(-target_horizon_days)
        x["target_horizon_days"] = int(target_horizon_days)
        x["target_rank"] = np.nan
        frames.append(x)
    out = pd.concat(frames, ignore_index=True)
    out["target_rank"] = out.groupby("date")["target_return_20d"].rank(pct=True)
    out["feature_available_at"] = pd.to_datetime(
        out.get("available_at", out["date"]), errors="coerce"
    )
    out["roe_pit"] = np.nan
    out["revenue_growth_yoy_pit"] = np.nan
    out["policy_rate_pit"] = np.nan
    return out


def attach_point_in_time_features(features: pd.DataFrame, paths: Paths) -> pd.DataFrame:
    out_frames = []
    financial_path = paths.normalized / "financial_statements.parquet"
    if financial_path.exists():
        financial = pd.read_parquet(financial_path)
        if "usable_for_model" in financial:
            financial = financial[financial["usable_for_model"].fillna(False)].copy()
        required_columns = {
            "ticker", "available_at", "fiscal_period_end", "revenue",
            "net_income", "equity",
        }
        if not financial.empty and required_columns.issubset(financial.columns):
            financial["available_at"] = pd.to_datetime(financial["available_at"], errors="coerce")
            financial["fiscal_period_end"] = pd.to_datetime(
                financial["fiscal_period_end"], errors="coerce"
            )
            financial = financial.dropna(
                subset=["ticker", "available_at", "fiscal_period_end"]
            ).sort_values(["ticker", "fiscal_period_end", "available_at"])
            financial = financial.drop_duplicates(
                ["ticker", "fiscal_period_end", "available_at"], keep="last"
            )
            financial["roe_pit"] = (
                financial["net_income"] / financial["equity"].replace(0, np.nan)
            )
            # Match the same fiscal period one year earlier using only a filing
            # that was already public at the current filing timestamp.  A plain
            # pct_change(4) is invalid when annual, quarterly and revised filings
            # coexist in the same disclosure stream.
            yoy_values = []
            for row in financial.itertuples(index=False):
                previous_period = pd.Timestamp(row.fiscal_period_end) - pd.DateOffset(years=1)
                candidates = financial[
                    financial["ticker"].astype(str).eq(str(row.ticker))
                    & financial["fiscal_period_end"].eq(previous_period)
                    & financial["available_at"].le(pd.Timestamp(row.available_at))
                ]
                if candidates.empty:
                    yoy_values.append(np.nan)
                    continue
                previous_revenue = pd.to_numeric(
                    candidates.sort_values("available_at").iloc[-1]["revenue"],
                    errors="coerce",
                )
                current_revenue = pd.to_numeric(getattr(row, "revenue"), errors="coerce")
                if pd.isna(previous_revenue) or previous_revenue == 0 or pd.isna(current_revenue):
                    yoy_values.append(np.nan)
                else:
                    yoy_values.append(float(current_revenue / previous_revenue - 1.0))
            financial["revenue_growth_yoy_pit"] = yoy_values
            # Multiple statements can become available at the same timestamp.
            # Collapse to one deterministic PIT feature snapshot before as-of joins.
            financial = (
                financial.sort_values(["ticker", "available_at", "fiscal_period_end"])
                .drop_duplicates(["ticker", "available_at"], keep="last")
            )
            for ticker, group in features.groupby("ticker", sort=False):
                right = financial[financial.ticker == ticker][
                    ["available_at", "roe_pit", "revenue_growth_yoy_pit"]
                ].rename(columns={"available_at": "financial_available_at"}).sort_values(
                    "financial_available_at"
                )
                left = group.drop(columns=["roe_pit", "revenue_growth_yoy_pit"]).sort_values(
                    "feature_available_at"
                )
                if right.empty:
                    left["roe_pit"] = np.nan
                    left["revenue_growth_yoy_pit"] = np.nan
                else:
                    left = pd.merge_asof(
                        left, right, left_on="feature_available_at",
                        right_on="financial_available_at", direction="backward",
                        allow_exact_matches=True,
                    ).drop(columns=["financial_available_at"])
                out_frames.append(left)
            features = pd.concat(out_frames, ignore_index=True)
    sector_path = paths.normalized / "sector_pit.parquet"
    if sector_path.exists():
        sector = pd.read_parquet(sector_path)
        sector["available_at"] = pd.to_datetime(sector["available_at"], errors="coerce")
        sector = sector.dropna(subset=["ticker", "sector", "available_at"])
        sector_frames = []
        for ticker, group in features.groupby("ticker", sort=False):
            right = sector[sector["ticker"].astype(str).eq(str(ticker))][
                ["available_at", "sector"]
            ].rename(columns={"available_at": "sector_available_at"}).sort_values(
                "sector_available_at"
            )
            left = group.drop(columns=["sector"], errors="ignore").sort_values(
                "feature_available_at"
            )
            if right.empty:
                left["sector"] = pd.NA
            else:
                left = pd.merge_asof(
                    left,
                    right,
                    left_on="feature_available_at",
                    right_on="sector_available_at",
                    direction="backward",
                    allow_exact_matches=True,
                ).drop(columns=["sector_available_at"])
            sector_frames.append(left)
        features = pd.concat(sector_frames, ignore_index=True)
    macro_path = paths.normalized / "macro.parquet"
    if macro_path.exists():
        macro = pd.read_parquet(macro_path)
        macro["available_at"] = pd.to_datetime(macro["available_at"])
        policy = macro[macro["series_id"].astype(str).str.contains("POLICY_RATE", case=False)].copy()
        if not policy.empty:
            policy = policy.sort_values("available_at")[["available_at", "value"]].rename(
                columns={"available_at": "macro_available_at", "value": "policy_rate_new"}
            )
            features = pd.merge_asof(
                features.sort_values("feature_available_at"), policy,
                left_on="feature_available_at", right_on="macro_available_at", direction="backward",
            ).drop(columns=["macro_available_at"])
            features["policy_rate_pit"] = features["policy_rate_new"]
            features = features.drop(columns=["policy_rate_new"])
    return features


def make_folds(dates: pd.Series, train_months: int, validation_months: int,
               test_months: int, max_folds: int | None,
               embargo_days: int = 0, selection: str = "evenly_spaced",
               final_holdout_months: int = 0) -> list[dict]:
    unique = pd.Series(pd.to_datetime(dates).sort_values().unique())
    first = unique.min() + pd.DateOffset(months=train_months + validation_months)
    final_date = unique.max()
    holdout_start = (
        final_date - pd.DateOffset(months=int(final_holdout_months))
        if final_holdout_months else None
    )
    last = (
        holdout_start - pd.DateOffset(months=test_months)
        if holdout_start is not None
        else final_date - pd.DateOffset(months=test_months)
    )
    anchors = pd.date_range(first, last, freq="ME")
    if max_folds and len(anchors) > max_folds:
        if selection == "first":
            anchors = anchors[:max_folds]
        elif selection == "last":
            anchors = anchors[-max_folds:]
        elif selection == "evenly_spaced":
            anchors = anchors[np.unique(np.linspace(0, len(anchors) - 1, max_folds).round().astype(int))]
        else:
            raise ValueError("fold selection must be first, last, evenly_spaced, or all")
    folds = []
    for i, test_start in enumerate(anchors):
        train_start = test_start - pd.DateOffset(months=train_months + validation_months)
        validation_start = test_start - pd.DateOffset(months=validation_months)
        test_end = test_start + pd.DateOffset(months=test_months)
        folds.append({
            "fold": i, "train_start": train_start, "train_end": validation_start,
            "validation_start": validation_start,
            "validation_end": test_start, "test_start": test_start, "test_end": test_end,
            "embargo_days": int(embargo_days), "phase": "development",
        })
    if holdout_start is not None:
        folds.append({
            "fold": len(folds),
            "train_start": holdout_start - pd.DateOffset(months=train_months + validation_months),
            "train_end": holdout_start - pd.DateOffset(months=validation_months),
            "validation_start": holdout_start - pd.DateOffset(months=validation_months),
            "validation_end": holdout_start,
            "test_start": holdout_start,
            "test_end": final_date,
            "embargo_days": int(embargo_days),
            "phase": "final_holdout",
        })
    return folds


def purged_fold_frames(features: pd.DataFrame, fold: dict) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict]:
    """Create chronological train/validation/test frames with label purging and embargo."""
    embargo = pd.Timedelta(days=int(fold.get("embargo_days", 0)))
    train_raw = features[
        (features.date >= fold["train_start"]) & (features.date < fold["train_end"])
    ].copy()
    val_raw = features[
        (features.date >= fold["validation_start"]) & (features.date < fold["validation_end"])
    ].copy()
    test = features[
        (features.date > fold["test_start"]) & (features.date <= fold["test_end"])
    ].copy()
    train_cutoff = pd.Timestamp(fold["validation_start"]) - embargo
    validation_cutoff = pd.Timestamp(fold["test_start"]) - embargo
    train = train_raw[
        train_raw["label_end_time"].notna() & (train_raw["label_end_time"] < train_cutoff)
        & (train_raw["feature_available_at"] < train_cutoff)
    ].copy()
    validation = val_raw[
        val_raw["label_end_time"].notna() & (val_raw["label_end_time"] < validation_cutoff)
        & (val_raw["feature_available_at"] < validation_cutoff)
    ].copy()
    audit = {
        **fold,
        "train_rows_raw": len(train_raw), "train_rows_after_purge": len(train),
        "train_rows_purged": len(train_raw) - len(train),
        "validation_rows_raw": len(val_raw), "validation_rows_after_purge": len(validation),
        "validation_rows_purged": len(val_raw) - len(validation),
        "test_rows": len(test), "train_label_cutoff": train_cutoff,
        "validation_label_cutoff": validation_cutoff,
    }
    return train, validation, test, audit


def fit_ranker(train: pd.DataFrame, validation: pd.DataFrame, cfg: dict):
    usable = train.dropna(subset=["target_rank"])
    if usable.empty:
        raise ValueError("No purged training labels are available for this fold.")
    coverage = usable[FEATURES].notna().mean()
    threshold = float(cfg.get("min_feature_coverage", 0.05))
    active_features = coverage[coverage >= threshold].index.tolist()
    if not active_features:
        raise ValueError("All features are below the configured fold coverage threshold.")
    imputer = SimpleImputer(strategy="median").fit(usable[active_features])
    scaler = StandardScaler().fit(imputer.transform(usable[active_features]))
    x_train = scaler.transform(imputer.transform(usable[active_features]))
    y_train = usable["target_rank"].to_numpy()
    validation_usable = validation.dropna(subset=["target_rank"])
    tuning_rows = []
    base_estimators = int(cfg["n_estimators"])
    base_depth = int(cfg["max_depth"])
    candidates = (
        [(base_estimators, base_depth)] if not cfg.get("tuning_enabled", True)
        else [
            (max(20, base_estimators // 2), max(2, base_depth - 1)),
            (base_estimators, base_depth),
            (base_estimators, max(2, base_depth + 1)),
        ]
    )
    best = None
    for n_estimators, max_depth in dict.fromkeys(candidates):
        model = XGBRegressor(
            n_estimators=n_estimators, max_depth=max_depth,
            learning_rate=cfg["learning_rate"], objective="reg:squarederror",
            random_state=int(cfg.get("seed", 42)), n_jobs=1,
        )
        model.fit(x_train, y_train)
        if validation_usable.empty:
            validation_ic = np.nan
        else:
            x_val = scaler.transform(imputer.transform(validation_usable[active_features]))
            validation_ic = float(spearmanr(
                model.predict(x_val), validation_usable["target_rank"]
            ).statistic)
        score = validation_ic if np.isfinite(validation_ic) else -np.inf
        tuning_rows.append({
            "n_estimators": n_estimators, "max_depth": max_depth,
            "validation_rank_ic": validation_ic,
        })
        if best is None or score > best[0]:
            best = (score, model, n_estimators, max_depth)
    return {
        "imputer": imputer, "scaler": scaler, "model": best[1],
        "active_features": active_features,
        "feature_coverage": coverage.to_dict(), "tuning": tuning_rows,
        "selected_params": {"n_estimators": best[2], "max_depth": best[3]},
    }


def predict(model_bundle, df: pd.DataFrame) -> np.ndarray:
    return model_bundle["model"].predict(model_bundle["scaler"].transform(
        model_bundle["imputer"].transform(df[model_bundle["active_features"]])
    ))


def calibrate_rank_signal_to_returns(
    model_bundle: dict, calibration: pd.DataFrame, snapshot: pd.DataFrame,
) -> tuple[np.ndarray, dict]:
    """Map the XGBoost rank signal to an ex-ante return vector without test data.

    The ranker predicts cross-sectional ranks. QUBO, however, requires return-scale
    coefficients. A linear calibration is fitted on the purged validation window (or
    purged training data only when validation is unavailable), with target winsorization
    and output clipping to prevent a few observations from dominating the QUBO.
    """
    usable = calibration.dropna(subset=["target_return_20d"]).copy()
    if len(usable) < 20:
        raise ValueError("At least 20 purged calibration observations are required.")
    scores = predict(model_bundle, usable)
    realized = usable["target_return_20d"].astype(float).to_numpy()
    low, high = np.nanquantile(realized, [0.01, 0.99])
    realized = np.clip(realized, low, high)
    design = np.column_stack([np.ones(len(scores)), scores])
    intercept, slope = np.linalg.lstsq(design, realized, rcond=None)[0]
    # A negative validation slope means the learned ranking has inverted out of sample.
    # Preserve that evidence instead of forcing a positive relationship.
    expected = intercept + slope * predict(model_bundle, snapshot)
    expected = np.clip(expected, low, high)
    fitted = design @ np.asarray([intercept, slope])
    ss_total = float(np.sum((realized - realized.mean()) ** 2))
    r_squared = 1 - float(np.sum((realized - fitted) ** 2)) / ss_total if ss_total > 0 else np.nan
    return expected, {
        "method": "purged_validation_linear_rank_to_return",
        "observations": int(len(usable)), "intercept": float(intercept),
        "slope": float(slope), "r_squared": float(r_squared),
        "target_clip_low": float(low), "target_clip_high": float(high),
    }


def adaptive_reduce(
    snapshot: pd.DataFrame, history: pd.DataFrame, cfg: dict,
    previous_candidates: set[str] | None = None,
) -> pd.DataFrame:
    snap = snapshot.copy()
    previous_candidates = set(previous_candidates or set())
    snap["aur_eligible"] = True
    liquidity_quantile = cfg.get("liquidity_floor_quantile")
    risk_quantile = cfg.get("risk_ceiling_quantile")
    expected_return_floor = cfg.get("minimum_expected_return")
    expected_return_column = str(cfg.get(
        "expected_return_column", "optimization_expected_return"
    ))
    expected_mask = pd.Series(True, index=snap.index)
    if expected_return_floor is not None:
        if expected_return_column not in snap:
            raise ValueError(
                f"Adaptive reduction expected-return column is missing: {expected_return_column}"
            )
        expected_mask = pd.to_numeric(
            snap[expected_return_column], errors="coerce"
        ).ge(float(expected_return_floor))
        snap["aur_eligible"] &= expected_mask
    liquidity_mask = pd.Series(True, index=snap.index)
    if liquidity_quantile is not None:
        liquidity_floor = float(snap["liquidity_20d"].quantile(float(liquidity_quantile)))
        liquidity_mask = snap["liquidity_20d"].ge(liquidity_floor)
        snap["aur_eligible"] &= liquidity_mask
    else:
        liquidity_floor = np.nan
    risk_mask = pd.Series(True, index=snap.index)
    if risk_quantile is not None:
        risk_ceiling = float(snap["volatility_20d"].quantile(float(risk_quantile)))
        risk_mask = snap["volatility_20d"].le(risk_ceiling)
        snap["aur_eligible"] &= risk_mask
    else:
        risk_ceiling = np.nan
    eligible = snap[snap["aur_eligible"]].copy()
    cardinality = int(cfg.get("cardinality", 1))
    expected_return_filter_status = "not_configured"
    force_defensive_exposure = False
    if len(eligible) < cardinality:
        if expected_return_floor is not None and cfg.get(
            "require_minimum_expected_return", False
        ):
            positive_count = int(expected_mask.sum())
            if positive_count >= cardinality:
                # Preserve the expected-return floor and relax only secondary
                # liquidity/risk screens.  This rule is deterministic and audited.
                snap["aur_eligible"] = expected_mask
                eligible = snap[snap["aur_eligible"]].copy()
                expected_return_filter_status = "secondary_filters_relaxed"
            elif cfg.get("insufficient_positive_policy", "raise") == "defensive_topk":
                # Do not manufacture positive forecasts.  Keep a tradable candidate
                # set for solver diagnostics, but force the portfolio to the declared
                # defensive equity floor later in the pipeline.
                secondary_mask = liquidity_mask & risk_mask
                snap["aur_eligible"] = secondary_mask if int(secondary_mask.sum()) >= cardinality else True
                eligible = snap[snap["aur_eligible"]].copy()
                expected_return_filter_status = "insufficient_positive_defensive_topk"
                force_defensive_exposure = True
            else:
                raise ValueError(
                    f"Only {positive_count} assets satisfy the declared expected-return "
                    f"floor; cardinality={cardinality}."
                )
        else:
            snap["aur_eligible"] = True
            eligible = snap.copy()
            expected_return_filter_status = "all_filters_relaxed"
        threshold_relaxed = True
    else:
        threshold_relaxed = False
        if expected_return_floor is not None:
            expected_return_filter_status = "enforced"
    z = lambda s: (s - s.mean()) / (s.std(ddof=0) + 1e-12)
    snap["signal_z"] = z(snap["signal"])
    snap["liquidity_z"] = z(snap["liquidity_20d"])
    snap["risk_z"] = z(snap["volatility_20d"])
    snap["base_score"] = (
        cfg["signal_weight"] * snap["signal_z"]
        + cfg["liquidity_weight"] * snap["liquidity_z"]
        - cfg["risk_weight"] * snap["risk_z"]
    )
    snap["was_previous_candidate"] = snap["ticker"].isin(previous_candidates)
    snap["stability_bonus"] = (
        snap["was_previous_candidate"].astype(float)
        * float(cfg.get("stability_weight", 0.0))
    )
    snap["base_score"] = snap["base_score"] + snap["stability_bonus"]
    eligible = snap[snap["aur_eligible"]].copy()
    m_max = min(int(cfg.get("max_candidate_size", cfg["candidate_size"])),
                int(cfg["qubit_budget"]), len(eligible))
    m_min = min(m_max, max(int(cfg.get("min_candidate_size", cfg.get("cardinality", 1))),
                           int(cfg.get("cardinality", 1))))
    selected: list[str] = []
    returns = history.pivot(index="date", columns="ticker", values="ret1").tail(120)
    corr = returns.corr().fillna(0)
    cluster_threshold = float(cfg.get("correlation_cluster_threshold", 1.01))
    parent = {ticker: ticker for ticker in eligible["ticker"].astype(str)}
    def find(item: str) -> str:
        while parent[item] != item:
            parent[item] = parent[parent[item]]
            item = parent[item]
        return item
    def union(left: str, right: str) -> None:
        root_left, root_right = find(left), find(right)
        if root_left != root_right:
            parent[max(root_left, root_right)] = min(root_left, root_right)
    eligible_names = sorted(parent)
    for left_index, left in enumerate(eligible_names):
        for right in eligible_names[left_index + 1:]:
            value = abs(float(corr.at[left, right])) if left in corr.index and right in corr.columns else 0.0
            if value >= cluster_threshold:
                union(left, right)
    cluster_map = {ticker: find(ticker) for ticker in eligible_names}
    snap["correlation_cluster"] = snap["ticker"].astype(str).map(cluster_map).fillna("INELIGIBLE")
    upper = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack()
    average_abs_correlation = float(upper.abs().mean()) if len(upper) else 0.0
    signal_dispersion = float(snap["signal"].std(ddof=0))
    dispersion_reference = float(snap["signal"].abs().median()) + 1e-12
    relative_dispersion = signal_dispersion / dispersion_reference
    m = m_max
    reasons = []
    if threshold_relaxed:
        reasons.append("liquidity_risk_threshold_relaxed_for_feasibility")
    if relative_dispersion < float(cfg.get("low_signal_dispersion_ratio", 0.10)):
        m = max(m_min, m - 1)
        reasons.append("low_signal_dispersion")
    if average_abs_correlation > float(cfg.get("high_correlation_threshold", 0.65)):
        m = max(m_min, m - 1)
        reasons.append("high_cross_sectional_correlation")
    if not reasons:
        reasons.append("full_budget_supported")
    retention = float(cfg.get("minimum_candidate_retention", 0))
    retention_count = (
        int(math.ceil(retention * m)) if 0 < retention < 1 else int(retention)
    )
    retention_count = min(m, max(0, retention_count))
    eligible_previous = eligible[eligible["ticker"].isin(previous_candidates)].sort_values(
        ["base_score", "ticker"], ascending=[False, True]
    )
    cluster_cap = int(cfg.get("max_candidates_per_cluster", m))
    sector_cap = int(cfg.get("max_candidates_per_sector", m))
    sector_available = "sector" in eligible.columns and eligible["sector"].notna().any()
    def can_add(ticker: str) -> bool:
        cluster = cluster_map.get(str(ticker), str(ticker))
        cluster_count = sum(cluster_map.get(str(item), str(item)) == cluster for item in selected)
        if cluster_count >= cluster_cap:
            return False
        if sector_available:
            ticker_sector = eligible.loc[eligible["ticker"].eq(ticker), "sector"].iloc[0]
            if pd.notna(ticker_sector):
                selected_sectors = eligible.loc[eligible["ticker"].isin(selected), "sector"]
                if int(selected_sectors.eq(ticker_sector).sum()) >= sector_cap:
                    return False
        return True
    for ticker in eligible_previous["ticker"]:
        if len(selected) >= retention_count:
            break
        if can_add(str(ticker)):
            selected.append(str(ticker))
    if selected:
        reasons.append(f"retained_{len(selected)}_prior_candidates")
    for _ in range(len(selected), m):
        remain = eligible[~eligible["ticker"].isin(selected)].copy()
        if selected:
            remain["corr_penalty"] = [
                float(corr.loc[t, selected].abs().mean()) if t in corr.index else 0.0
                for t in remain["ticker"]
            ]
        else:
            remain["corr_penalty"] = 0.0
        remain["adaptive_score"] = remain["base_score"] - cfg["correlation_penalty"] * remain["corr_penalty"]
        allowed = remain[remain["ticker"].astype(str).map(can_add)]
        if allowed.empty:
            allowed = remain
            reasons.append("cluster_or_sector_cap_relaxed_for_feasibility")
        selected.append(str(allowed.sort_values(
            ["adaptive_score", "ticker"], ascending=[False, True]
        ).iloc[0]["ticker"]))
    snap["selected_candidate"] = snap["ticker"].isin(selected)
    snap["decision_reason"] = np.where(snap["selected_candidate"],
        "selected_by_signal_liquidity_risk_and_correlation", "outside_qubit_budget")
    snap["eligible_count"] = len(snap)
    snap["selected_m"] = m
    snap["signal_dispersion"] = signal_dispersion
    snap["relative_signal_dispersion"] = relative_dispersion
    snap["average_abs_correlation"] = average_abs_correlation
    snap["candidate_size_reason"] = "|".join(reasons)
    snap["retained_prior_candidates"] = len(set(selected) & previous_candidates)
    snap["liquidity_floor"] = liquidity_floor
    snap["risk_ceiling"] = risk_ceiling
    snap["minimum_expected_return"] = expected_return_floor
    snap["expected_return_column"] = expected_return_column
    snap["expected_return_filter_status"] = expected_return_filter_status
    snap["force_defensive_exposure"] = force_defensive_exposure
    snap["cluster_cap"] = cluster_cap
    snap["sector_candidate_cap"] = sector_cap if sector_available else np.nan
    return snap.sort_values(["selected_candidate", "base_score"], ascending=[False, False])


def qubo_instance(mu: np.ndarray, cov: np.ndarray, risk_aversion: float) -> np.ndarray:
    return risk_aversion * cov - (1 - risk_aversion) * np.diag(mu)


def qubo_to_ising(q: np.ndarray) -> tuple[float, np.ndarray, np.ndarray]:
    """Map symmetric ``x.T @ Q @ x`` to ``offset + h.z + sum J_ij z_i z_j``.

    The binary/spin convention is x=(1-z)/2 with z in {-1,+1}.
    """
    q = (np.asarray(q, dtype=float) + np.asarray(q, dtype=float).T) / 2
    n = len(q)
    offset = 0.0
    h = np.zeros(n)
    j = np.zeros((n, n))
    for i in range(n):
        offset += q[i, i] / 2
        h[i] -= q[i, i] / 2
        for k in range(i + 1, n):
            offset += q[i, k] / 2
            h[i] -= q[i, k] / 2
            h[k] -= q[i, k] / 2
            j[i, k] = j[k, i] = q[i, k] / 2
    return float(offset), h, j


def ising_energy(spins: np.ndarray, offset: float, h: np.ndarray, j: np.ndarray) -> float:
    pair = sum(j[i, k] * spins[i] * spins[k]
               for i in range(len(spins)) for k in range(i + 1, len(spins)))
    return float(offset + h @ spins + pair)


def ewma_mean_cov(returns: pd.DataFrame, span: int = 60,
                  horizon: int = 20) -> tuple[np.ndarray, np.ndarray]:
    """Estimate a multivariate EWMA mean vector and covariance matrix.

    Rows are observations ordered from oldest to newest and columns are assets.
    The newest observations receive the largest weights. The estimates are scaled
    to the requested holding horizon.
    """
    values = np.asarray(returns, dtype=float)
    if values.ndim != 2 or values.shape[0] < 2 or values.shape[1] < 1:
        raise ValueError("EWMA requires at least two observations and one asset.")
    if not np.isfinite(values).all():
        raise ValueError("EWMA input must contain only finite returns.")
    if span <= 1 or horizon <= 0:
        raise ValueError("EWMA span must exceed one and horizon must be positive.")
    alpha = 2.0 / (span + 1.0)
    ages = np.arange(values.shape[0] - 1, -1, -1)
    weights = alpha * np.power(1.0 - alpha, ages)
    weights /= weights.sum()
    mean_daily = weights @ values
    centered = values - mean_daily
    covariance_daily = (centered * weights[:, None]).T @ centered
    # Correct the finite-sample bias of normalized reliability weights.
    denominator = 1.0 - float(weights @ weights)
    if denominator > 1e-12:
        covariance_daily /= denominator
    covariance_daily = (covariance_daily + covariance_daily.T) / 2
    covariance_daily += np.eye(values.shape[1]) * 1e-10
    return mean_daily * horizon, covariance_daily * horizon


def aligned_previous_weights(tickers: list[str], previous: dict[str, float]) -> np.ndarray:
    """Return pre-trade weights aligned to the currently selected tickers."""
    return np.asarray([previous.get(ticker, 0.0) for ticker in tickers], dtype=float)


def portfolio_turnover(previous: dict[str, float],
                       target: dict[str, float]) -> tuple[float, dict[str, float]]:
    """Compute one-way turnover over the union of old and new holdings."""
    names = sorted(set(previous) | set(target))
    trades = {name: target.get(name, 0.0) - previous.get(name, 0.0) for name in names}
    return float(sum(abs(value) for value in trades.values())), trades


def round_target_weights_to_board_lot(
    target: dict[str, float], prices: dict[str, float], notional_vnd: float,
    board_lot: int = 100,
) -> tuple[dict[str, float], dict]:
    """Convert continuous target weights to executable board-lot weights.

    Residual cash is explicit and remains uninvested. The function never reallocates
    that cash to another security because doing so could silently violate bounds or
    alter the solver-selected cardinality.
    """
    if notional_vnd <= 0 or board_lot <= 0:
        raise ValueError("notional_vnd and board_lot must be positive")
    executed: dict[str, float] = {}
    shares: dict[str, int] = {}
    for ticker, weight in target.items():
        price = float(prices.get(ticker, np.nan))
        if not np.isfinite(price) or price <= 0:
            raise ValueError(f"Missing positive execution price for {ticker}")
        lots = math.floor(float(weight) * notional_vnd / (price * board_lot))
        quantity = int(lots * board_lot)
        shares[ticker] = quantity
        executed[ticker] = quantity * price / notional_vnd
    invested = float(sum(executed.values()))
    diagnostics = {
        "notional_vnd": float(notional_vnd), "board_lot": int(board_lot),
        "share_quantities": shares, "invested_weight": invested,
        "cash_residual_weight": max(0.0, 1.0 - invested),
        "selected_assets_with_positive_shares": sum(quantity > 0 for quantity in shares.values()),
        "cardinality_preserved": all(quantity > 0 for quantity in shares.values()),
    }
    return executed, diagnostics


def technical_factor_score(frame: pd.DataFrame) -> pd.Series:
    """Return a point-in-time cross-sectional momentum/risk score in ``[0, 1]``.

    Positive weights reward momentum and trend.  Negative weights reward lower
    volatility.  Ranking is performed independently at every date, which makes the
    score comparable across market regimes without fitting on the test window.
    """
    if frame.empty:
        return pd.Series(dtype=float, index=frame.index)
    numerator = pd.Series(0.0, index=frame.index, dtype=float)
    denominator = pd.Series(0.0, index=frame.index, dtype=float)
    dates = pd.to_datetime(frame["date"])
    for column, signed_weight in TECHNICAL_FACTOR_WEIGHTS.items():
        if column not in frame:
            continue
        values = pd.to_numeric(frame[column], errors="coerce")
        ranked = values.groupby(dates).rank(pct=True, method="average")
        if signed_weight < 0:
            ranked = 1.0 - ranked
        weight = abs(float(signed_weight))
        present = ranked.notna().astype(float)
        numerator = numerator.add(ranked.fillna(0.0) * weight, fill_value=0.0)
        denominator = denominator.add(present * weight, fill_value=0.0)
    return (numerator / denominator.replace(0.0, np.nan)).fillna(0.5).clip(0.0, 1.0)


def _daily_rank_ic(scores: pd.Series, target: pd.Series, dates: pd.Series) -> pd.Series:
    values = pd.DataFrame({"score": scores, "target": target, "date": pd.to_datetime(dates)})
    rows = []
    for date, group in values.dropna().groupby("date"):
        if len(group) < 3 or group["score"].nunique() < 2 or group["target"].nunique() < 2:
            continue
        rows.append((date, float(spearmanr(group["score"], group["target"]).statistic)))
    return pd.Series({date: value for date, value in rows}, dtype=float)


def select_validation_signal_blend(
    model_bundle: dict,
    calibration: pd.DataFrame,
    snapshot: pd.DataFrame,
    cfg: dict,
) -> tuple[np.ndarray, pd.DataFrame, np.ndarray, dict]:
    """Select an XGBoost/technical blend using purged validation observations only."""
    usable = calibration.dropna(subset=["target_rank", "target_return_20d"]).copy()
    if len(usable) < 20:
        raise ValueError("At least 20 purged observations are required for signal blending.")
    raw_xgb = pd.Series(predict(model_bundle, usable), index=usable.index, dtype=float)
    xgb_rank = raw_xgb.groupby(pd.to_datetime(usable["date"])).rank(pct=True, method="average")
    factor = technical_factor_score(usable)
    grid = sorted({float(value) for value in cfg.get(
        "xgboost_weight_grid", [0.25, 0.50, 0.75, 1.00]
    )})
    if not grid or min(grid) < 0 or max(grid) > 1:
        raise ValueError("xgboost_weight_grid must contain values in [0, 1].")
    stability_penalty = float(cfg.get("blend_stability_penalty", 0.25))
    candidates = []
    for xgb_weight in grid:
        blended = xgb_weight * xgb_rank + (1.0 - xgb_weight) * factor
        daily_ic = _daily_rank_ic(blended, usable["target_rank"], usable["date"])
        mean_ic = float(daily_ic.mean()) if len(daily_ic) else -np.inf
        standard_error = (
            float(daily_ic.std(ddof=1) / math.sqrt(len(daily_ic))) if len(daily_ic) > 1 else 0.0
        )
        objective = mean_ic - stability_penalty * standard_error
        candidates.append({
            "xgboost_weight": xgb_weight,
            "technical_weight": 1.0 - xgb_weight,
            "validation_mean_daily_rank_ic": mean_ic,
            "validation_rank_ic_standard_error": standard_error,
            "validation_objective": objective,
            "validation_days": int(len(daily_ic)),
            "validation_positive_ic_ratio": (
                float((daily_ic > 0).mean()) if len(daily_ic) else np.nan
            ),
        })
    best = max(
        candidates,
        key=lambda row: (row["validation_objective"], row["xgboost_weight"]),
    )
    chosen_weight = float(best["xgboost_weight"])
    calibration_scores = chosen_weight * xgb_rank + (1.0 - chosen_weight) * factor
    snapshot_xgb = pd.Series(predict(model_bundle, snapshot), index=snapshot.index, dtype=float)
    snapshot_xgb_rank = snapshot_xgb.rank(pct=True, method="average")
    snapshot_factor = technical_factor_score(snapshot)
    snapshot_scores = chosen_weight * snapshot_xgb_rank + (1.0 - chosen_weight) * snapshot_factor
    diagnostics = {
        "method": "purged_validation_xgboost_technical_blend",
        **best,
        "grid": candidates,
    }
    return (
        snapshot_scores.to_numpy(dtype=float), usable,
        calibration_scores.to_numpy(dtype=float), diagnostics,
    )


def calibrate_scores_to_returns(
    calibration: pd.DataFrame,
    calibration_scores: np.ndarray,
    snapshot_scores: np.ndarray,
    method: str,
) -> tuple[np.ndarray, dict]:
    """Map arbitrary point-in-time scores to the 20-day return scale."""
    realized = pd.to_numeric(calibration["target_return_20d"], errors="coerce").to_numpy()
    scores = np.asarray(calibration_scores, dtype=float)
    valid = np.isfinite(realized) & np.isfinite(scores)
    if valid.sum() < 20:
        raise ValueError("At least 20 finite calibration observations are required.")
    realized = realized[valid]
    scores = scores[valid]
    low, high = np.nanquantile(realized, [0.01, 0.99])
    realized = np.clip(realized, low, high)
    design = np.column_stack([np.ones(len(scores)), scores])
    intercept, slope = np.linalg.lstsq(design, realized, rcond=None)[0]
    expected = np.clip(intercept + slope * np.asarray(snapshot_scores, dtype=float), low, high)
    fitted = design @ np.asarray([intercept, slope])
    ss_total = float(np.sum((realized - realized.mean()) ** 2))
    r_squared = 1 - float(np.sum((realized - fitted) ** 2)) / ss_total if ss_total > 0 else np.nan
    return expected, {
        "method": method,
        "observations": int(len(realized)), "intercept": float(intercept),
        "slope": float(slope), "r_squared": float(r_squared),
        "target_clip_low": float(low), "target_clip_high": float(high),
    }


def market_regime_exposure(history: pd.DataFrame, cfg: dict) -> tuple[float, dict]:
    """Determine a pre-decision equity exposure from broad-market trend and volatility."""
    if not cfg or cfg.get("mode", "none") == "none":
        return 1.0, {"mode": "none", "equity_exposure": 1.0, "cash_weight": 0.0}
    if cfg.get("mode") != "market_trend_volatility":
        raise ValueError(f"Unsupported exposure mode: {cfg.get('mode')}")
    fast_days = int(cfg.get("fast_days", 63))
    slow_days = int(cfg.get("slow_days", 126))
    market = (
        history.dropna(subset=["date", "ret1"])
        .groupby("date")["ret1"].median().sort_index().tail(max(fast_days, slow_days))
    )
    if len(market) < max(20, fast_days):
        return 1.0, {
            "mode": "market_trend_volatility", "equity_exposure": 1.0,
            "cash_weight": 0.0, "fallback": "insufficient_market_history",
        }
    fast_return = float((1.0 + market.tail(fast_days)).prod() - 1.0)
    slow_return = float((1.0 + market.tail(slow_days)).prod() - 1.0)
    if fast_return > 0 and slow_return > 0:
        trend_exposure = 1.0
        regime = "risk_on"
    elif fast_return > 0 or slow_return > 0:
        trend_exposure = float(cfg.get("neutral_exposure", 0.60))
        regime = "mixed"
    else:
        trend_exposure = float(cfg.get("minimum_exposure", 0.25))
        regime = "risk_off"
    realized_volatility = float(market.tail(60).std(ddof=1) * math.sqrt(252))
    target_volatility = float(cfg.get("target_market_volatility", 0.22))
    volatility_scale = (
        min(1.0, target_volatility / realized_volatility)
        if np.isfinite(realized_volatility) and realized_volatility > 1e-12 else 1.0
    )
    minimum = float(cfg.get("minimum_exposure", 0.25))
    exposure = float(np.clip(trend_exposure * volatility_scale, minimum, 1.0))
    return exposure, {
        "mode": "market_trend_volatility", "regime": regime,
        "fast_days": fast_days, "slow_days": slow_days,
        "fast_market_return": fast_return, "slow_market_return": slow_return,
        "market_volatility_annualized": realized_volatility,
        "target_market_volatility": target_volatility,
        "volatility_scale": volatility_scale,
        "equity_exposure": exposure, "cash_weight": 1.0 - exposure,
    }
def drift_weights(target: dict[str, float], test_returns: pd.DataFrame) -> dict[str, float]:
    """Carry target weights to the next rebalance after realized asset returns."""
    if not target:
        return {}
    columns = list(target)
    aligned = test_returns.reindex(columns=columns)
    if not np.isfinite(aligned.to_numpy(dtype=float)).all():
        raise ValueError("Realized returns must be resolved before weight drift is calculated.")
    growth = (1.0 + aligned).prod(axis=0).to_numpy()
    values = np.asarray([target[name] for name in columns]) * growth
    residual_cash = max(0.0, 1.0 - float(sum(target.values())))
    total = float(values.sum() + residual_cash)
    if total <= 0 or not np.isfinite(total):
        return target.copy()
    return {name: float(value / total) for name, value in zip(columns, values) if value > 1e-12}


def simulate_buy_and_hold(
    target: dict[str, float], test_returns: pd.DataFrame, transaction_cost: float = 0.0,
) -> dict:
    """Simulate units drifting between rebalances, with an exact initial cost debit."""
    if not target or test_returns.empty:
        empty = pd.Series(dtype=float)
        return {"gross_returns": empty, "net_returns": empty, "ending_weights": target.copy(),
                "gross_wealth": empty, "net_wealth": empty}
    tickers = list(target)
    returns = test_returns.reindex(columns=tickers).astype(float)
    if not np.isfinite(returns.to_numpy()).all():
        raise ValueError(
            "Realized return panel contains unresolved missing/invalid observations; "
            "zero imputation is prohibited inside the portfolio simulator."
        )
    weights = np.asarray([target[ticker] for ticker in tickers], dtype=float)
    if weights.sum() > 1 + 1e-9:
        raise ValueError("Target asset weights exceed full investment.")
    residual_cash = max(0.0, 1.0 - float(weights.sum()))
    net_scale = max(0.0, 1.0 - float(transaction_cost))
    gross_values = weights.copy()
    net_values = weights * net_scale
    gross_cash = residual_cash
    net_cash = residual_cash * net_scale
    gross_wealth, net_wealth = [], []
    for row in returns.to_numpy():
        gross_values *= 1.0 + row
        net_values *= 1.0 + row
        gross_wealth.append(float(gross_values.sum() + gross_cash))
        net_wealth.append(float(net_values.sum() + net_cash))
    gross_wealth = pd.Series(gross_wealth, index=returns.index, name="gross_wealth")
    net_wealth = pd.Series(net_wealth, index=returns.index, name="net_wealth")
    gross_returns = gross_wealth.pct_change()
    net_returns = net_wealth.pct_change()
    gross_returns.iloc[0] = gross_wealth.iloc[0] - 1.0
    net_returns.iloc[0] = net_wealth.iloc[0] - 1.0
    total = float(net_values.sum() + net_cash)
    ending = ({ticker: float(value / total) for ticker, value in zip(tickers, net_values)}
              if total > 0 else target.copy())
    return {
        "gross_returns": gross_returns, "net_returns": net_returns,
        "ending_weights": ending, "gross_wealth": gross_wealth, "net_wealth": net_wealth,
    }


def prepare_realized_return_panel(
    test: pd.DataFrame,
    tickers: list[str],
    security_master: pd.DataFrame,
    *,
    research_mode: bool,
    delisting_return: float = -1.0,
    maximum_unexplained_gap_days: int = 5,
) -> tuple[pd.DataFrame, list[dict]]:
    """Resolve non-trading gaps and verified delistings without blanket fillna(0).

    Interior gaps for a still-listed security are carried at a zero mark-to-market
    return and explicitly logged. A disappearance before the end of the test window is
    fatal in research mode unless a point-in-time delisting event exists. A verified
    delisting applies the configured conservative liquidation return once; proceeds are
    then held as cash (zero return).
    """
    if not tickers:
        return pd.DataFrame(), []
    calendar = pd.DatetimeIndex(sorted(pd.to_datetime(test["date"]).dropna().unique()))
    if calendar.empty:
        return pd.DataFrame(), []
    raw = test[test["ticker"].isin(tickers)].pivot(
        index="date", columns="ticker", values="ret1"
    ).reindex(index=calendar, columns=tickers)
    master = security_master.copy()
    master["delisting_date"] = pd.to_datetime(master.get("delisting_date"), errors="coerce")
    master = master.drop_duplicates("ticker", keep="last").set_index("ticker")
    diagnostics: list[dict] = []
    for ticker in tickers:
        series = raw[ticker].copy()
        valid_dates = series.dropna().index
        if valid_dates.empty:
            raise ValueError(f"{ticker} has no realized observation in the test window.")
        last_valid = valid_dates.max()
        delisting_date = (
            master.at[ticker, "delisting_date"] if ticker in master.index else pd.NaT
        )
        suffix_missing = series.index[(series.index > last_valid) & series.isna()]
        verified_delisting = pd.notna(delisting_date) and delisting_date <= calendar.max()
        if len(suffix_missing) > maximum_unexplained_gap_days and not verified_delisting:
            message = (
                f"{ticker} disappears for {len(suffix_missing)} test observations after "
                f"{last_valid.date()} without a verified delisting event."
            )
            if research_mode:
                raise ValueError(message)
            diagnostics.append({"ticker": ticker, "event": "demo_unexplained_suffix", "detail": message})
        if verified_delisting:
            liquidation_candidates = series.index[series.index >= delisting_date]
            if len(liquidation_candidates):
                liquidation_date = liquidation_candidates[0]
                series.loc[liquidation_date] = float(delisting_return)
                series.loc[series.index > liquidation_date] = 0.0
                diagnostics.append({
                    "ticker": ticker, "event": "verified_delisting_liquidation",
                    "date": liquidation_date, "return_applied": float(delisting_return),
                })
        missing_before = int(series.isna().sum())
        series = series.fillna(0.0)
        if missing_before:
            diagnostics.append({
                "ticker": ticker, "event": "non_trading_mark_carry",
                "observations": missing_before,
            })
        raw[ticker] = series
    return raw.astype(float), diagnostics


def transaction_cost_breakdown(
    trades: dict[str, float],
    *,
    commission_bps: float,
    sell_tax_bps: float = 0.0,
    slippage_bps: float = 0.0,
    impact_coefficient: float = 0.0,
    adv_capacity_weights: dict[str, float] | None = None,
) -> tuple[float, dict[str, dict[str, float]]]:
    """Return weight-based commission, sell tax, slippage and square-root impact."""
    details: dict[str, dict[str, float]] = {}
    total = 0.0
    capacities = adv_capacity_weights or {}
    for ticker, trade in trades.items():
        absolute = abs(float(trade))
        commission = absolute * commission_bps / 10000
        sell_tax = max(0.0, -float(trade)) * sell_tax_bps / 10000
        slippage = absolute * slippage_bps / 10000
        capacity = max(float(capacities.get(ticker, 1.0)), 1e-12)
        impact = absolute * impact_coefficient * math.sqrt(absolute / capacity) if absolute else 0.0
        cost = commission + sell_tax + slippage + impact
        details[ticker] = {
            "commission_cost": commission, "sell_tax_cost": sell_tax,
            "slippage_cost": slippage, "market_impact_cost": impact,
            "transaction_cost": cost,
        }
        total += cost
    return float(total), details


def record_rebalanced_strategy(
    name: str, fold: dict, target: dict[str, float], test_returns: pd.DataFrame,
    previous_weights: dict[str, dict[str, float]], cost_rate: float | dict,
    weight_rows: list[dict], trade_rows: list[dict], return_rows: list[dict],
) -> dict:
    """Apply one common accounting policy to every benchmark and proposed strategy."""
    previous = previous_weights.setdefault(name, {})
    turnover, changes = portfolio_turnover(previous, target)
    if isinstance(cost_rate, dict):
        total_cost, cost_details = transaction_cost_breakdown(changes, **cost_rate)
    else:
        total_cost = float(cost_rate) * turnover
        cost_details = {
            ticker: {"commission_cost": abs(change) * float(cost_rate),
                     "sell_tax_cost": 0.0, "slippage_cost": 0.0,
                     "market_impact_cost": 0.0,
                     "transaction_cost": abs(change) * float(cost_rate)}
            for ticker, change in changes.items()
        }
    simulation = simulate_buy_and_hold(target, test_returns, total_cost)
    previous_weights[name] = simulation["ending_weights"]
    decision_time = fold["test_start"]
    trade_time = test_returns.index.min() if not test_returns.empty else pd.NaT
    for ticker in sorted(set(previous) | set(target)):
        weight_rows.append({
            "fold": fold["fold"], "decision_time": decision_time, "strategy": name,
            "ticker": ticker, "weight": target.get(ticker, 0.0),
            "pre_trade_weight": previous.get(ticker, 0.0),
        })
        trade_rows.append({
            "fold": fold["fold"], "trade_time": trade_time, "strategy": name,
            "ticker": ticker, "pre_trade_weight": previous.get(ticker, 0.0),
            "target_weight": target.get(ticker, 0.0), "trade_weight": changes[ticker],
            "turnover": abs(changes[ticker]),
            **cost_details[ticker],
        })
    for date in simulation["net_returns"].index:
        return_rows.append({
            "fold": fold["fold"], "date": date, "strategy": name,
            "gross_return": simulation["gross_returns"].loc[date],
            "net_return": simulation["net_returns"].loc[date],
            "return": simulation["net_returns"].loc[date],
        })
    return {"turnover": turnover, "transaction_cost": total_cost,
            **simulation}


def energy(bits: np.ndarray, q: np.ndarray) -> float:
    return float(bits @ q @ bits)


def feasible_states(n: int, k: int) -> np.ndarray:
    states = []
    for combo in itertools.combinations(range(n), k):
        b = np.zeros(n, dtype=int)
        b[list(combo)] = 1
        states.append(b)
    return np.asarray(states)


def exact_solver(q: np.ndarray, k: int) -> dict:
    states = feasible_states(len(q), k)
    energies = np.array([energy(s, q) for s in states])
    idx = int(np.argmin(energies))
    return {"method": "exact", "bits": states[idx], "energy": float(energies[idx]),
            "feasibility_rate": 1.0, "runtime_seconds": 0.0}


def simulated_annealing(q: np.ndarray, k: int, seed: int, steps: int = 800) -> dict:
    start = time.perf_counter()
    rng = np.random.default_rng(seed)
    n = len(q)
    bits = np.zeros(n, dtype=int)
    bits[rng.choice(n, k, replace=False)] = 1
    best, best_e = bits.copy(), energy(bits, q)
    cur_e = best_e
    # When k is 0 or n the cardinality-feasible subspace contains a single
    # state, so there is no one-for-zero swap to anneal.
    if k in (0, n):
        return {
            "method": "simulated_annealing", "bits": best,
            "energy": float(best_e), "feasibility_rate": 1.0,
            "runtime_seconds": time.perf_counter() - start, "seed": seed,
        }
    for step in range(steps):
        ones, zeros = np.flatnonzero(bits), np.flatnonzero(1 - bits)
        proposal = bits.copy()
        proposal[rng.choice(ones)] = 0
        proposal[rng.choice(zeros)] = 1
        e = energy(proposal, q)
        temp = max(0.001, 0.1 * (1 - step / steps))
        if e < cur_e or rng.random() < math.exp((cur_e - e) / temp):
            bits, cur_e = proposal, e
        if cur_e < best_e:
            best, best_e = bits.copy(), cur_e
    return {"method": "simulated_annealing", "bits": best, "energy": float(best_e),
            "feasibility_rate": 1.0, "runtime_seconds": time.perf_counter() - start,
            "seed": seed}


def _optimize_qaoa_angles(evaluate, p: int, budget: int, seed: int) -> dict:
    """Deterministic multi-start COBYLA optimization with an auditable trace."""
    rng = np.random.default_rng(seed)
    starts = max(1, min(3, budget // 8))
    per_start = max(5, budget // starts)
    trace = []
    best = None
    bounds = [(0.0, 2 * np.pi)] * p + [(0.0, np.pi)] * p
    for start_id in range(starts):
        x0 = np.r_[rng.uniform(0, 2 * np.pi, p), rng.uniform(0, np.pi, p)]
        evaluations = []

        def objective(params):
            expected, _ = evaluate(params)
            evaluations.append(float(expected))
            return expected

        result = minimize(
            objective, x0, method="COBYLA", bounds=bounds,
            options={"maxiter": per_start, "tol": 1e-7, "catol": 1e-7},
        )
        expected, probabilities = evaluate(result.x)
        trace.append({
            "start": start_id, "initial_parameters": x0.tolist(),
            "final_parameters": result.x.tolist(), "objective_trace": evaluations,
            "final_expected_energy": float(expected), "success": bool(result.success),
            "stopping_reason": str(result.message), "evaluations": int(result.nfev),
        })
        if best is None or expected < best["expected_energy"]:
            best = {
                "expected_energy": float(expected), "probabilities": probabilities,
                "parameters": result.x, "success": bool(result.success),
                "stopping_reason": str(result.message),
            }
    best["trace"] = trace
    best["optimizer"] = "COBYLA_multi_start"
    best["optimizer_budget"] = int(budget)
    return best


def xy_qaoa_statevector(
    q: np.ndarray, k: int, p: int, trials: int, shots: int, seed: int,
    uniform_probability_noise_proxy: float = 0.0,
    depolarizing_probability: float = 0.0,
    readout_error_probability: float = 0.0,
) -> dict:
    """Ideal statevector simulation in the fixed-Hamming-weight subspace.

    The initial state is the Dicke state. The mixer connects feasible bitstrings that
    differ by swapping one selected and one unselected asset, equivalent to an all-to-all
    XY exchange mixer restricted to the feasible subspace.
    """
    start = time.perf_counter()
    rng = np.random.default_rng(seed)
    states = feasible_states(len(q), k)
    costs = np.array([energy(s, q) for s in states])
    dim = len(states)
    mixer = np.zeros((dim, dim))
    for i in range(dim):
        for j in range(i + 1, dim):
            if np.abs(states[i] - states[j]).sum() == 2:
                mixer[i, j] = mixer[j, i] = 1.0
    mixer_eigenvalues, mixer_eigenvectors = np.linalg.eigh(mixer)
    initial = np.ones(dim, dtype=complex) / np.sqrt(dim)
    cost_scale = max(float(np.max(np.abs(costs))), 1e-12)
    phase_costs = costs / cost_scale

    def evaluate(params):
        gammas, betas = params[:p], params[p:]
        psi = initial.copy()
        for gamma, beta in zip(gammas, betas):
            psi *= np.exp(-1j * gamma * phase_costs)
            coefficients = mixer_eigenvectors.T.conj() @ psi
            psi = mixer_eigenvectors @ (
                np.exp(-1j * beta * mixer_eigenvalues) * coefficients
            )
        probs = np.abs(psi) ** 2
        return float(probs @ costs), probs / probs.sum()

    optimized = _optimize_qaoa_angles(evaluate, p, trials, seed)
    sample_probs = optimized["probabilities"].copy()
    if uniform_probability_noise_proxy:
        level = float(uniform_probability_noise_proxy)
        if not 0 <= level <= 1:
            raise ValueError("uniform_probability_noise_proxy must be in [0, 1].")
        sample_probs = (1 - level) * sample_probs + level * np.ones(dim) / dim
    for probability, label in [
        (depolarizing_probability, "depolarizing_probability"),
        (readout_error_probability, "readout_error_probability"),
    ]:
        if not 0 <= float(probability) <= 1:
            raise ValueError(f"{label} must be in [0, 1].")
    counts_idx = rng.choice(dim, size=shots, p=sample_probs)
    sampled_bits = states[counts_idx].copy()
    if depolarizing_probability:
        affected = rng.random(shots) < float(depolarizing_probability)
        sampled_bits[affected] = rng.integers(0, 2, size=(int(affected.sum()), len(q)))
    if readout_error_probability:
        flips = rng.random(sampled_bits.shape) < float(readout_error_probability)
        sampled_bits = np.bitwise_xor(sampled_bits, flips.astype(int))
    measured_counts: dict[str, int] = {}
    for bits in sampled_bits:
        key = "".join(map(str, bits))
        measured_counts[key] = measured_counts.get(key, 0) + 1
    measured_feasible = sampled_bits.sum(axis=1) == k
    feasible_measured = sampled_bits[measured_feasible]
    if len(feasible_measured):
        feasible_keys, feasible_key_counts = np.unique(feasible_measured, axis=0, return_counts=True)
        primary_bits = feasible_keys[int(np.argmax(feasible_key_counts))]
        feasible_energies = np.asarray([energy(bits, q) for bits in feasible_keys])
        best_observed_bits = feasible_keys[int(np.argmin(feasible_energies))]
    else:
        primary_bits = states[int(np.argmax(sample_probs))]
        best_observed_bits = primary_bits.copy()
    if not depolarizing_probability and not readout_error_probability:
        primary_bits = states[int(np.argmax(sample_probs))]
    primary = int(np.flatnonzero((states == primary_bits).all(axis=1))[0])
    best_observed = int(np.flatnonzero((states == best_observed_bits).all(axis=1))[0])
    counts = np.asarray([
        measured_counts.get("".join(map(str, state)), 0) for state in states
    ])
    optimal_mask = np.isclose(costs, costs.min(), atol=1e-10, rtol=1e-8)
    measured_success = float(np.mean([
        bits.sum() == k and np.isclose(energy(bits, q), costs.min(), atol=1e-10, rtol=1e-8)
        for bits in sampled_bits
    ]))
    bit_counts = measured_counts
    bit_probs = {"".join(map(str, states[i])): float(prob)
                 for i, prob in enumerate(sample_probs) if prob > 1e-12}
    return {
        "method": "xy_qaoa_dicke_ideal_statevector", "bits": states[primary],
        "seed": seed,
        "energy": float(costs[primary]), "expected_energy": float(sample_probs @ costs),
        "mean_energy": float(sample_probs @ costs),
        "primary_probability": (
            float(measured_counts.get("".join(map(str, states[primary])), 0) / shots)
            if depolarizing_probability or readout_error_probability
            else float(sample_probs[primary])
        ),
        "success_probability": (
            measured_success if depolarizing_probability or readout_error_probability
            else float(sample_probs[optimal_mask].sum())
        ),
        "best_observed_bits": states[best_observed],
        "best_observed_energy": float(costs[best_observed]),
        "feasibility_rate": float(measured_feasible.mean()),
        "runtime_seconds": time.perf_counter() - start,
        "shots": shots, "depth_p": p, "two_qubit_gate_estimate": p * len(q) * (len(q) - 1) // 2,
        "bitstring_counts": bit_counts, "backend": "internal_ideal_statevector_fixed_weight",
        "bitstring_probabilities": bit_probs,
        "uniform_probability_noise_proxy": float(uniform_probability_noise_proxy),
        "depolarizing_probability": float(depolarizing_probability),
        "readout_error_probability": float(readout_error_probability),
        "noise_model": (
            "phenomenological_depolarizing_plus_readout_sampling"
            if depolarizing_probability or readout_error_probability
            else ("legacy_uniform_probability_proxy" if uniform_probability_noise_proxy else "ideal")
        ),
        "optimizer": optimized["optimizer"], "optimizer_budget": optimized["optimizer_budget"],
        "optimal_parameters": optimized["parameters"].tolist(),
        "parameter_trace": optimized["trace"],
        "optimizer_success": optimized["success"],
        "stopping_reason": optimized["stopping_reason"],
    }


def penalty_qaoa_baseline(q: np.ndarray, k: int, seed: int, shots: int = 1024) -> dict:
    """Transparent stochastic penalty baseline; not labeled as a circuit simulation."""
    start = time.perf_counter()
    rng = np.random.default_rng(seed)
    n = len(q)
    samples = rng.integers(0, 2, size=(shots, n))
    penalty = max(1.0, float(np.abs(q).sum())) * (samples.sum(axis=1) - k) ** 2
    energies = np.array([energy(s, q) for s in samples]) + penalty
    idx = int(np.argmin(energies))
    feasible = samples.sum(axis=1) == k
    return {"method": "penalty_stochastic_baseline", "bits": samples[idx],
            "energy": energy(samples[idx], q), "feasibility_rate": float(feasible.mean()),
            "runtime_seconds": time.perf_counter() - start, "shots": shots, "seed": seed,
            "backend": "classical_stochastic_baseline_not_qaoa_circuit"}


def penalty_qaoa_statevector(
    q: np.ndarray, k: int, p: int, trials: int, shots: int, seed: int,
    penalty_strength: float | None = None,
) -> dict:
    """Ideal full-Hilbert-space penalty-QAOA circuit simulation.

    Uses |+> initialization, diagonal economic+cardinality cost Hamiltonian and
    transverse-X mixer. This is an actual statevector QAOA simulation, not hardware.
    """
    start = time.perf_counter()
    rng = np.random.default_rng(seed)
    n = len(q)
    states = np.array([
        [(index >> (n - 1 - bit)) & 1 for bit in range(n)]
        for index in range(2 ** n)
    ], dtype=int)
    economic = np.array([energy(s, q) for s in states])
    penalty_strength = penalty_strength or max(1.0, 2 * float(np.abs(q).sum()))
    total_cost = economic + penalty_strength * (states.sum(axis=1) - k) ** 2
    dim = len(states)
    initial = np.ones(dim, dtype=complex) / np.sqrt(dim)
    cost_scale = max(float(np.max(np.abs(total_cost))), 1e-12)
    phase_cost = total_cost / cost_scale

    def evaluate(params):
        gammas, betas = params[:p], params[p:]
        psi = initial.copy()
        for gamma, beta in zip(gammas, betas):
            psi *= np.exp(-1j * gamma * phase_cost)
            # exp(-i beta sum X_j) factorizes into independent single-qubit
            # rotations because all X_j terms commute.
            cosine, sine = np.cos(beta), -1j * np.sin(beta)
            for bit in range(n):
                stride = 1 << bit
                for base in range(0, dim, stride * 2):
                    lo = slice(base, base + stride)
                    hi = slice(base + stride, base + 2 * stride)
                    a, b = psi[lo].copy(), psi[hi].copy()
                    psi[lo] = cosine * a + sine * b
                    psi[hi] = sine * a + cosine * b
        probs = np.abs(psi) ** 2
        return float(probs @ total_cost), probs / probs.sum()

    optimized = _optimize_qaoa_angles(evaluate, p, trials, seed)
    probabilities = optimized["probabilities"]
    sampled = rng.choice(dim, size=shots, p=probabilities)
    counts = np.bincount(sampled, minlength=dim)
    feasible = states.sum(axis=1) == k
    feasible_indices = np.flatnonzero(feasible)
    chosen_idx = int(feasible_indices[np.argmax(probabilities[feasible_indices])])
    observed_pool = np.flatnonzero((counts > 0) & feasible)
    best_observed_idx = (
        int(observed_pool[np.argmin(economic[observed_pool])])
        if len(observed_pool) else int(np.argmin(np.where(counts > 0, total_cost, np.inf)))
    )
    bit_counts = {"".join(map(str, states[i])): int(c) for i, c in enumerate(counts) if c}
    sampled_feasible = sum(c for i, c in enumerate(counts) if feasible[i]) / shots
    optimum = economic[feasible].min()
    optimal_mask = feasible & np.isclose(economic, optimum, atol=1e-10, rtol=1e-8)
    return {
        "method": "penalty_qaoa_ideal_statevector", "bits": states[chosen_idx],
        "seed": seed,
        "energy": float(economic[chosen_idx]),
        "expected_energy": float(probabilities @ economic),
        "penalized_mean_energy": optimized["expected_energy"],
        "primary_probability": float(probabilities[chosen_idx]),
        "success_probability": float(probabilities[optimal_mask].sum()),
        "best_observed_bits": states[best_observed_idx],
        "best_observed_energy": float(economic[best_observed_idx]),
        "feasibility_rate": float(sampled_feasible),
        "runtime_seconds": time.perf_counter() - start, "shots": shots, "depth_p": p,
        "two_qubit_gate_estimate": p * (n * (n - 1) // 2 + n),
        "bitstring_counts": bit_counts, "backend": "internal_ideal_statevector_full_hilbert",
        "penalty_strength": penalty_strength,
        "optimizer": optimized["optimizer"], "optimizer_budget": optimized["optimizer_budget"],
        "optimal_parameters": optimized["parameters"].tolist(),
        "parameter_trace": optimized["trace"],
        "optimizer_success": optimized["success"],
        "stopping_reason": optimized["stopping_reason"],
    }


def holm_adjust(p_values: list[float]) -> list[float]:
    if not p_values:
        return []
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p)
    adjusted = np.empty_like(p)
    running = 0.0
    m = len(p)
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * p[idx])
        adjusted[idx] = min(1.0, running)
    return adjusted.tolist()


def paired_block_bootstrap_test(
    a: pd.Series, b: pd.Series, seed: int, samples: int = 500, block: int = 10
) -> dict:
    joined = pd.concat([a.rename("a"), b.rename("b")], axis=1).dropna()
    diff = (joined["a"] - joined["b"]).to_numpy()
    if len(diff) < block * 2:
        return {"mean_difference": float(np.mean(diff)) if len(diff) else np.nan,
                "ci_low": np.nan, "ci_high": np.nan, "p_value": np.nan}
    rng = np.random.default_rng(seed)
    means = []
    for _ in range(samples):
        starts = rng.integers(0, len(diff) - block + 1, math.ceil(len(diff) / block))
        sample = np.concatenate([diff[s:s + block] for s in starts])[:len(diff)]
        means.append(sample.mean())
    means = np.asarray(means)
    centered = diff - diff.mean()
    null_means = []
    for _ in range(samples):
        starts = rng.integers(0, len(centered) - block + 1, math.ceil(len(centered) / block))
        sample = np.concatenate([centered[s:s + block] for s in starts])[:len(centered)]
        null_means.append(sample.mean())
    null_means = np.asarray(null_means)
    p_value = float((np.abs(null_means) >= abs(diff.mean())).mean())
    return {
        "mean_difference": float(diff.mean()),
        "ci_low": float(np.quantile(means, 0.025)),
        "ci_high": float(np.quantile(means, 0.975)),
        "p_value": float(min(1.0, p_value)),
        "effect_size_daily": float(diff.mean()),
        "bootstrap_centered_under_null": True,
    }


def optimize_weights(
    mu: np.ndarray, cov: np.ndarray, lower: float | np.ndarray, upper: float | np.ndarray,
    risk_aversion: float, previous: np.ndarray | None, turnover_penalty: float,
    *, turnover_limit: float | None = None, sectors: list[str] | None = None,
    sector_cap: float | None = None,
) -> np.ndarray:
    n = len(mu)
    mu = np.nan_to_num(np.asarray(mu, dtype=float))
    cov = np.nan_to_num(np.asarray(cov, dtype=float))
    cov = (cov + cov.T) / 2 + np.eye(n) * 1e-10
    lower_bounds = np.broadcast_to(np.asarray(lower, dtype=float), (n,)).copy()
    upper_bounds = np.broadcast_to(np.asarray(upper, dtype=float), (n,)).copy()
    if (
        lower_bounds.sum() > 1 + 1e-12
        or upper_bounds.sum() < 1 - 1e-12
        or np.any(lower_bounds > upper_bounds + 1e-12)
    ):
        raise ValueError(
            f"Infeasible weight bounds for selected cardinality n={n}, "
            f"total_lower={lower_bounds.sum()}, total_upper={upper_bounds.sum()}, "
            f"lower={lower_bounds.tolist()}, upper={upper_bounds.tolist()}"
        )
    prev = np.ones(n) / n if previous is None or len(previous) != n else previous
    def objective(w):
        # Scaling avoids SLSQP terminating on the very small daily-return objective.
        return 1000.0 * (
            risk_aversion * (w @ cov @ w) - mu @ w
            + turnover_penalty * np.sqrt((w - prev) ** 2 + 1e-12).sum()
        )
    constraints: list[dict] = [{"type": "eq", "fun": lambda w: w.sum() - 1}]
    if turnover_limit is not None:
        constraints.append({
            "type": "ineq",
            "fun": lambda w: float(turnover_limit) - np.abs(w - prev).sum(),
        })
    if sectors is not None and sector_cap is not None:
        if len(sectors) != n:
            raise ValueError("sectors must align with the selected assets")
        for sector in sorted(set(map(str, sectors))):
            mask = np.asarray([str(value) == sector for value in sectors], dtype=float)
            constraints.append({
                "type": "ineq", "fun": lambda w, m=mask: float(sector_cap) - float(m @ w),
            })
    residual_capacity = upper_bounds - lower_bounds
    residual_weight = 1.0 - float(lower_bounds.sum())
    initial = lower_bounds + (
        residual_weight * residual_capacity / residual_capacity.sum()
        if residual_capacity.sum() > 1e-12 else 0.0
    )
    result = minimize(objective, initial, method="SLSQP",
                      bounds=[(float(floor), float(limit)) for floor, limit in zip(lower_bounds, upper_bounds)],
                      constraints=constraints,
                      options={"maxiter": 1000, "ftol": 1e-10})
    if result.success and np.isfinite(result.x).all():
        return result.x
    if turnover_limit is not None or (sectors is not None and sector_cap is not None):
        raise ValueError(f"Constrained weight optimization failed: {result.message}")
    # Deterministic projected-gradient fallback remains a genuine convex classical
    # optimizer and is more stable than silently returning arbitrary weights.
    def project_box_simplex(v):
        lo, hi = np.min(v - upper_bounds), np.max(v - lower_bounds)
        for _ in range(100):
            mid = (lo + hi) / 2
            w = np.clip(v - mid, lower_bounds, upper_bounds)
            if w.sum() > 1:
                lo = mid
            else:
                hi = mid
        return np.clip(v - (lo + hi) / 2, lower_bounds, upper_bounds)
    w = initial.copy()
    lipschitz = max(1e-6, 2 * risk_aversion * np.linalg.eigvalsh(cov).max())
    step = min(0.1, 1 / lipschitz)
    for _ in range(3000):
        smooth_turnover_grad = turnover_penalty * (w - prev) / np.sqrt((w - prev) ** 2 + 1e-12)
        grad = 2 * risk_aversion * cov @ w - mu + smooth_turnover_grad
        updated = project_box_simplex(w - step * grad)
        if np.linalg.norm(updated - w) < 1e-10:
            break
        w = updated
    return w


def financial_metrics(returns: pd.Series, rf_annual: float | pd.Series) -> dict:
    r = returns.dropna()
    if r.empty:
        return {}
    equity = (1 + r).cumprod()
    ann_ret = equity.iloc[-1] ** (252 / len(r)) - 1
    ann_vol = r.std(ddof=1) * np.sqrt(252)
    if isinstance(rf_annual, pd.Series):
        annual_rates = rf_annual.reindex(r.index).ffill().bfill().astype(float)
        daily_rf = np.power(1.0 + annual_rates, 1 / 252) - 1
        excess = r - daily_rf
        rf_for_calmar = float(annual_rates.mean())
    else:
        daily_rf = pd.Series(np.power(1.0 + float(rf_annual), 1 / 252) - 1, index=r.index)
        excess = r - daily_rf
        rf_for_calmar = float(rf_annual)
    downside = r[r < 0].std(ddof=1) * np.sqrt(252)
    drawdown = equity / equity.cummax() - 1
    return {
        "cumulative_return": float(equity.iloc[-1] - 1), "annualized_return": float(ann_ret),
        "annualized_volatility": float(ann_vol),
        "sharpe": float(excess.mean() * 252 / ann_vol) if ann_vol else np.nan,
        "sortino": float((ann_ret - rf_for_calmar) / downside) if downside else np.nan,
        "max_drawdown": float(drawdown.min()),
        "calmar": float(ann_ret / abs(drawdown.min())) if drawdown.min() else np.nan,
        "positive_day_ratio": float((r > 0).mean()), "observations": int(len(r)),
    }


def resolve_risk_free_series(paths: Paths, cfg: dict, dates: pd.Series) -> pd.Series:
    """Resolve the declared annual risk-free rate without looking ahead."""
    index = pd.DatetimeIndex(sorted(pd.to_datetime(dates).dropna().unique()))
    risk_cfg = cfg.get("risk_free", {})
    mode = risk_cfg.get("mode", "fixed_annual")
    if mode == "fixed_annual":
        value = float(risk_cfg.get(
            "annual_rate", cfg.get("backtest", {}).get("risk_free_annual", 0.0)
        ))
        return pd.Series(value, index=index, name="risk_free_annual")
    if mode != "pit_macro_series":
        raise ValueError(f"Unsupported risk_free mode: {mode}")
    macro_path = paths.normalized / "macro.parquet"
    if not macro_path.exists():
        raise ValueError("pit_macro_series risk-free mode requires macro.parquet")
    macro = pd.read_parquet(macro_path)
    series_id = risk_cfg.get("series_id")
    macro = macro[macro["series_id"].astype(str) == str(series_id)].copy()
    if macro.empty:
        raise ValueError(f"Risk-free macro series is unavailable: {series_id}")
    macro["available_at"] = pd.to_datetime(macro["available_at"], errors="coerce")
    macro = macro.sort_values("available_at")[["available_at", "value"]]
    left = pd.DataFrame({"date": index})
    aligned = pd.merge_asof(
        left, macro, left_on="date", right_on="available_at", direction="backward"
    )
    if aligned["value"].isna().any():
        raise ValueError("Risk-free PIT series does not cover every OOS observation.")
    return pd.Series(aligned["value"].to_numpy(dtype=float), index=index, name="risk_free_annual")


def block_bootstrap_sharpe(returns: pd.Series, rf: float, seed: int, samples: int = 300,
                           block: int = 10) -> tuple[float, float]:
    r = returns.dropna().to_numpy()
    if len(r) < block * 2:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(samples):
        starts = rng.integers(0, len(r) - block + 1, math.ceil(len(r) / block))
        b = np.concatenate([r[s:s + block] for s in starts])[:len(r)]
        vol = b.std(ddof=1) * np.sqrt(252)
        vals.append(((b.mean() * 252 - rf) / vol) if vol else np.nan)
    return tuple(np.nanquantile(vals, [0.025, 0.975]))


def _blocked_experiment(
    project_root: Path, config_path: Path, cfg: dict, quality: dict, leak: dict,
    blockers: list[str], message: str,
) -> Path:
    stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
    cfg_hash = hashlib.sha256(config_path.read_bytes()).hexdigest()[:10]
    experiment_id = f"{stamp}-{cfg_hash}-blocked"
    out = project_root / "outputs" / "experiments" / experiment_id
    out.mkdir(parents=True, exist_ok=True)
    (out / "resolved_config.yaml").write_text(
        yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8"
    )
    (out / "data_quality.json").write_text(json.dumps(quality, indent=2), encoding="utf-8")
    (out / "leakage_audit.json").write_text(json.dumps(leak, indent=2), encoding="utf-8")
    outlier_review = project_root / "outputs" / "reports" / "return_outlier_review.csv"
    if outlier_review.exists():
        shutil.copy2(outlier_review, out / "return_outlier_review.csv")
    manifest = {
        "experiment_id": experiment_id, "status": "blocked", "label": cfg.get("label"),
        "mode": cfg.get("mode"), "created_at": datetime.now(timezone.utc).isoformat(),
        "blockers": blockers, "message": message, "config": str(config_path),
        "artifacts": [
            "RESEARCH_BLOCKED.md", "data_quality.json", "leakage_audit.json",
            "resolved_config.yaml",
            *(["return_outlier_review.csv"] if outlier_review.exists() else []),
        ],
    }
    (out / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    (out / "RESEARCH_BLOCKED.md").write_text(
        "# Research run blocked\n\n"
        f"- Experiment: `{experiment_id}`\n"
        f"- Reason: {message}\n"
        f"- Blockers: `{blockers}`\n\n"
        "No model, backtest, or research metric was produced. The system did not infer or "
        "fabricate historical universe records. Supply verified point-in-time source data, "
        "rebuild the universe snapshots, rerun the leakage audit, and then rerun this config.\n",
        encoding="utf-8",
    )
    return out


def run_experiment(project_root: Path, config_path: Path) -> Path:
    cfg = load_config(config_path)
    mode = cfg.get("mode")
    paths = Paths(project_root)
    quality, _ = validate_data(paths)
    universe_cfg = cfg.get("universe", {})
    try:
        build_universe(
            paths, cfg.get("data", {}).get("rebalance", "monthly"),
            universe_cfg.get("definition", "hose_all_listed"),
            universe_cfg.get("index_code"),
            universe_cfg.get("max_assets"),
            universe_cfg.get("liquidity_lookback_days", 60),
            universe_cfg.get("minimum_observations", 40),
        )
    except (ValueError, KeyError) as exc:
        # The leakage audit below will capture a missing/invalid universe contract.
        universe_build_error = str(exc)
    else:
        universe_build_error = None
    leak = leakage_audit(paths)
    if universe_build_error:
        leak.setdefault("blockers", []).append("universe_build_failed")
        leak["status"] = "blocked" if cfg.get("mode") == "research" else leak["status"]
        leak["universe_build_error"] = universe_build_error
    benchmark_cfg = cfg.get("benchmark", {})
    benchmark_path = paths.normalized / "benchmark.parquet"
    if cfg.get("mode") == "research" and benchmark_cfg.get("required", False):
        benchmark_valid = False
        if benchmark_path.exists():
            benchmark_check = pd.read_parquet(benchmark_path)
            required_benchmark_columns = {
                "date", "benchmark", "total_return_index", "available_at",
                "source", "source_url", "fetched_at", "raw_checksum", "data_class",
                "index_type", "methodology_url",
            }
            benchmark_valid = bool(
                not benchmark_check.empty
                and required_benchmark_columns <= set(benchmark_check.columns)
                and not benchmark_check["data_class"].astype(str).eq("fixture").any()
                and benchmark_check["index_type"].astype(str).str.lower().eq("total_return").all()
                and benchmark_check["methodology_url"].astype(str).str.startswith(
                    ("http://", "https://")
                ).all()
                and (pd.to_datetime(benchmark_check["available_at"], errors="coerce")
                     >= pd.to_datetime(benchmark_check["date"], errors="coerce")).all()
            )
        leak["checks"]["verified_total_return_benchmark_available"] = benchmark_valid
        if not benchmark_valid:
            leak.setdefault("blockers", []).append("verified_total_return_benchmark_available")
            leak["status"] = "blocked"
    risk_cfg = cfg.get("risk_free", {})
    if cfg.get("mode") == "research" and risk_cfg.get("mode") == "pit_macro_series":
        macro_path = paths.normalized / "macro.parquet"
        risk_series_valid = False
        if macro_path.exists():
            macro_check = pd.read_parquet(macro_path)
            risk_rows = macro_check[
                macro_check.get("series_id", pd.Series(dtype=str)).astype(str)
                == str(risk_cfg.get("series_id"))
            ]
            risk_series_valid = bool(
                not risk_rows.empty
                and {"available_at", "value", "source", "source_url", "data_class"}
                <= set(risk_rows.columns)
                and not risk_rows["data_class"].astype(str).eq("fixture").any()
            )
        leak["checks"]["risk_free_pit_series_available_when_required"] = risk_series_valid
        if not risk_series_valid:
            leak.setdefault("blockers", []).append(
                "risk_free_pit_series_available_when_required"
            )
            leak["status"] = "blocked"
    if quality["status"] != "pass":
        message = "Data quality failed; refusing to run."
        blocked = _blocked_experiment(project_root, config_path, cfg, quality, leak,
                                      list(dict.fromkeys([
                                          "data_quality", *leak.get("blockers", [])
                                      ])), message)
        raise ResearchRunBlocked(message, blocked)
    if cfg.get("mode") == "research" and "fixture" in quality["data_class"]:
        message = ("Research mode refuses fixture data. Supply verified real point-in-time "
                   "data and pass the leakage audit before using a research config.")
        blocked = _blocked_experiment(project_root, config_path, cfg, quality, leak,
                                      ["fixture_data_in_research_mode"], message)
        raise ResearchRunBlocked(message, blocked)
    if mode == "research" and leak["status"] not in {"pass", "pass_with_limitations"}:
        message = "Historical point-in-time leakage audit blocked the research run."
        blocked = _blocked_experiment(project_root, config_path, cfg, quality, leak,
                                      leak.get("blockers", ["leakage_audit"]), message)
        raise ResearchRunBlocked(message, blocked)
    if mode == "exploratory" and leak["status"] == "blocked":
        declared = set(cfg.get("exploratory", {}).get("allowed_leakage_blockers", []))
        actual = set(leak.get("blockers", []))
        unexpected = sorted(actual - declared)
        if unexpected:
            raise RuntimeError(
                "Exploratory leakage audit has undeclared blockers: "
                + ", ".join(unexpected)
            )
        leak["audit_status_before_exploratory_acceptance"] = "blocked"
        leak["accepted_exploratory_limitations"] = sorted(actual)
        leak["blockers"] = []
        leak["status"] = "pass_for_exploratory_with_declared_limitations"
        leak["note"] = (
            f"{leak.get('note', '')} The original blockers were explicitly accepted only "
            "for this exploratory run; this does not satisfy the research contract."
        ).strip()
    if leak["status"] == "blocked":
        raise RuntimeError("Leakage audit blocked; refusing to label results.")
    if cfg["reduction"]["candidate_size"] > 8 and cfg.get("mode") != "research":
        raise RuntimeError("Internal exact statevector demo is limited to 8 candidate qubits.")
    prices = pd.read_parquet(paths.normalized / "prices.parquet")
    prices["date"] = pd.to_datetime(prices["date"])
    data_cfg = cfg.get("data", {})
    if data_cfg.get("start"):
        prices = prices[prices["date"] >= pd.Timestamp(data_cfg["start"])].copy()
    if data_cfg.get("end"):
        prices = prices[prices["date"] <= pd.Timestamp(data_cfg["end"])].copy()
    if prices.empty:
        raise ValueError("No price observations remain inside the configured data interval.")
    prices = prices.sort_values(["ticker", "date"])
    prices["ret1"] = prices.groupby("ticker")["adjusted_close"].pct_change()
    target_horizon = int(cfg.get("target", {}).get(
        "horizon_days", cfg.get("covariance", {}).get("horizon_days", 20)
    ))
    features = attach_point_in_time_features(build_features(prices, target_horizon), paths)
    wf = cfg["walk_forward"]
    folds = make_folds(features["date"], wf["train_months"], wf["validation_months"],
                       wf["test_months"], wf.get("max_folds"),
                       wf.get("embargo_days", cfg.get("target", {}).get("horizon_days", 20)),
                       wf.get("selection", "evenly_spaced"),
                       wf.get("final_holdout_months", 0))
    evaluation_start = pd.Timestamp(wf["evaluation_start"]) if wf.get("evaluation_start") else None
    evaluation_end = pd.Timestamp(wf["evaluation_end"]) if wf.get("evaluation_end") else None
    if evaluation_start is not None:
        folds = [fold for fold in folds if pd.Timestamp(fold["test_start"]) >= evaluation_start]
    if evaluation_end is not None:
        folds = [fold for fold in folds if pd.Timestamp(fold["test_end"]) <= evaluation_end]
    for fold_id, fold in enumerate(folds):
        fold["fold"] = fold_id
    if not folds:
        raise ValueError("The configured evaluation interval produced no walk-forward folds.")
    stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
    cfg_hash = hashlib.sha256(config_path.read_bytes()).hexdigest()[:10]
    experiment_id = f"{stamp}-{cfg_hash}"
    out = project_root / "outputs" / "experiments" / experiment_id
    fig_dir = out / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)
    features.to_parquet(out / "features.parquet", index=False)
    outlier_review = paths.reports / "return_outlier_review.csv"
    if outlier_review.exists():
        shutil.copy2(outlier_review, out / "return_outlier_review.csv")
    universe = pd.read_parquet(paths.curated / "universe_monthly.parquet")
    universe["decision_time"] = pd.to_datetime(universe["decision_time"])
    security_master = pd.read_parquet(paths.normalized / "security_master.parquet")
    benchmark_data = pd.DataFrame()
    if benchmark_path.exists():
        benchmark_data = pd.read_parquet(benchmark_path)
        benchmark_data["date"] = pd.to_datetime(benchmark_data["date"])
        benchmark_data["available_at"] = pd.to_datetime(benchmark_data["available_at"])
        benchmark_name = benchmark_cfg.get("name", "VNINDEX_TOTAL_RETURN")
        benchmark_data = benchmark_data[
            benchmark_data["benchmark"].astype(str) == str(benchmark_name)
        ].sort_values("date")
        benchmark_data["ret1"] = benchmark_data["total_return_index"].pct_change()
    ranking_rows, selection_rows, instance_rows = [], [], []
    solver_rows, weight_rows, trade_rows, return_rows = [], [], [], []
    ablation_rows, sensitivity_rows, fold_audit_rows = [], [], []
    feature_coverage_rows, tuning_rows, aur_diagnostic_rows = [], [], []
    calibration_rows, missing_return_rows, constraint_rows = [], [], []
    signal_model_rows, exposure_rows = [], []
    capacity_scenario_rows = []
    previous_weights: dict[str, dict[str, float]] = {
        "full_pipeline_xy_qaoa": {},
    }
    sensitivity_cfg = cfg.get("sensitivity", {})
    representative_count = min(
        len(folds), int(sensitivity_cfg.get("representative_folds", 1))
    )
    sensitivity_fold_ids = set(
        int(index) for index in np.linspace(0, max(0, len(folds) - 1),
                                           max(1, representative_count)).round()
    )
    sensitivity_seeds = sensitivity_cfg.get("seeds", cfg["solver"]["seeds"][:1])
    frozen_model_params: dict[str, int] | None = None
    for fold in folds:
        train, val, test, fold_audit = purged_fold_frames(features, fold)
        fold_audit_rows.append(fold_audit)
        eligible_snapshots = universe[universe["decision_time"] <= fold["test_start"]]
        if eligible_snapshots.empty:
            continue
        universe_time = eligible_snapshots["decision_time"].max()
        eligible_tickers = set(eligible_snapshots[
            (eligible_snapshots["decision_time"] == universe_time)
            & eligible_snapshots["eligible"].astype(bool)
        ]["ticker"])
        available_features = features[
            (features.date <= fold["test_start"])
            & (features.feature_available_at <= fold["test_start"])
            & features.ticker.isin(eligible_tickers)
        ]
        if available_features.empty:
            continue
        snapshot_date = available_features["date"].max()
        market_features = [
            name for name in FEATURES
            if name not in {"roe_pit", "revenue_growth_yoy_pit", "policy_rate_pit"}
        ]
        snap = available_features[available_features.date == snapshot_date].dropna(
            subset=market_features
        ).copy()
        if len(snap) < cfg["reduction"]["candidate_size"] or test.empty:
            continue
        model_cfg = {**cfg["model"], "seed": cfg.get("seed", 42)}
        if fold.get("phase") == "final_holdout":
            if frozen_model_params is None:
                development_tuning = pd.DataFrame(tuning_rows)
                if development_tuning.empty:
                    raise ValueError("Final holdout cannot start before development tuning is frozen.")
                aggregate = (
                    development_tuning.groupby(["n_estimators", "max_depth"], as_index=False)
                    ["validation_rank_ic"].mean().sort_values(
                        ["validation_rank_ic", "n_estimators", "max_depth"],
                        ascending=[False, True, True],
                    )
                )
                frozen_model_params = {
                    "n_estimators": int(aggregate.iloc[0]["n_estimators"]),
                    "max_depth": int(aggregate.iloc[0]["max_depth"]),
                }
                (out / "config_freeze.json").write_text(json.dumps({
                    "frozen_at": datetime.now(timezone.utc).isoformat(),
                    "config_sha256": sha256_file(config_path),
                    "config_hash_short": cfg_hash,
                    "holdout_start": str(pd.Timestamp(fold["test_start"])),
                    "holdout_end": str(pd.Timestamp(fold["test_end"])),
                    "model_params": frozen_model_params,
                    "frozen_components": [
                        "feature_set", "model_family", "model_params", "adaptive_universe_reduction",
                        "qubo", "solver", "constraints", "costs", "hypotheses", "metrics",
                    ],
                    "policy": "single_final_holdout_evaluation_no_post_holdout_tuning",
                }, indent=2), encoding="utf-8")
            model_cfg.update(frozen_model_params)
            model_cfg["tuning_enabled"] = False
        bundle = fit_ranker(train, val, model_cfg)
        snap["xgboost_signal"] = predict(bundle, snap)
        snap["signal"] = snap["xgboost_signal"]
        calibration_frame = val.dropna(subset=["target_return_20d"])
        calibration_source = "purged_validation"
        if len(calibration_frame) < 20:
            calibration_frame = train.dropna(subset=["target_return_20d"])
            calibration_source = "purged_training_fallback"
        snap["xgboost_expected_return"], calibration = calibrate_rank_signal_to_returns(
            bundle, calibration_frame, snap
        )
        calibration_rows.append({
            "fold": fold["fold"], "decision_time": snapshot_date,
            "calibration_source": calibration_source, **calibration,
        })
        signal_mode = str(model_cfg.get("signal_mode", "xgboost"))
        snap["technical_factor_signal"] = technical_factor_score(snap)
        if signal_mode == "validation_blend":
            (
                blended_signal, blend_calibration_frame, blend_calibration_scores,
                blend_diagnostics,
            ) = select_validation_signal_blend(
                bundle, calibration_frame, snap, model_cfg,
            )
            snap["signal"] = blended_signal
            (
                snap["optimization_expected_return"], blend_calibration,
            ) = calibrate_scores_to_returns(
                blend_calibration_frame, blend_calibration_scores,
                snap["signal"].to_numpy(dtype=float),
                "purged_validation_blended_signal_to_return",
            )
            signal_model_rows.append({
                "fold": fold["fold"], "decision_time": snapshot_date,
                "signal_mode": signal_mode,
                "selected_xgboost_weight": blend_diagnostics["xgboost_weight"],
                "selected_technical_weight": blend_diagnostics["technical_weight"],
                "validation_mean_daily_rank_ic": blend_diagnostics[
                    "validation_mean_daily_rank_ic"
                ],
                "validation_rank_ic_standard_error": blend_diagnostics[
                    "validation_rank_ic_standard_error"
                ],
                "validation_objective": blend_diagnostics["validation_objective"],
                "validation_days": blend_diagnostics["validation_days"],
                "validation_positive_ic_ratio": blend_diagnostics[
                    "validation_positive_ic_ratio"
                ],
                "grid": json.dumps(blend_diagnostics["grid"], sort_keys=True),
                "calibration_method": blend_calibration["method"],
                "calibration_slope": blend_calibration["slope"],
                "calibration_r_squared": blend_calibration["r_squared"],
            })
        elif signal_mode == "xgboost":
            snap["optimization_expected_return"] = snap["xgboost_expected_return"]
            signal_model_rows.append({
                "fold": fold["fold"], "decision_time": snapshot_date,
                "signal_mode": signal_mode, "selected_xgboost_weight": 1.0,
                "selected_technical_weight": 0.0,
            })
        else:
            raise ValueError(f"Unsupported model signal_mode: {signal_mode}")
        for feature, coverage in bundle["feature_coverage"].items():
            feature_coverage_rows.append({
                "fold": fold["fold"], "feature": feature, "training_coverage": coverage,
                "active": feature in bundle["active_features"],
            })
        for tuning in bundle["tuning"]:
            tuning_rows.append({"fold": fold["fold"], **tuning,
                                "selected": all(tuning[key] == bundle["selected_params"][key]
                                                for key in ("n_estimators", "max_depth"))})
        history = features[
            (features.date <= snapshot_date) & features.ticker.isin(eligible_tickers)
        ].copy()
        history["ret1"] = history.sort_values(["ticker", "date"]).groupby("ticker")[
            "adjusted_close"
        ].pct_change()
        ewma_panel = history.pivot(index="date", columns="ticker", values="ret1").tail(252)
        ewma_signal = ewma_panel.ewm(
            span=int(cfg.get("covariance", {}).get("span", 60)), adjust=False
        ).mean().iloc[-1]
        snap["ewma_signal"] = snap["ticker"].map(ewma_signal)
        known = snap.dropna(subset=["target_rank"])
        ic = spearmanr(known["xgboost_signal"], known["target_rank"]).statistic if len(known) > 2 else np.nan
        optimization_ic = (
            spearmanr(known["signal"], known["target_rank"]).statistic
            if len(known) > 2 else np.nan
        )
        ewma_known = known.dropna(subset=["ewma_signal"])
        ewma_ic = (spearmanr(ewma_known["ewma_signal"], ewma_known["target_rank"]).statistic
                   if len(ewma_known) > 2 else np.nan)
        for row in snap.itertuples():
            ranking_rows.append({"fold": fold["fold"], "decision_time": snapshot_date,
                                 "ticker": row.ticker, "signal": row.signal,
                                 "xgboost_signal": row.xgboost_signal,
                                 "technical_factor_signal": row.technical_factor_signal,
                                 "xgboost_expected_return": row.xgboost_expected_return,
                                 "optimization_expected_return": row.optimization_expected_return,
                                 "ewma_signal": row.ewma_signal, "fold_rank_ic": ic,
                                 "xgboost_rank_ic": ic, "optimization_rank_ic": optimization_ic,
                                 "ewma_rank_ic": ewma_ic,
                                 "universe_snapshot_time": universe_time})
        previous_candidates = (
            set(aur_diagnostic_rows[-1]["selected_tickers"].split("|"))
            if aur_diagnostic_rows else set()
        )
        aur_snapshot = snap
        pre_solver_execution_exclusions = 0
        maximum_executable_price = np.nan
        pre_constraints = cfg.get("constraints", {})
        pre_notional = pre_constraints.get("portfolio_notional_vnd")
        pre_board_lot = int(pre_constraints.get("board_lot", 1))
        pre_adv_participation = pre_constraints.get("max_adv_participation")
        minimum_exposure = float(cfg.get("exposure", {}).get("minimum_exposure", 1.0))
        pre_cardinality = int(cfg["reduction"].get("cardinality", 1))
        if pre_notional and pre_board_lot > 1 and minimum_exposure > 0:
            # The conservative 1/K cap makes every possible K-asset combination
            # board-lot feasible, not merely each asset in isolation.
            per_lot_sleeve_cap = min(
                float(cfg["weights"]["upper"]), 1.0 / pre_cardinality
            )
            maximum_executable_price = (
                float(pre_notional) * minimum_exposure * per_lot_sleeve_cap
                / pre_board_lot / 1.000001
            )
            execution_eligible = pd.to_numeric(snap["close"], errors="coerce").le(
                maximum_executable_price
            )
            # A selected security must also permit at least one board lot under
            # the declared absolute ADV participation limit. This feasibility
            # screen is determined before QUBO construction and is independent
            # of subsequent portfolio returns.
            if pre_adv_participation is not None:
                lot_value = pre_board_lot * pd.to_numeric(
                    snap["close"], errors="coerce"
                )
                adv_capacity = float(pre_adv_participation) * pd.to_numeric(
                    snap["adv_20d"], errors="coerce"
                )
                execution_eligible &= lot_value.le(adv_capacity)
            pre_solver_execution_exclusions = int((~execution_eligible).sum())
            aur_snapshot = snap[execution_eligible].copy()
            if len(aur_snapshot) < pre_cardinality:
                raise ValueError(
                    "Too few assets can satisfy board-lot, minimum-exposure and "
                    "upper-weight constraints before solver selection."
                )
        reduced = adaptive_reduce(
            aur_snapshot, history, cfg["reduction"], previous_candidates=previous_candidates,
        )
        reduced["fold"] = fold["fold"]
        reduced["decision_time"] = snapshot_date
        selected_now = set(reduced.loc[reduced.selected_candidate, "ticker"])
        candidate_turnover = (len(selected_now.symmetric_difference(previous_candidates))
                              / max(1, len(selected_now | previous_candidates)))
        aur_diagnostic_rows.append({
            "fold": fold["fold"], "decision_time": snapshot_date,
            "eligible_count": int(reduced["eligible_count"].iloc[0]),
            "selected_m": int(reduced["selected_m"].iloc[0]),
            "signal_dispersion": float(reduced["signal_dispersion"].iloc[0]),
            "relative_signal_dispersion": float(reduced["relative_signal_dispersion"].iloc[0]),
            "average_abs_correlation": float(reduced["average_abs_correlation"].iloc[0]),
            "candidate_size_reason": reduced["candidate_size_reason"].iloc[0],
            "expected_return_filter_status": reduced["expected_return_filter_status"].iloc[0],
            "force_defensive_exposure": bool(reduced["force_defensive_exposure"].iloc[0]),
            "pre_solver_execution_exclusions": pre_solver_execution_exclusions,
            "maximum_executable_price_at_minimum_exposure": maximum_executable_price,
            "retained_prior_candidates": int(reduced["retained_prior_candidates"].iloc[0]),
            "candidate_turnover": candidate_turnover,
            "selected_tickers": "|".join(sorted(selected_now)),
            "cluster_concentration": float(
                reduced.loc[reduced["selected_candidate"], "correlation_cluster"]
                .value_counts(normalize=True).max()
            ),
            "sector_concentration": (
                float(reduced.loc[reduced["selected_candidate"], "sector"]
                      .value_counts(normalize=True).max())
                if "sector" in reduced and reduced["sector"].notna().any() else np.nan
            ),
        })
        fixed_m = int(reduced["selected_m"].iloc[0])
        fixed_topm = set(aur_snapshot.nlargest(fixed_m, "signal")["ticker"])
        adaptive_eval = snap[snap["ticker"].isin(selected_now)]
        fixed_eval = snap[snap["ticker"].isin(fixed_topm)]
        history_corr = history.pivot(index="date", columns="ticker", values="ret1").tail(120).corr()
        def _set_abs_corr(names: set[str]) -> float:
            matrix = history_corr.reindex(index=sorted(names), columns=sorted(names))
            upper = matrix.where(np.triu(np.ones(matrix.shape), 1).astype(bool)).stack()
            return float(upper.abs().mean()) if len(upper) else np.nan
        aur_diagnostic_rows[-1].update({
            "fixed_topm_tickers": "|".join(sorted(fixed_topm)),
            "adaptive_forward_return_mean": float(adaptive_eval["target_return_20d"].mean()),
            "fixed_topm_forward_return_mean": float(fixed_eval["target_return_20d"].mean()),
            "adaptive_liquidity_mean": float(adaptive_eval["adv_20d"].mean()),
            "fixed_topm_liquidity_mean": float(fixed_eval["adv_20d"].mean()),
            "adaptive_risk_mean": float(adaptive_eval["volatility_20d"].mean()),
            "fixed_topm_risk_mean": float(fixed_eval["volatility_20d"].mean()),
            "adaptive_abs_correlation": _set_abs_corr(selected_now),
            "fixed_topm_abs_correlation": _set_abs_corr(fixed_topm),
        })
        selection_columns = [
            "fold", "decision_time", "ticker", "signal", "xgboost_signal",
            "technical_factor_signal", "xgboost_expected_return",
            "optimization_expected_return",
            "ewma_signal", "liquidity_20d", "adv_20d",
            "volatility_20d", "signal_z", "liquidity_z", "risk_z", "base_score",
            "selected_candidate", "decision_reason", "eligible_count", "selected_m",
            "signal_dispersion", "relative_signal_dispersion", "average_abs_correlation",
            "candidate_size_reason",
            "aur_eligible", "liquidity_floor", "risk_ceiling",
            "minimum_expected_return", "expected_return_column",
            "expected_return_filter_status", "force_defensive_exposure",
            "correlation_cluster", "cluster_cap", "sector_candidate_cap",
        ]
        selection_rows.extend(reduced[selection_columns].to_dict("records"))
        candidates = reduced[reduced.selected_candidate]["ticker"].tolist()
        hist_returns = history[history.ticker.isin(candidates)].pivot(
            index="date", columns="ticker", values="ret1").tail(252).reindex(columns=candidates)
        minimum_history_coverage = float(cfg.get("covariance", {}).get("minimum_coverage", 0.95))
        history_coverage = hist_returns.notna().mean()
        candidates = [c for c in candidates if history_coverage.get(c, 0.0) >= minimum_history_coverage]
        hist_returns = hist_returns[candidates]
        for ticker in candidates:
            missing_count = int(hist_returns[ticker].isna().sum())
            if missing_count:
                missing_return_rows.append({
                    "fold": fold["fold"], "window": "estimation", "ticker": ticker,
                    "event": "non_trading_mark_carry", "observations": missing_count,
                })
        hist_returns = hist_returns.fillna(0.0)
        if len(candidates) < int(cfg["reduction"]["cardinality"]) or len(hist_returns) < 2:
            fold_audit_rows[-1]["skip_reason"] = "insufficient_candidate_return_coverage"
            continue
        covariance_cfg = cfg.get("covariance", {})
        covariance_method = covariance_cfg.get("method", "ewma")
        covariance_span = int(covariance_cfg.get("span", 60))
        holding_horizon = int(covariance_cfg.get("horizon_days", 20))
        if covariance_method == "ewma":
            ewma_mu, cov = ewma_mean_cov(hist_returns, covariance_span, holding_horizon)
        elif covariance_method == "ledoit_wolf":
            ewma_mu = hist_returns.mean().to_numpy() * holding_horizon
            cov = LedoitWolf().fit(hist_returns.to_numpy()).covariance_ * holding_horizon
        else:
            raise ValueError(f"Unsupported covariance method: {covariance_method}")
        expected_return_source = cfg.get("qubo", {}).get(
            "expected_return_source", "xgboost_calibrated"
        )
        if expected_return_source == "xgboost_calibrated":
            expected_map = snap.set_index("ticker")["xgboost_expected_return"]
            mu = expected_map.reindex(candidates).to_numpy(dtype=float)
        elif expected_return_source == "validation_blend_calibrated":
            expected_map = snap.set_index("ticker")["optimization_expected_return"]
            mu = expected_map.reindex(candidates).to_numpy(dtype=float)
        elif expected_return_source == "ewma":
            mu = ewma_mu
        else:
            raise ValueError(f"Unsupported QUBO expected_return_source: {expected_return_source}")
        q = qubo_instance(mu, cov, cfg["qubo"]["risk_aversion"])
        ising_offset, ising_h, ising_j = qubo_to_ising(q)
        k = min(cfg["reduction"]["cardinality"], len(candidates))
        instance_rows.append({
            "fold": fold["fold"], "decision_time": str(snapshot_date),
            "tickers": candidates, "cardinality": k,
            "expected_return": mu.tolist(), "covariance": cov.tolist(), "qubo_matrix": q.tolist(),
            "ising_offset": ising_offset, "ising_h": ising_h.tolist(),
            "ising_j": ising_j.tolist(), "binary_spin_convention": "x=(1-z)/2",
            "covariance_method": covariance_method, "covariance_span": covariance_span,
            "holding_horizon_days": holding_horizon,
            "expected_return_source": expected_return_source,
            "signal_mode": signal_mode,
            "ewma_expected_return_reference": ewma_mu.tolist(),
            "minimum_history_coverage": minimum_history_coverage,
        })
        exact = exact_solver(q, k)
        xy_runs = [
            xy_qaoa_statevector(
                q, k, cfg["solver"]["qaoa_depth"], cfg["solver"]["parameter_trials"],
                cfg["solver"]["shots"], seed,
            ) for seed in cfg["solver"]["seeds"]
        ]
        penalty_runs = [
            penalty_qaoa_statevector(
                q, k, cfg["solver"]["qaoa_depth"], cfg["solver"]["parameter_trials"],
                cfg["solver"]["shots"], seed,
            ) for seed in cfg["solver"]["seeds"]
        ]
        annealing_runs = [
            simulated_annealing(q, k, seed) for seed in cfg["solver"]["seeds"]
        ]
        runs = [
            exact,
            *annealing_runs,
            penalty_qaoa_baseline(q, k, cfg["solver"]["seeds"][0], cfg["solver"]["shots"]),
            *penalty_runs,
            *xy_runs,
        ]
        chosen_xy = min(xy_runs, key=lambda result: result["expected_energy"])
        solution_selection = str(cfg.get("solver", {}).get(
            "solution_selection", "highest_probability_feasible"
        ))
        if solution_selection == "best_observed_feasible":
            chosen_xy_bits = np.asarray(chosen_xy["best_observed_bits"]).copy()
        elif solution_selection == "highest_probability_feasible":
            chosen_xy_bits = np.asarray(chosen_xy["bits"]).copy()
        else:
            raise ValueError(f"Unsupported solver solution_selection: {solution_selection}")
        for run in runs:
            primary_bits = np.asarray(run["bits"]).copy()
            run["fold"] = fold["fold"]
            run["decision_time"] = str(snapshot_date)
            run["optimality_gap"] = float((run["energy"] - exact["energy"]) / (abs(exact["energy"]) + 1e-12))
            run["selected_tickers"] = [candidates[i] for i in np.flatnonzero(primary_bits)]
            run["bits"] = "".join(map(str, primary_bits))
            if "best_observed_bits" in run:
                run["best_observed_bits"] = "".join(map(str, run["best_observed_bits"]))
                run["best_observed_gap"] = float(
                    (run["best_observed_energy"] - exact["energy"])
                    / (abs(exact["energy"]) + 1e-12)
                )
            run["bitstring_counts"] = json.dumps(run.get("bitstring_counts", {}))
            run["bitstring_probabilities"] = json.dumps(run.get("bitstring_probabilities", {}))
            run["parameter_trace"] = json.dumps(run.get("parameter_trace", []))
            run["optimal_parameters"] = json.dumps(run.get("optimal_parameters", []))
            solver_rows.append(run)
        chosen = [candidates[i] for i in np.flatnonzero(chosen_xy_bits)]
        idx = [candidates.index(c) for c in chosen]
        strategy_name = "full_pipeline_xy_qaoa"
        previous = previous_weights[strategy_name]
        previous_selected = aligned_previous_weights(chosen, previous)
        constraints_cfg = cfg.get("constraints", {})
        portfolio_notional = constraints_cfg.get("portfolio_notional_vnd")
        adv_participation = constraints_cfg.get("max_adv_participation")
        selected_snapshot = snap.set_index("ticker").reindex(chosen)
        equity_exposure, exposure_diagnostics = market_regime_exposure(
            history, cfg.get("exposure", {}),
        )
        if bool(reduced["force_defensive_exposure"].iloc[0]):
            defensive_floor = float(cfg.get("exposure", {}).get("minimum_exposure", 0.25))
            equity_exposure = min(equity_exposure, defensive_floor)
            exposure_diagnostics.update({
                "regime": "insufficient_positive_candidates",
                "equity_exposure": equity_exposure,
                "cash_weight": 1.0 - equity_exposure,
                "forced_by_expected_return_filter": True,
            })
        else:
            exposure_diagnostics["forced_by_expected_return_filter"] = False
        board_lot = int(constraints_cfg.get("board_lot", 1))
        execution_prices = {
            ticker: float(selected_snapshot.at[ticker, "close"]) for ticker in chosen
        }
        # The market-regime overlay may propose an equity sleeve too small to
        # execute one board lot within the absolute ADV limit. Raise exposure
        # only to the minimum mechanically feasible level; this is an execution
        # constraint, not a return-dependent signal adjustment.
        if portfolio_notional and board_lot > 1 and adv_participation:
            required_exposure = max(
                board_lot * execution_prices[ticker]
                / max(
                    float(adv_participation)
                    * float(selected_snapshot.at[ticker, "adv_20d"]),
                    1e-12,
                )
                for ticker in chosen
            )
            if required_exposure > 1 + 1e-12:
                raise ValueError(
                    f"Fold {fold['fold']} contains a board lot exceeding its absolute ADV capacity."
                )
            if required_exposure > equity_exposure:
                equity_exposure = min(1.0, required_exposure * 1.000001)
                exposure_diagnostics.update({
                    "equity_exposure": equity_exposure,
                    "cash_weight": 1.0 - equity_exposure,
                    "raised_for_board_lot_adv_feasibility": True,
                })
            else:
                exposure_diagnostics["raised_for_board_lot_adv_feasibility"] = False
        effective_lower = np.full(len(chosen), float(cfg["weights"]["lower"]))
        if portfolio_notional and board_lot > 1 and equity_exposure > 0:
            board_lot_sleeve_floor = np.asarray([
                board_lot * execution_prices[ticker]
                / (float(portfolio_notional) * equity_exposure) * 1.000001
                for ticker in chosen
            ])
            effective_lower = np.maximum(effective_lower, board_lot_sleeve_floor)
        capacity_weights = {
            ticker: float(
                adv_participation * selected_snapshot.at[ticker, "adv_20d"]
                / (portfolio_notional * equity_exposure)
            )
            for ticker in chosen
        } if portfolio_notional and adv_participation else {}
        per_asset_upper = np.asarray([
            min(float(cfg["weights"]["upper"]), capacity_weights.get(ticker, np.inf))
            for ticker in chosen
        ])
        if per_asset_upper.sum() < 1 - 1e-12:
            raise ValueError(
                f"Fold {fold['fold']} is not investable at the declared portfolio notional/ADV "
                f"capacity: aggregate upper bound={per_asset_upper.sum():.6f}."
            )
        master_latest = security_master.drop_duplicates("ticker", keep="last").set_index("ticker")
        sector_metadata_available = (
            "sector" in selected_snapshot.columns
            and selected_snapshot["sector"].notna().any()
        )
        selected_sectors = [
            str(selected_snapshot.at[ticker, "sector"])
            if (
                sector_metadata_available
                and ticker in selected_snapshot.index
                and pd.notna(selected_snapshot.at[ticker, "sector"])
            )
            else "UNCLASSIFIED"
            for ticker in chosen
        ]
        apply_sector_cap = (
            constraints_cfg.get("sector_cap") is not None
            and len(set(selected_sectors) - {"UNCLASSIFIED"}) >= 2
        )
        exit_weight = sum(weight for ticker, weight in previous.items() if ticker not in chosen)
        turnover_limit = constraints_cfg.get("max_one_way_turnover")
        remaining_turnover = (
            max(0.0, float(turnover_limit) - exit_weight)
            if turnover_limit is not None else None
        )
        constraint_rows.append({
            "fold": fold["fold"], "decision_time": snapshot_date,
            "selected_tickers": "|".join(chosen),
            "long_only": float(cfg["weights"]["lower"]) >= 0,
            "full_investment": True,
            "configured_lower_bound": float(cfg["weights"]["lower"]),
            "effective_lower_bounds_equity_sleeve": "|".join(
                f"{value:.10g}" for value in effective_lower
            ),
            "configured_upper_bound": float(cfg["weights"]["upper"]),
            "effective_upper_bounds": "|".join(f"{value:.10g}" for value in per_asset_upper),
            "portfolio_notional_vnd": portfolio_notional,
            "max_adv_participation": adv_participation,
            "capacity_constraint_applied": bool(capacity_weights),
            "sector_cap": constraints_cfg.get("sector_cap"),
            "sector_metadata_available": sector_metadata_available,
            "sector_constraint_applied": apply_sector_cap,
            "sector_constraint_reason": (
                "applied" if apply_sector_cap else "disabled_or_insufficient_sector_metadata"
            ),
            "max_one_way_turnover": turnover_limit,
            "prior_exit_weight": exit_weight,
            "remaining_turnover_limit": remaining_turnover,
        })
        weights = optimize_weights(
            mu[idx], cov[np.ix_(idx, idx)], effective_lower, per_asset_upper,
            cfg["weights"]["risk_aversion"], previous_selected,
            cfg["weights"]["turnover_penalty"], turnover_limit=remaining_turnover,
            sectors=selected_sectors if apply_sector_cap else None,
            sector_cap=constraints_cfg.get("sector_cap") if apply_sector_cap else None,
        )
        weights = weights * equity_exposure
        exposure_rows.append({
            "fold": fold["fold"], "decision_time": snapshot_date,
            **exposure_diagnostics,
            "selected_expected_return": float(mu[idx] @ weights),
            "solver_solution_selection": solution_selection,
        })
        constraint_rows[-1].update({
            "full_investment": bool(np.isclose(equity_exposure, 1.0)),
            "cash_allowed_by_exposure_overlay": True,
            "target_equity_exposure": equity_exposure,
            "target_cash_weight": 1.0 - equity_exposure,
            "solver_solution_selection": solution_selection,
        })
        target = {ticker: float(w) for ticker, w in zip(chosen, weights)}
        scenario_notionals = constraints_cfg.get(
            "capacity_scenarios_vnd", [portfolio_notional] if portfolio_notional else []
        )
        for scenario_notional in scenario_notionals:
            if not scenario_notional:
                continue
            _, scenario = round_target_weights_to_board_lot(
                target, execution_prices, float(scenario_notional), board_lot,
            )
            capacity_scenario_rows.append({
                "fold": fold["fold"], "decision_time": snapshot_date,
                **{key: value for key, value in scenario.items() if key != "share_quantities"},
                "share_quantities": json.dumps(scenario["share_quantities"], sort_keys=True),
            })
        if portfolio_notional and board_lot > 1:
            target, execution = round_target_weights_to_board_lot(
                target, execution_prices, float(portfolio_notional), board_lot,
            )
            if not execution["cardinality_preserved"]:
                raise ValueError(
                    f"Fold {fold['fold']} board-lot execution gives zero shares to a selected asset."
                )
            constraint_rows[-1].update({
                "board_lot": board_lot,
                "cash_residual_weight": execution["cash_residual_weight"],
                "executed_cardinality": execution["selected_assets_with_positive_shares"],
                "solver_cardinality_preserved_after_rounding": execution["cardinality_preserved"],
            })
        executed_turnover, _ = portfolio_turnover(previous, target)
        constraint_rows[-1]["executed_one_way_turnover"] = executed_turnover
        constraint_rows[-1]["turnover_cap_satisfied_after_rounding"] = (
            turnover_limit is None or executed_turnover <= float(turnover_limit) + 1e-8
        )
        if turnover_limit is not None and executed_turnover > float(turnover_limit) + 1e-8:
            raise ValueError(
                f"Fold {fold['fold']} board-lot execution violates hard turnover cap: "
                f"{executed_turnover:.8f} > {float(turnover_limit):.8f}."
            )
        test_ret, resolved = prepare_realized_return_panel(
            test, chosen, security_master, research_mode=cfg.get("mode") == "research",
            delisting_return=float(cfg.get("backtest", {}).get("delisting_return", -1.0)),
            maximum_unexplained_gap_days=int(
                cfg.get("backtest", {}).get("maximum_unexplained_gap_days", 5)
            ),
        )
        missing_return_rows.extend({"fold": fold["fold"], "window": "test", **row} for row in resolved)
        cost_model = {
            "commission_bps": float(cfg["backtest"].get(
                "commission_bps", cfg["backtest"].get("transaction_cost_bps", 0)
            )),
            "sell_tax_bps": float(cfg["backtest"].get("sell_tax_bps", 0)),
            "slippage_bps": float(cfg["backtest"].get("slippage_bps", 0)),
            "impact_coefficient": float(cfg["backtest"].get("impact_coefficient", 0)),
            "adv_capacity_weights": capacity_weights,
        }
        simulation = record_rebalanced_strategy(
            strategy_name, fold, target, test_ret, previous_weights, cost_model,
            weight_rows, trade_rows, return_rows,
        )
        equal_name = "equal_weight_candidates"
        equal_target = {ticker: 1 / len(candidates) for ticker in candidates}
        equal_test, resolved = prepare_realized_return_panel(
            test, candidates, security_master, research_mode=cfg.get("mode") == "research",
            delisting_return=float(cfg.get("backtest", {}).get("delisting_return", -1.0)),
        )
        missing_return_rows.extend({"fold": fold["fold"], "window": "test", **row} for row in resolved)
        record_rebalanced_strategy(
            equal_name, fold, equal_target, equal_test, previous_weights, cost_model,
            weight_rows, trade_rows, return_rows,
        )
        # Eight pre-declared ablations use their own candidate construction and solver.
        ablation_specs = [
            ("liquidity_topk_exact", "liquidity", "exact"),
            ("ewma_topk_exact", "ewma", "exact"),
            ("xgboost_topk_exact", "xgboost", "exact"),
            ("adaptive_exact", "adaptive", "exact"),
            ("xgboost_penalty_qaoa", "xgboost", "penalty"),
            ("xgboost_xy_qaoa", "xgboost", "xy"),
            ("adaptive_penalty_qaoa", "adaptive", "penalty"),
            ("adaptive_xy_qaoa", "adaptive", "xy"),
            ("adaptive_simulated_annealing", "adaptive", "sa"),
        ]
        m = min(cfg["reduction"]["candidate_size"], len(snap))
        for ablation_name, selector, solver_name in ablation_specs:
            if selector == "liquidity":
                pool = snap.nlargest(m, "liquidity_20d")["ticker"].tolist()
            elif selector == "ewma":
                ewma_universe = history[history.ticker.isin(snap["ticker"])].pivot(
                    index="date", columns="ticker", values="ret1"
                ).tail(252)
                ewma_scores = ewma_universe.ewm(
                    span=covariance_span, adjust=False
                ).mean().iloc[-1]
                pool = ewma_scores.nlargest(m).index.tolist()
            elif selector == "xgboost":
                pool = snap.nlargest(m, "xgboost_signal")["ticker"].tolist()
            else:
                pool = candidates
            variant_hist = history[history.ticker.isin(pool)].pivot(
                index="date", columns="ticker", values="ret1"
            ).tail(252).reindex(columns=pool)
            variant_coverage = variant_hist.notna().mean()
            pool = [ticker for ticker in pool if variant_coverage.get(ticker, 0) >= minimum_history_coverage]
            variant_hist = variant_hist[pool]
            for ticker in pool:
                missing_count = int(variant_hist[ticker].isna().sum())
                if missing_count:
                    missing_return_rows.append({
                        "fold": fold["fold"], "window": f"estimation_{ablation_name}",
                        "ticker": ticker, "event": "non_trading_mark_carry",
                        "observations": missing_count,
                    })
            variant_hist = variant_hist.fillna(0.0)
            if len(pool) < 2 or variant_hist.empty:
                continue
            if covariance_method == "ewma":
                variant_ewma_mu, variant_cov = ewma_mean_cov(
                    variant_hist, covariance_span, holding_horizon
                )
            else:
                variant_ewma_mu = variant_hist.mean().to_numpy() * holding_horizon
                variant_cov = LedoitWolf().fit(
                    variant_hist.to_numpy()
                ).covariance_ * holding_horizon
            if selector == "xgboost":
                variant_mu = snap.set_index("ticker")["xgboost_expected_return"].reindex(
                    pool
                ).to_numpy(dtype=float)
                variant_return_source = "xgboost_calibrated"
            elif selector == "adaptive":
                variant_mu = snap.set_index("ticker")["optimization_expected_return"].reindex(
                    pool
                ).to_numpy(dtype=float)
                variant_return_source = expected_return_source
            else:
                variant_mu = variant_ewma_mu
                variant_return_source = "ewma"
            variant_q = qubo_instance(variant_mu, variant_cov, cfg["qubo"]["risk_aversion"])
            variant_k = min(cfg["reduction"]["cardinality"], len(pool))
            exact_variant = exact_solver(variant_q, variant_k)
            if solver_name == "penalty":
                chosen_run = penalty_qaoa_statevector(
                    variant_q, variant_k, cfg["solver"]["qaoa_depth"],
                    max(8, cfg["solver"]["parameter_trials"] // 2),
                    cfg["solver"]["shots"], cfg["solver"]["seeds"][0],
                )
            elif solver_name == "xy":
                chosen_run = xy_qaoa_statevector(
                    variant_q, variant_k, cfg["solver"]["qaoa_depth"],
                    max(8, cfg["solver"]["parameter_trials"] // 2),
                    cfg["solver"]["shots"], cfg["solver"]["seeds"][0],
                )
            elif solver_name == "sa":
                chosen_run = simulated_annealing(
                    variant_q, variant_k, cfg["solver"]["seeds"][0]
                )
            else:
                chosen_run = exact_variant
            selected = [pool[i] for i in np.flatnonzero(chosen_run["bits"])]
            selected_idx = [pool.index(ticker) for ticker in selected]
            strategy_previous = previous_weights.setdefault(ablation_name, {})
            previous_selected = aligned_previous_weights(selected, strategy_previous)
            variant_weights = optimize_weights(
                variant_mu[selected_idx], variant_cov[np.ix_(selected_idx, selected_idx)],
                cfg["weights"]["lower"], cfg["weights"]["upper"],
                cfg["weights"]["risk_aversion"], previous_selected,
                cfg["weights"]["turnover_penalty"],
            )
            variant_test, resolved = prepare_realized_return_panel(
                test, selected, security_master, research_mode=cfg.get("mode") == "research",
                delisting_return=float(cfg.get("backtest", {}).get("delisting_return", -1.0)),
            )
            missing_return_rows.extend(
                {"fold": fold["fold"], "window": "test", **row} for row in resolved
            )
            variant_target = {
                ticker: float(weight) for ticker, weight in zip(selected, variant_weights)
            }
            variant_result = record_rebalanced_strategy(
                ablation_name, fold, variant_target, variant_test, previous_weights,
                cost_model, weight_rows, trade_rows, return_rows,
            )
            gap = (chosen_run["energy"] - exact_variant["energy"]) / (
                abs(exact_variant["energy"]) + 1e-12
            )
            ablation_rows.append({
                "fold": fold["fold"], "configuration": ablation_name,
                "selector": selector, "solver": chosen_run["method"],
                "selected_tickers": "|".join(selected), "objective": chosen_run["energy"],
                "optimality_gap": gap, "feasibility_rate": chosen_run["feasibility_rate"],
                "turnover": variant_result["turnover"],
                "transaction_cost": variant_result["transaction_cost"],
                "covariance_method": covariance_method,
                "expected_return_source": variant_return_source,
            })
        # Independent classical benchmarks use the complete eligible universe at the
        # same decision time, schedule, holding convention, and transaction-cost model.
        benchmark_hist = history[history.ticker.isin(snap["ticker"])].pivot(
            index="date", columns="ticker", values="ret1"
        ).tail(252)
        minimum_observations = min(60, max(2, len(benchmark_hist) // 2))
        benchmark_hist = benchmark_hist.dropna(axis=1, thresh=minimum_observations)
        benchmark_tickers = benchmark_hist.columns.tolist()
        if len(benchmark_tickers) >= 2:
            for ticker in benchmark_tickers:
                missing_count = int(benchmark_hist[ticker].isna().sum())
                if missing_count:
                    missing_return_rows.append({
                        "fold": fold["fold"], "window": "estimation_classical_benchmark",
                        "ticker": ticker, "event": "non_trading_mark_carry",
                        "observations": missing_count,
                    })
            benchmark_hist = benchmark_hist.fillna(0.0)
            benchmark_mu, benchmark_cov = ewma_mean_cov(
                benchmark_hist, covariance_span, holding_horizon
            )
            benchmark_test, resolved = prepare_realized_return_panel(
                test, benchmark_tickers, security_master,
                research_mode=cfg.get("mode") == "research",
                delisting_return=float(cfg.get("backtest", {}).get("delisting_return", -1.0)),
            )
            missing_return_rows.extend(
                {"fold": fold["fold"], "window": "test", **row} for row in resolved
            )
            benchmark_targets = {
                "equal_weight_universe": np.ones(len(benchmark_tickers)) / len(benchmark_tickers),
                "markowitz_mean_variance": optimize_weights(
                    benchmark_mu, benchmark_cov,
                    float(cfg["weights"].get("benchmark_lower", 0.0)),
                    float(cfg["weights"].get("benchmark_upper", cfg["weights"]["upper"])),
                    cfg["weights"]["risk_aversion"],
                    aligned_previous_weights(
                        benchmark_tickers, previous_weights.get("markowitz_mean_variance", {})
                    ), cfg["weights"]["turnover_penalty"],
                ),
                "minimum_variance": optimize_weights(
                    np.zeros(len(benchmark_tickers)), benchmark_cov,
                    float(cfg["weights"].get("benchmark_lower", 0.0)),
                    float(cfg["weights"].get("benchmark_upper", cfg["weights"]["upper"])),
                    1.0,
                    aligned_previous_weights(
                        benchmark_tickers, previous_weights.get("minimum_variance", {})
                    ), cfg["weights"]["turnover_penalty"],
                ),
            }
            for benchmark_name, benchmark_weights in benchmark_targets.items():
                record_rebalanced_strategy(
                    benchmark_name, fold,
                    {ticker: float(weight) for ticker, weight in zip(
                        benchmark_tickers, benchmark_weights
                    )}, benchmark_test, previous_weights, cost_model,
                    weight_rows, trade_rows, return_rows,
                )
        if not benchmark_data.empty:
            market_slice = benchmark_data[
                (benchmark_data["date"] > fold["test_start"])
                & (benchmark_data["date"] <= fold["test_end"])
                & (benchmark_data["available_at"] <= benchmark_data["date"] + pd.Timedelta(days=1))
            ].dropna(subset=["ret1"])
            market_strategy = f"benchmark_{benchmark_cfg.get('name', 'VNINDEX_TOTAL_RETURN').lower()}"
            for row in market_slice.itertuples():
                return_rows.append({
                    "fold": fold["fold"], "date": row.date, "strategy": market_strategy,
                    "gross_return": float(row.ret1), "net_return": float(row.ret1),
                    "return": float(row.ret1),
                })
            if not market_slice.empty:
                trade_rows.append({
                    "fold": fold["fold"], "trade_time": market_slice["date"].min(),
                    "strategy": market_strategy, "ticker": benchmark_cfg.get("name"),
                    "pre_trade_weight": 1.0, "target_weight": 1.0,
                    "trade_weight": 0.0, "turnover": 0.0,
                    "commission_cost": 0.0, "sell_tax_cost": 0.0,
                    "slippage_cost": 0.0, "market_impact_cost": 0.0,
                    "transaction_cost": 0.0,
                })
        if fold["fold"] in sensitivity_fold_ids:
            declared_depths = cfg["solver"].get(
                "declared_depth_grid", [1, 2, 3, cfg["solver"]["qaoa_depth"]]
            )
            declared_shots = cfg["solver"].get(
                "declared_shots_grid", [256, 1024, 2048, cfg["solver"]["shots"]]
            )
            for depth in sorted({int(value) for value in declared_depths}):
                for shots in sorted({int(value) for value in declared_shots}):
                    for sensitivity_k in sorted({max(1, k - 1), k}):
                        for noise in (0.0, 0.02):
                            for sensitivity_seed in map(int, sensitivity_seeds):
                                sensitivity = xy_qaoa_statevector(
                                    q, sensitivity_k, depth,
                                    max(6, cfg["solver"]["parameter_trials"] // 3),
                                    shots, sensitivity_seed,
                                    depolarizing_probability=noise,
                                    readout_error_probability=noise,
                                )
                                exact_sensitivity = exact_solver(q, sensitivity_k)
                                sensitivity_selected = [
                                    candidates[i] for i in np.flatnonzero(sensitivity["bits"])
                                ]
                                sensitivity_test, resolved = prepare_realized_return_panel(
                                    test, sensitivity_selected, security_master,
                                    research_mode=cfg.get("mode") == "research",
                                    delisting_return=float(
                                        cfg.get("backtest", {}).get("delisting_return", -1.0)
                                    ),
                                )
                                missing_return_rows.extend(
                                    {"fold": fold["fold"], "window": "sensitivity_test", **row}
                                    for row in resolved
                                )
                                sensitivity_target = {
                                    ticker: 1 / len(sensitivity_selected)
                                    for ticker in sensitivity_selected
                                }
                                for sensitivity_cost in (
                                    0, cfg["backtest"].get("transaction_cost_bps", 0), 25
                                ):
                                    sensitivity_sim = simulate_buy_and_hold(
                                        sensitivity_target, sensitivity_test,
                                        sensitivity_cost / 10000,
                                    )
                                    sensitivity_rows.append({
                                        "sensitivity_factor": "core_partial_factorial",
                                        "fold": fold["fold"], "depth_p": depth, "shots": shots,
                                        "seed": sensitivity_seed,
                                        "cardinality": sensitivity_k,
                                        "candidate_size": len(candidates),
                                        "qubit_budget": cfg["reduction"]["qubit_budget"],
                                        "uniform_probability_noise_proxy": 0.0,
                                        "depolarizing_probability": noise,
                                        "readout_error_probability": noise,
                                        "noise_model": sensitivity["noise_model"],
                                        "transaction_cost_bps": sensitivity_cost,
                                        "energy": sensitivity["energy"],
                                        "optimality_gap": (
                                            sensitivity["energy"] - exact_sensitivity["energy"]
                                        ) / (abs(exact_sensitivity["energy"]) + 1e-12),
                                        "feasibility_rate": sensitivity["feasibility_rate"],
                                        "runtime_seconds": sensitivity["runtime_seconds"],
                                        "net_cumulative_return": float(
                                            (1 + sensitivity_sim["net_returns"]).prod() - 1
                                        ),
                                    })
            for sensitivity_n in sorted({max(k, len(candidates) - 2), len(candidates)}):
                q_reduced = q[:sensitivity_n, :sensitivity_n]
                k_reduced = min(k, sensitivity_n)
                for sensitivity_seed in map(int, sensitivity_seeds):
                    size_run = xy_qaoa_statevector(
                        q_reduced, k_reduced, cfg["solver"]["qaoa_depth"],
                        max(6, cfg["solver"]["parameter_trials"] // 3),
                        cfg["solver"]["shots"], sensitivity_seed,
                    )
                    size_exact = exact_solver(q_reduced, k_reduced)
                    size_selected = [
                        candidates[:sensitivity_n][i] for i in np.flatnonzero(size_run["bits"])
                    ]
                    size_test, resolved = prepare_realized_return_panel(
                        test, size_selected, security_master,
                        research_mode=cfg.get("mode") == "research",
                        delisting_return=float(
                            cfg.get("backtest", {}).get("delisting_return", -1.0)
                        ),
                    )
                    missing_return_rows.extend(
                        {"fold": fold["fold"], "window": "sensitivity_test", **row}
                        for row in resolved
                    )
                    size_target = {ticker: 1 / len(size_selected) for ticker in size_selected}
                    size_sim = simulate_buy_and_hold(
                        size_target, size_test,
                        cfg["backtest"].get("transaction_cost_bps", 0) / 10000,
                    )
                    sensitivity_rows.append({
                        "sensitivity_factor": "candidate_size_and_qubit_budget",
                        "fold": fold["fold"], "depth_p": cfg["solver"]["qaoa_depth"],
                        "shots": cfg["solver"]["shots"], "seed": sensitivity_seed,
                        "cardinality": k_reduced, "candidate_size": sensitivity_n,
                        "qubit_budget": sensitivity_n,
                        "uniform_probability_noise_proxy": 0.0,
                        "depolarizing_probability": 0.0,
                        "readout_error_probability": 0.0,
                        "noise_model": "ideal",
                        "transaction_cost_bps": cfg["backtest"].get("transaction_cost_bps", 0),
                        "energy": size_run["energy"],
                        "optimality_gap": (size_run["energy"] - size_exact["energy"])
                        / (abs(size_exact["energy"]) + 1e-12),
                        "feasibility_rate": size_run["feasibility_rate"],
                        "runtime_seconds": size_run["runtime_seconds"],
                        "net_cumulative_return": float(
                            (1 + size_sim["net_returns"]).prod() - 1
                        ),
                    })
    rankings = pd.DataFrame(ranking_rows)
    selections = pd.DataFrame(selection_rows)
    solvers = pd.DataFrame(solver_rows)
    weights_df, trades, returns = pd.DataFrame(weight_rows), pd.DataFrame(trade_rows), pd.DataFrame(return_rows)
    rankings.to_csv(out / "rankings.csv", index=False)
    pd.DataFrame(signal_model_rows).to_csv(out / "signal_model_selection.csv", index=False)
    pd.DataFrame(exposure_rows).to_csv(out / "exposure_by_fold.csv", index=False)
    selections.to_csv(out / "selected_universe.csv", index=False)
    solvers.to_csv(out / "solver_runs.csv", index=False)
    (out / "optimization_instances.json").write_text(
        json.dumps(instance_rows, indent=2), encoding="utf-8"
    )
    weights_df.to_csv(out / "weights.csv", index=False)
    trades.to_csv(out / "trades.csv", index=False)
    cost_columns = [
        "commission_cost", "sell_tax_cost", "slippage_cost",
        "market_impact_cost", "transaction_cost",
    ]
    for column in cost_columns:
        if column not in trades:
            trades[column] = 0.0
    cost_ledger = trades.groupby(["fold", "strategy"], as_index=False).agg(
        turnover=("turnover", "sum"), commission_cost=("commission_cost", "sum"),
        sell_tax_cost=("sell_tax_cost", "sum"), slippage_cost=("slippage_cost", "sum"),
        market_impact_cost=("market_impact_cost", "sum"),
        transaction_cost=("transaction_cost", "sum"),
    ) if not trades.empty else pd.DataFrame(
        columns=["fold", "strategy", "turnover", *cost_columns]
    )
    # Re-write after adding backwards-compatible zero component columns.
    trades.to_csv(out / "trades.csv", index=False)
    cost_ledger.to_csv(out / "cost_ledger.csv", index=False)
    main_weights = weights_df[weights_df.get("strategy", pd.Series(dtype=str)).eq(
        "full_pipeline_xy_qaoa"
    )].copy()
    main_trades = trades[trades.get("strategy", pd.Series(dtype=str)).eq(
        "full_pipeline_xy_qaoa"
    )].copy()
    selected_assets = selections[selections.get(
        "selected_candidate", pd.Series(dtype=bool)
    ).astype(bool)].copy()
    if not selected_assets.empty:
        selected_assets = selected_assets.merge(
            main_weights[["fold", "ticker", "weight", "pre_trade_weight"]],
            on=["fold", "ticker"], how="left",
        ).merge(
            main_trades[["fold", "ticker", "trade_weight", "transaction_cost"]],
            on=["fold", "ticker"], how="left",
        )
        master_columns = [column for column in [
            "ticker", "security_id", "company_name", "sector"
        ] if column in security_master.columns]
        master_lookup = security_master.drop_duplicates("ticker", keep="last")[master_columns]
        selected_assets = selected_assets.merge(master_lookup, on="ticker", how="left")
        selected_assets["target_weight"] = selected_assets["weight"].fillna(0.0)
        selected_assets["previous_weight"] = selected_assets["pre_trade_weight"].fillna(0.0)
        selected_assets["trade_weight"] = selected_assets["trade_weight"].fillna(0.0)
        selected_assets["selected_by_solver"] = selected_assets["target_weight"] > 1e-12
        notional = float(cfg.get("constraints", {}).get("portfolio_notional_vnd") or 0.0)
        selected_assets["estimated_cost"] = (
            selected_assets["transaction_cost"].fillna(0.0) * notional
        )
        selected_assets["adv_participation"] = np.where(
            selected_assets["adv_20d"].fillna(0.0) > 0,
            selected_assets["trade_weight"].abs() * notional / selected_assets["adv_20d"],
            np.nan,
        )
        selected_assets["selection_reason"] = np.where(
            selected_assets["selected_by_solver"],
            "selected_by_xy_qaoa_then_classical_weighting",
            "aur_candidate_not_selected_by_xy_qaoa",
        )
    selected_assets.to_csv(out / "selected_assets_by_fold.csv", index=False)
    latest_columns = [
        "fold", "decision_time", "ticker", "security_id", "company_name", "sector",
        "signal", "xgboost_signal", "technical_factor_signal",
        "xgboost_expected_return", "optimization_expected_return",
        "selected_by_solver", "target_weight",
        "previous_weight", "trade_weight", "estimated_cost", "liquidity_20d", "adv_20d",
        "adv_participation", "selection_reason",
    ]
    latest = pd.DataFrame(columns=latest_columns)
    if not selected_assets.empty:
        latest_fold = selected_assets["fold"].max()
        latest = selected_assets[
            (selected_assets["fold"] == latest_fold) & selected_assets["selected_by_solver"]
        ].reindex(columns=latest_columns).sort_values("target_weight", ascending=False)
    latest.to_csv(out / "latest_selected_portfolio.csv", index=False)
    latest_exposure = (
        pd.DataFrame(exposure_rows).sort_values("fold").iloc[-1].to_dict()
        if exposure_rows else {"equity_exposure": 1.0, "cash_weight": 0.0}
    )
    executed_equity_weight = float(latest["target_weight"].sum()) if not latest.empty else 0.0
    executed_cash_weight = max(0.0, 1.0 - executed_equity_weight)
    executed_expected_return = (
        float((latest["target_weight"] * latest["optimization_expected_return"]).sum())
        if not latest.empty else np.nan
    )
    (out / "latest_portfolio_summary.json").write_text(json.dumps({
        "decision_time": str(latest_exposure.get("decision_time", "")),
        "target_equity_exposure": float(latest_exposure.get("equity_exposure", 1.0)),
        "target_cash_weight": float(latest_exposure.get("cash_weight", 0.0)),
        "executed_equity_weight": executed_equity_weight,
        "executed_cash_weight": executed_cash_weight,
        "market_regime": latest_exposure.get("regime", "not_configured"),
        "target_selected_expected_return": float(latest_exposure.get(
            "selected_expected_return", np.nan
        )),
        "executed_selected_expected_return": executed_expected_return,
        "solver_solution_selection": latest_exposure.get(
            "solver_solution_selection", "highest_probability_feasible"
        ),
        "tickers": latest.get("ticker", pd.Series(dtype=str)).astype(str).tolist(),
        "weights": {
            str(row.ticker): float(row.target_weight) for row in latest.itertuples()
        },
    }, indent=2), encoding="utf-8")
    latest_md = ["# Latest selected portfolio", ""]
    if latest.empty:
        latest_md.append("No completed research portfolio.")
    else:
        latest_md.extend([
            f"- Target equity exposure before board-lot rounding: "
            f"{float(latest_exposure.get('equity_exposure', 1.0)):.4f}",
            f"- Target cash weight before board-lot rounding: "
            f"{float(latest_exposure.get('cash_weight', 0.0)):.4f}",
            f"- Executed equity weight: {executed_equity_weight:.4f}",
            f"- Executed cash weight after board-lot rounding: {executed_cash_weight:.4f}",
            f"- Market regime: {latest_exposure.get('regime', 'not_configured')}",
            f"- Target ex-ante selected-portfolio return: "
            f"{float(latest_exposure.get('selected_expected_return', np.nan)):.6f}",
            f"- Executed ex-ante selected-portfolio return: {executed_expected_return:.6f}",
            "",
        ])
        view = latest[[
            "ticker", "company_name", "sector", "target_weight", "signal",
            "optimization_expected_return", "estimated_cost"
        ]].copy()
        latest_md.extend([
            "| " + " | ".join(view.columns) + " |",
            "| " + " | ".join(["---"] * len(view.columns)) + " |",
        ])
        latest_md.extend(
            "| " + " | ".join(map(str, row)) + " |" for row in view.to_numpy()
        )
    (out / "latest_selected_portfolio.md").write_text(
        "\n".join(latest_md) + "\n", encoding="utf-8"
    )
    returns.to_csv(out / "portfolio_returns.csv", index=False)
    pd.DataFrame(ablation_rows).to_csv(out / "ablation_results.csv", index=False)
    pd.DataFrame(sensitivity_rows).to_csv(out / "sensitivity_results.csv", index=False)
    pd.DataFrame(fold_audit_rows).to_csv(out / "fold_manifest.csv", index=False)
    pd.DataFrame(feature_coverage_rows).to_csv(out / "feature_coverage_by_fold.csv", index=False)
    pd.DataFrame(tuning_rows).to_csv(out / "model_tuning.csv", index=False)
    pd.DataFrame(aur_diagnostic_rows).to_csv(out / "aur_diagnostics.csv", index=False)
    pd.DataFrame(calibration_rows).to_csv(out / "signal_calibration.csv", index=False)
    pd.DataFrame(constraint_rows).to_csv(out / "constraint_diagnostics.csv", index=False)
    pd.DataFrame(capacity_scenario_rows).to_csv(out / "capacity_scenarios.csv", index=False)
    pd.DataFrame(missing_return_rows, columns=(
        sorted(set().union(*(row.keys() for row in missing_return_rows)))
        if missing_return_rows else ["fold", "window", "ticker", "event", "observations"]
    )).to_csv(out / "missing_return_resolution.csv", index=False)
    risk_free_series = resolve_risk_free_series(paths, cfg, returns["date"])
    risk_free_series.rename_axis("date").reset_index().to_csv(
        out / "risk_free_series.csv", index=False
    )
    bootstrap_rf = float(risk_free_series.mean())
    metric_rows = []
    for strategy, g in returns.groupby("strategy"):
        strategy_returns = g.sort_values("date").set_index("date")["return"]
        metrics = financial_metrics(strategy_returns, risk_free_series)
        lo, hi = block_bootstrap_sharpe(g["return"], bootstrap_rf, cfg["seed"])
        metrics.update({"strategy": strategy, "sharpe_ci_low": lo, "sharpe_ci_high": hi})
        metric_rows.append(metrics)
    metrics_df = pd.DataFrame(metric_rows)
    metrics_df.to_csv(out / "metrics_long.csv", index=False)
    risk_free_sensitivity_rows = []
    declared_rf_rates = cfg.get("risk_free", {}).get("sensitivity_rates", [])
    for annual_rate in declared_rf_rates:
        for strategy, group in returns.groupby("strategy"):
            strategy_returns = group.sort_values("date").set_index("date")["return"]
            assumed = pd.Series(
                float(annual_rate), index=strategy_returns.index, name="risk_free_annual"
            )
            row = financial_metrics(strategy_returns, assumed)
            row.update({
                "strategy": strategy, "assumed_risk_free_annual": float(annual_rate),
                "interpretation": "sensitivity_assumption_not_observed_market_series",
            })
            risk_free_sensitivity_rows.append(row)
    pd.DataFrame(risk_free_sensitivity_rows).to_csv(
        out / "risk_free_sensitivity.csv", index=False
    )
    cost_summary = cost_ledger.groupby("strategy", as_index=False).agg(
        turnover=("turnover", "sum"), total_cost=("transaction_cost", "sum")
    ) if not cost_ledger.empty else pd.DataFrame(columns=["strategy", "turnover", "total_cost"])
    metrics_df.merge(cost_summary, on="strategy", how="left").to_csv(
        out / "strategy_metrics_summary.csv", index=False
    )
    # Descriptive regime analysis uses only trailing market information.
    market_daily = (
        prices.pivot(index="date", columns="ticker", values="ret1").mean(axis=1).sort_index()
    )
    trailing_60 = (1 + market_daily).rolling(60).apply(np.prod, raw=True) - 1
    trailing_vol = market_daily.rolling(60).std()
    vol_median = trailing_vol.expanding(min_periods=60).median()
    regime = pd.Series("sideway", index=market_daily.index)
    regime[trailing_60 > 0.05] = "bull"
    regime[trailing_60 < -0.05] = "bear"
    regime = regime + np.where(trailing_vol > vol_median, "_high_vol", "_low_vol")
    regime_rows = []
    returns_with_regime = returns.copy()
    returns_with_regime["regime"] = pd.to_datetime(returns_with_regime["date"]).map(regime)
    for (strategy, label), group in returns_with_regime.groupby(["strategy", "regime"], dropna=False):
        regime_returns = group.sort_values("date").set_index("date")["return"]
        row = financial_metrics(regime_returns, risk_free_series)
        row.update({"strategy": strategy, "regime": label})
        regime_rows.append(row)
    pd.DataFrame(regime_rows).to_csv(out / "regime_metrics.csv", index=False)
    comparisons = solvers.groupby("method").agg(
        energy_mean=("energy", "mean"), feasibility_rate=("feasibility_rate", "mean"),
        optimality_gap_mean=("optimality_gap", "mean"), runtime_seconds=("runtime_seconds", "mean"),
        runs=("method", "size")).reset_index()
    comparisons.to_csv(out / "comparisons.csv", index=False)
    tests = []
    returns_wide = returns.pivot_table(index="date", columns="strategy", values="return", aggfunc="mean")
    if "full_pipeline_xy_qaoa" in returns_wide:
        for baseline in ["equal_weight_universe", "markowitz_mean_variance", "minimum_variance",
                         "liquidity_topk_exact", "ewma_topk_exact", "adaptive_exact",
                         "adaptive_simulated_annealing", "adaptive_penalty_qaoa"]:
            if baseline in returns_wide:
                result = paired_block_bootstrap_test(
                    returns_wide["full_pipeline_xy_qaoa"], returns_wide[baseline], cfg["seed"]
                )
                result.update({"test": f"full_pipeline_xy_qaoa_vs_{baseline}",
                               "hypothesis": "H5"})
                tests.append(result)
    fold_ic = rankings.groupby("fold")[["xgboost_rank_ic", "ewma_rank_ic"]].first().dropna()
    if len(fold_ic) >= 2:
        result = paired_block_bootstrap_test(
            fold_ic["xgboost_rank_ic"], fold_ic["ewma_rank_ic"], cfg["seed"],
            samples=1000, block=max(1, min(3, len(fold_ic) // 2)),
        )
        result.update({"test": "xgboost_rank_ic_vs_ewma_rank_ic", "hypothesis": "H1"})
        tests.append(result)
    aur_df = pd.DataFrame(aur_diagnostic_rows)
    if len(aur_df) >= 2:
        result = paired_block_bootstrap_test(
            aur_df.set_index("fold")["adaptive_forward_return_mean"],
            aur_df.set_index("fold")["fixed_topm_forward_return_mean"],
            cfg["seed"], samples=1000, block=max(1, min(3, len(aur_df) // 2)),
        )
        result.update({
            "test": "adaptive_universe_forward_return_vs_fixed_topm",
            "hypothesis": "H2", "direction": "higher_is_better",
        })
        tests.append(result)
        # Lower correlation is better; reverse the pair so a positive difference
        # consistently supports the stated hypothesis.
        result = paired_block_bootstrap_test(
            aur_df.set_index("fold")["fixed_topm_abs_correlation"],
            aur_df.set_index("fold")["adaptive_abs_correlation"],
            cfg["seed"], samples=1000, block=max(1, min(3, len(aur_df) // 2)),
        )
        result.update({
            "test": "adaptive_universe_diversification_vs_fixed_topm",
            "hypothesis": "H2", "direction": "higher_is_better_after_reversal",
        })
        tests.append(result)
    quantum = solvers[solvers["method"].isin([
        "xy_qaoa_dicke_ideal_statevector", "penalty_qaoa_ideal_statevector"
    ])].copy()
    if not quantum.empty:
        feasibility = quantum.pivot_table(
            index=["fold", "seed"], columns="method", values="feasibility_rate", aggfunc="mean"
        ).dropna()
        if len(feasibility) >= 2:
            result = paired_block_bootstrap_test(
                feasibility["xy_qaoa_dicke_ideal_statevector"],
                feasibility["penalty_qaoa_ideal_statevector"], cfg["seed"],
                samples=1000, block=max(1, min(5, len(feasibility) // 2)),
            )
            result.update({"test": "xy_feasibility_vs_penalty_qaoa", "hypothesis": "H3"})
            tests.append(result)
        gaps = quantum.pivot_table(
            index=["fold", "seed"], columns="method", values="optimality_gap", aggfunc="mean"
        ).dropna()
        if len(gaps) >= 2:
            result = paired_block_bootstrap_test(
                gaps["penalty_qaoa_ideal_statevector"],
                gaps["xy_qaoa_dicke_ideal_statevector"], cfg["seed"],
                samples=1000, block=max(1, min(5, len(gaps) // 2)),
            )
            result.update({
                "test": "xy_optimality_gap_vs_penalty_qaoa",
                "hypothesis": "H4", "direction": "higher_is_better_after_reversal",
            })
            tests.append(result)
    finite_indices = [i for i, row in enumerate(tests) if np.isfinite(row["p_value"])]
    adjusted = holm_adjust([tests[i]["p_value"] for i in finite_indices])
    for i, value in zip(finite_indices, adjusted):
        tests[i]["p_value_holm"] = value
        tests[i]["conclusion"] = "significant" if value < 0.05 else "not_significant"
    for row in tests:
        row.setdefault("p_value_holm", np.nan)
        row.setdefault("conclusion", "insufficient_observations")
    tests_frame = pd.DataFrame(tests or [{
        "test": "no_valid_pair", "hypothesis": "none", "p_value": np.nan,
        "p_value_holm": np.nan, "conclusion": "insufficient_observations",
    }])
    tests_frame.to_csv(out / "statistical_tests.csv", index=False)
    hypothesis_rows = []
    for hypothesis in ["H1", "H2", "H3", "H4", "H5"]:
        subset = tests_frame[tests_frame.get("hypothesis", pd.Series(dtype=str)).eq(hypothesis)]
        if subset.empty:
            status = "not_testable_due_to_data"
        elif subset["conclusion"].eq("significant").any():
            status = "statistically_supported"
        else:
            status = "not_statistically_supported"
        hypothesis_rows.append({
            "hypothesis": hypothesis, "status": status,
            "tests": "|".join(subset.get("test", pd.Series(dtype=str)).astype(str)),
            "interpretation_scope": "conditional_on_period_universe_costs_and_model_specification",
        })
    hypothesis_rows.append({
        "hypothesis": "H6",
        "status": "sensitivity_completed" if sensitivity_rows else "not_testable_due_to_data",
        "tests": "declared_sensitivity_grid",
        "interpretation_scope": "no_claim_outside_tested_grid",
    })
    pd.DataFrame(hypothesis_rows).to_csv(out / "hypothesis_results.csv", index=False)
    if not returns.empty:
        pivot = returns.pivot_table(index="date", columns="strategy", values="return", aggfunc="mean")
        if mode == "research":
            chart_title = "Walk-forward cumulative wealth - verified research data"
        elif mode == "exploratory":
            chart_title = "Walk-forward cumulative wealth - exploratory complete-case HOSE data"
        else:
            chart_title = "Demo cumulative wealth - fixture"
        (1 + pivot).cumprod().plot(title=chart_title)
        plt.ylabel("Growth of 1 unit")
        plt.tight_layout()
        plt.savefig(fig_dir / "equity_curve.png", dpi=160)
        plt.close()
        wealth = (1 + pivot).cumprod()
        drawdown = wealth.div(wealth.cummax()).sub(1)
        drawdown.plot(title="Walk-forward drawdown")
        plt.ylabel("Drawdown")
        plt.tight_layout()
        plt.savefig(fig_dir / "drawdown.png", dpi=160)
        plt.close()
    if not metrics_df.empty:
        metrics_df.plot.scatter(
            x="annualized_volatility", y="annualized_return", c="sharpe",
            colormap="viridis", title="Risk-return comparison",
        )
        for row in metrics_df.itertuples():
            plt.annotate(str(row.strategy), (row.annualized_volatility, row.annualized_return), fontsize=6)
        plt.tight_layout()
        plt.savefig(fig_dir / "risk_return.png", dpi=160)
        plt.close()
    if not comparisons.empty:
        comparisons.set_index("method")[["feasibility_rate"]].plot.bar(
            title="Solver feasibility rate", legend=False
        )
        plt.tight_layout()
        plt.savefig(fig_dir / "feasibility_rate.png", dpi=160)
        plt.close()
        comparisons.set_index("method")[["optimality_gap_mean"]].plot.bar(
            title="Mean optimality gap", legend=False
        )
        plt.tight_layout()
        plt.savefig(fig_dir / "optimality_gap.png", dpi=160)
        plt.close()
    if not cost_ledger.empty:
        cost_ledger.groupby("strategy")[["turnover", "transaction_cost"]].sum().plot.bar(
            secondary_y="transaction_cost", title="Turnover and transaction cost"
        )
        plt.tight_layout()
        plt.savefig(fig_dir / "turnover_and_cost.png", dpi=160)
        plt.close()
    if not rankings.empty:
        rankings.groupby("fold")[["xgboost_rank_ic", "ewma_rank_ic"]].first().plot(
            marker="o", title="Rank IC by fold"
        )
        plt.tight_layout()
        plt.savefig(fig_dir / "rank_ic_by_fold.png", dpi=160)
        plt.close()
    sensitivity_frame = pd.DataFrame(sensitivity_rows)
    if not sensitivity_frame.empty:
        sensitivity_frame.groupby("depth_p")[["optimality_gap", "feasibility_rate"]].mean().plot(
            marker="o", title="QAOA sensitivity by depth"
        )
        plt.tight_layout()
        plt.savefig(fig_dir / "sensitivity_analysis.png", dpi=160)
        plt.close()
    (out / "resolved_config.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    (out / "data_quality.json").write_text(json.dumps(quality, indent=2), encoding="utf-8")
    (out / "leakage_audit.json").write_text(json.dumps(leak, indent=2), encoding="utf-8")
    (out / "data_quality.md").write_text(
        "# Data quality\n\n"
        f"- Status: `{quality['status']}`\n- Class: `{quality['data_class']}`\n"
        f"- Records: {quality['records']}\n- Tickers: {quality['tickers']}\n"
        f"- Issues: `{quality['issues']}`\n", encoding="utf-8"
    )
    (out / "leakage_audit.md").write_text(
        "# Leakage audit\n\n"
        f"- Status: `{leak['status']}`\n- Blockers: `{leak['blockers']}`\n\n"
        f"{leak['note']}\n", encoding="utf-8"
    )
    try:
        git_commit = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=config_path.parent,
            text=True, stderr=subprocess.DEVNULL,
        ).strip()
        git_status = subprocess.check_output(
            ["git", "status", "--porcelain=v1"], cwd=config_path.parent,
            text=True, stderr=subprocess.DEVNULL,
        ).strip().splitlines()
    except (OSError, subprocess.CalledProcessError):
        git_commit = "unknown"
        git_status = ["unavailable"]
    source_hashes = {
        "src/research.py": sha256_file(Path(__file__)),
        "src/cli.py": sha256_file(Path(__file__).with_name("cli.py")),
        "config": sha256_file(config_path),
    }
    env = (
        f"python={sys.version}\nplatform={platform.platform()}\n"
        f"git_commit={git_commit}\ngit_dirty={bool(git_status)}\n"
        f"source_sha256={json.dumps(source_hashes, sort_keys=True)}\n"
    )
    (out / "environment.txt").write_text(env, encoding="utf-8")
    provenance = {
        "price_sources": prices.groupby(
            ["source", "source_url", "data_class"], dropna=False
        ).size().reset_index(name="records").to_dict("records"),
        "price_dataset_sha256": sha256_file(paths.normalized / "prices.parquet"),
        "universe_dataset_sha256": sha256_file(paths.curated / "universe_monthly.parquet"),
        "source_manifest": str(paths.raw / "manifest.json"),
        "security_master_sha256": sha256_file(paths.normalized / "security_master.parquet"),
        "corporate_actions_sha256": (
            sha256_file(paths.normalized / "corporate_actions.parquet")
            if (paths.normalized / "corporate_actions.parquet").exists() else None
        ),
        "benchmark_sha256": sha256_file(benchmark_path) if benchmark_path.exists() else None,
    }
    (out / "data_provenance.json").write_text(
        json.dumps(provenance, indent=2, default=str), encoding="utf-8"
    )
    actual_oos_start = str(pd.to_datetime(returns["date"]).min().date()) if not returns.empty else None
    actual_oos_end = str(pd.to_datetime(returns["date"]).max().date()) if not returns.empty else None
    adjustment_contract_path = paths.normalized / "price_adjustment_contract.json"
    adjustment_metadata = (
        json.loads(adjustment_contract_path.read_text(encoding="utf-8"))
        if adjustment_contract_path.exists() else {}
    )
    manifest = {
        "experiment_id": experiment_id, "status": "success", "mode": cfg.get("mode"),
        "label": cfg["label"],
        "data_class": quality["data_class"], "started_from_config": str(config_path),
        "created_at": datetime.now(timezone.utc).isoformat(), "config_hash": cfg_hash,
        "dataset_hash": sha256_file(paths.normalized / "prices.parquet"),
        "universe_hash": sha256_file(paths.curated / "universe_monthly.parquet"),
        "git_commit": git_commit, "git_dirty": bool(git_status),
        "git_status_porcelain": git_status, "source_sha256": source_hashes,
        "adjustment_version": adjustment_metadata.get(
            "adjustment_version", cfg.get("data", {}).get("adjustment_version", "unknown")
        ),
        "adjustment_contract_sha256": (
            sha256_file(adjustment_contract_path) if adjustment_contract_path.exists() else None
        ),
        "audit_status": leak.get("status"),
        "requested_data_start": cfg.get("data", {}).get("start"),
        "requested_data_end": cfg.get("data", {}).get("end"),
        "actual_data_start": quality.get("start"), "actual_data_end": quality.get("end"),
        "actual_oos_start": actual_oos_start, "actual_oos_end": actual_oos_end,
        "folds_requested": len(folds), "folds_completed": int(returns["fold"].nunique()) if not returns.empty else 0,
        "artifacts": [],
    }
    (out / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    create_report(out, cfg, quality, leak, metrics_df, comparisons, rankings)
    report_kind = {
        "research": "Research run report",
        "exploratory": "Exploratory complete-case run report",
    }.get(mode, "Demo run report")
    run_report = [
        f"# {report_kind}", "", f"- Experiment: `{experiment_id}`",
        f"- Status: `success`", f"- Label: **{cfg['label']}**",
        f"- Data records: {quality['records']}", f"- Tickers: {quality['tickers']}",
        f"- Folds requested/completed: {len(folds)}/{manifest['folds_completed']}",
        f"- Data quality: `{quality['status']}`", f"- Leakage audit: `{leak['status']}`",
        f"- Command: `python -m src.cli run-experiment --config {config_path}`",
        "- Quantum backend: internal ideal fixed-Hamming-weight statevector simulator; not hardware.",
        f"- Limitations: {', '.join(leak.get('limitations', [])) or 'none reported'}.",
        "", "## Artifact index", "",
    ]
    current = sorted(p.relative_to(out).as_posix() for p in out.rglob("*") if p.is_file())
    run_report.extend(f"- `{name}`" for name in current)
    run_report_name = {
        "research": "RUN_REPORT.md",
        "exploratory": "EXPLORATORY_RUN_REPORT.md",
    }.get(mode, "DEMO_RUN_REPORT.md")
    (out / run_report_name).write_text("\n".join(run_report) + "\n", encoding="utf-8")
    manifest["artifacts"] = sorted(
        p.relative_to(out).as_posix() for p in out.rglob("*") if p.is_file()
    )
    manifest["artifact_sha256"] = {
        p.relative_to(out).as_posix(): sha256_file(p)
        for p in out.rglob("*") if p.is_file() and p.name != "manifest.json"
    }
    (out / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return out


def create_report(out: Path, cfg: dict, quality: dict, leak: dict, metrics: pd.DataFrame,
                  comparisons: pd.DataFrame, rankings: pd.DataFrame) -> None:
    def markdown_table(df: pd.DataFrame) -> str:
        if df.empty:
            return "No results."
        clean = df.copy()
        for col in clean.select_dtypes(include=[np.number]).columns:
            clean[col] = clean[col].map(lambda x: "" if pd.isna(x) else f"{x:.6g}")
        header = "| " + " | ".join(map(str, clean.columns)) + " |"
        rule = "| " + " | ".join(["---"] * len(clean.columns)) + " |"
        rows = ["| " + " | ".join(map(str, row)) + " |" for row in clean.astype(str).to_numpy()]
        return "\n".join([header, rule, *rows])
    label = cfg["label"]
    ic = rankings.groupby("fold")["fold_rank_ic"].first().mean() if not rankings.empty else np.nan
    ablations = pd.read_csv(out / "ablation_results.csv") if (out / "ablation_results.csv").exists() else pd.DataFrame()
    sensitivity = pd.read_csv(out / "sensitivity_results.csv") if (out / "sensitivity_results.csv").exists() else pd.DataFrame()
    statistics = pd.read_csv(out / "statistical_tests.csv") if (out / "statistical_tests.csv").exists() else pd.DataFrame()
    regimes = pd.read_csv(out / "regime_metrics.csv") if (out / "regime_metrics.csv").exists() else pd.DataFrame()
    is_research = cfg.get("mode") == "research"
    is_exploratory = cfg.get("mode") == "exploratory"
    manifest = json.loads((out / "manifest.json").read_text(encoding="utf-8"))
    h1_test = (statistics[statistics["hypothesis"] == "H1"]
               if "hypothesis" in statistics else pd.DataFrame())
    h5_tests = (statistics[statistics["hypothesis"] == "H5"]
                if "hypothesis" in statistics else pd.DataFrame())
    interpretation_prefix = (
        "research" if is_research else ("exploratory-only" if is_exploratory else "demo-only")
    )
    hypotheses = [
        ("H1", interpretation_prefix,
         f"Mean XGBoost walk-forward Rank IC={ic:.4f}; paired XGBoost–EWMA test rows={len(h1_test)}."),
        ("H2", interpretation_prefix,
         "AUR diagnostics report signal, liquidity, risk, correlation, selected M and candidate turnover; causal superiority is not inferred."),
        ("H3", "implementation-supported" if is_research else interpretation_prefix,
         "Fixed-weight XY simulation preserves cardinality by construction; penalty feasibility is measured from samples."),
        ("H4", interpretation_prefix,
         "Primary-solution and best-observed gaps are separated against the exact small-instance oracle."),
        ("H5", interpretation_prefix,
         f"Net buy-and-hold performance uses common costs; paired benchmark comparisons={len(h5_tests)}."),
        ("H6", interpretation_prefix,
         "Sensitivity reruns solver/accounting on the declared grid and representative folds; inference is conditional on that grid."),
    ]
    if is_research:
        title = "AI-Quantum Portfolio Research Report"
        scope = (
            "This run evaluates the complete walk-forward pipeline on a normalized "
            "real-market panel that satisfies the declared research contracts."
        )
    elif is_exploratory:
        title = "AI-Quantum Portfolio Exploratory Complete-Case Report"
        scope = (
            "This run uses real HOSE price observations retained by an explicit "
            "complete-case rule. It verifies the empirical pipeline on usable data, "
            "but it is not confirmatory evidence for the full-HOSE study because the "
            "adjustment contract and full point-in-time coverage are not established."
        )
    else:
        title = "AI-Quantum Portfolio Demo Report"
        scope = (
            "This run verifies the software path end-to-end. It is not evidence for "
            "the 2015-2025 HOSE study."
        )
    lines = [
        f"# {title}", "", f"> **{label}**", "",
        "| Report field | Value |", "|---|---|",
        f"| Experiment ID | `{manifest.get('experiment_id')}` |",
        f"| Dataset hash | `{manifest.get('dataset_hash')}` |",
        f"| Adjustment version | `{manifest.get('adjustment_version')}` |",
        f"| Config hash | `{manifest.get('config_hash')}` |",
        f"| Git commit | `{manifest.get('git_commit')}` |",
        f"| Created at | `{manifest.get('created_at')}` |",
        f"| Mode | `{manifest.get('mode')}` |",
        f"| Research/exploratory label | `{manifest.get('label')}` |",
        f"| Folds | `{manifest.get('folds_completed')}/{manifest.get('folds_requested')}` |",
        f"| OOS start/end | `{manifest.get('actual_oos_start')}` / `{manifest.get('actual_oos_end')}` |",
        f"| Audit status | `{manifest.get('audit_status')}` |", "",
        "## Scope", "",
        scope,
        "", "## Data validation", "", f"- Quality: `{quality['status']}`",
        f"- Leakage audit: `{leak['status']}`", f"- Records: {quality['records']}",
        f"- Tickers: {quality['tickers']}",
        f"- Requested range: {manifest.get('requested_data_start')} to {manifest.get('requested_data_end')}",
        f"- Actual data range: {manifest.get('actual_data_start')} to {manifest.get('actual_data_end')}",
        f"- Actual OOS range: {manifest.get('actual_oos_start')} to {manifest.get('actual_oos_end')}",
        f"- Folds completed/requested: {manifest.get('folds_completed')}/{manifest.get('folds_requested')}",
        "", "## Predictive ranking", "",
        f"- Mean fold rank IC: {ic:.6f}", "", "## Solver comparison", "",
        markdown_table(comparisons),
        "", "## Portfolio metrics", "",
        markdown_table(metrics),
        "", "## Ablation study", "",
        markdown_table(
            ablations.groupby(["configuration", "selector", "solver"]).agg(
                objective_mean=("objective", "mean"),
                optimality_gap_mean=("optimality_gap", "mean"),
                feasibility_rate=("feasibility_rate", "mean"),
                folds=("fold", "nunique"),
            ).reset_index()
        ) if not ablations.empty else "No ablation results.",
        "", "## Robustness and sensitivity", "",
        markdown_table(
            sensitivity.groupby(["depth_p", "shots", "cardinality",
                                 "uniform_probability_noise_proxy", "transaction_cost_bps"]).agg(
                optimality_gap=("optimality_gap", "mean"),
                feasibility_rate=("feasibility_rate", "mean"),
                runtime_seconds=("runtime_seconds", "mean"),
            ).reset_index()
        ) if not sensitivity.empty else "No sensitivity results.",
        "", "## Statistical comparison", "",
        markdown_table(statistics),
        "", "## Market-regime description", "",
        markdown_table(regimes),
        "", "## H1–H6 interpretation", "",
    ]
    for h, status, reason in hypotheses:
        lines.append(f"- **{h}: {status}.** {reason}")
    if is_research:
        limitations = [
            "- Research execution requires a verified historical universe; current-listing or first-price proxies are blocked before this report can be produced.",
            "- Corporate actions are core whenever prices are not covered by a verified adjusted-price contract; optional fundamentals, macroeconomic data and foreign flow are excluded when verified point-in-time tables are unavailable.",
            "- A verified total-return market benchmark is required when `benchmark.required` is enabled.",
            "- Missing realized returns are resolved only as logged non-trading marks or verified delisting liquidations; unexplained disappearance blocks research execution.",
            "- The XY-QAOA implementation is an ideal fixed-Hamming-weight statevector simulator, not quantum hardware.",
            "- The optional depolarizing/readout channels are phenomenological simulator stress tests, not a calibrated hardware noise model.",
            "- Statistical results are conditional on the selected period, universe, costs and model specification; they are not investment advice or proof of quantum advantage.",
        ]
        reproduce = "python -m src.cli run-experiment --config configs/hose300_real.yaml"
    elif is_exploratory:
        limitations = [
            "- Securities are retained using full-period availability, which can create coverage or survivorship selection bias.",
            "- Price completeness does not certify corporate-action adjustment; abnormal economic returns may remain even after row-level quality checks.",
            "- No verified total-return benchmark is used, and optional point-in-time fundamentals, macroeconomic variables and foreign flow are excluded.",
            "- Results are exploratory and must not be described as confirmatory evidence for all HOSE securities.",
            "- The XY-QAOA implementation is an ideal fixed-Hamming-weight statevector simulator, not quantum hardware.",
            "- Statistical results are conditional on the retained sample, period, costs and tested parameter grid.",
        ]
        reproduce = (
            "python -m src.cli run-complete-case --config "
            "configs/hose300_complete_case_exploratory.yaml"
        )
    else:
        limitations = [
            "- Data are deterministic fixtures, explicitly not real HOSE observations.",
            "- The XY-QAOA implementation is an ideal fixed-Hamming-weight statevector simulator, not quantum hardware.",
            "- Penalty-QAOA and XY-QAOA are ideal internal statevector simulations, not quantum hardware.",
            "- Robustness, regimes and statistical tests are implemented, but fixture results are not confirmatory evidence.",
        ]
        reproduce = "python -m src.cli run-experiment --config configs/quick.yaml"
    lines += ["", "## Limitations", ""] + limitations + [
        "", "## Reproduce", "", "```powershell", reproduce, "```", "",
    ]
    md = "\n".join(lines)
    (out / "RESEARCH_REPORT.md").write_text(md, encoding="utf-8")
    html = "<html><meta charset='utf-8'><body>" + (
        md.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;").replace("\n", "<br>\n")
    ) + "</body></html>"
    (out / "report.html").write_text(html, encoding="utf-8")


In [ ]:
%%writefile /content/ai_quantum_standalone/scripts/__init__.py
"""Standalone helper scripts."""


In [ ]:
%%writefile /content/ai_quantum_standalone/scripts/import_colab_complete_csv.py
"""Validate one complete Colab CSV and restore the Data 17/8 runtime workspace."""

from __future__ import annotations

import argparse
import json
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_pipeline import (
    PRICE_COLUMNS,
    Paths,
    build_universe,
    leakage_audit,
    sha256_file,
    validate_data,
)


DEFAULT_WORKSPACE = ROOT / "outputs" / "Data 17_8"

BENCHMARK_COLUMNS = [
    "date", "benchmark", "total_return_index", "index_type", "methodology_url",
    "available_at", "source", "source_url", "fetched_at", "data_class",
]
SECURITY_COLUMNS = [
    "security_id", "ticker", "company_name", "exchange", "isin", "figi",
    "hose_security_id", "listing_date", "delisting_date", "effective_from",
    "effective_to", "available_at", "source", "source_url", "fetched_at",
    "history_method", "data_class", "raw_checksum",
]
ACTION_COLUMNS = [
    "security_id", "ticker", "event_type", "announcement_date", "record_date",
    "ex_date", "effective_date", "payment_date", "cash_dividend_per_share",
    "stock_dividend_ratio", "bonus_share_ratio", "split_ratio",
    "reverse_split_ratio", "rights_ratio", "rights_subscription_price",
    "adjustment_factor", "currency", "source", "source_url",
    "corroboration_source", "corroboration_url", "fetched_at", "available_at",
    "raw_checksum", "parser_version", "verification_status", "verification_notes",
]


def require_columns(frame: pd.DataFrame, columns: list[str], table: str) -> None:
    missing = sorted(set(columns) - set(frame.columns))
    if missing:
        raise ValueError(f"{table} is missing required columns: {missing}")


def parse_dates(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    output = frame.copy()
    for column in columns:
        if column in output:
            output[column] = pd.to_datetime(output[column], errors="coerce", format="mixed")
    return output


def bool_series(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes"})


def import_dataset(csv_path: Path, workspace: Path, replace: bool = True) -> dict:
    frame = pd.read_csv(csv_path, encoding="utf-8-sig", low_memory=False)
    if "record_type" not in frame:
        raise ValueError("CSV must contain record_type.")
    counts = frame["record_type"].value_counts().to_dict()
    required_types = {"METADATA", "PRICE", "BENCHMARK", "SECURITY", "CORPORATE_ACTION"}
    missing_types = sorted(required_types - set(counts))
    if missing_types:
        raise ValueError(f"CSV is missing record types: {missing_types}")

    prices = frame.loc[frame.record_type.eq("PRICE")].copy()
    benchmark = frame.loc[frame.record_type.eq("BENCHMARK")].copy()
    securities = frame.loc[frame.record_type.eq("SECURITY")].copy()
    actions = frame.loc[frame.record_type.eq("CORPORATE_ACTION")].copy()
    require_columns(prices, PRICE_COLUMNS, "PRICE")
    require_columns(benchmark, BENCHMARK_COLUMNS, "BENCHMARK")
    require_columns(securities, SECURITY_COLUMNS + ["runtime_eligible"], "SECURITY")
    require_columns(actions, ACTION_COLUMNS, "CORPORATE_ACTION")

    prices = parse_dates(prices[PRICE_COLUMNS], ["date", "available_at"])
    benchmark = parse_dates(
        benchmark[BENCHMARK_COLUMNS], ["date", "available_at"]
    )
    full_master = parse_dates(
        securities[SECURITY_COLUMNS],
        ["listing_date", "delisting_date", "effective_from", "effective_to", "available_at"],
    )
    runtime_mask = bool_series(securities["runtime_eligible"])
    runtime_master = full_master.loc[runtime_mask.to_numpy()].copy()
    if "research_eligibility_status" in securities:
        runtime_master["research_eligibility_status"] = securities.loc[
            runtime_mask, "research_eligibility_status"
        ].to_numpy()
        runtime_master["research_eligibility_as_of"] = pd.Timestamp("2026-08-17")
    actions = parse_dates(
        actions[ACTION_COLUMNS],
        ["announcement_date", "record_date", "ex_date", "effective_date", "payment_date", "available_at"],
    )

    if prices.empty or runtime_master.empty or benchmark.empty:
        raise ValueError("PRICE, runtime SECURITY and BENCHMARK records must be non-empty.")
    if prices.duplicated(["ticker", "date"]).any():
        raise ValueError("Duplicate ticker-date rows in PRICE records.")
    if not set(prices["ticker"].astype(str)) <= set(runtime_master["ticker"].astype(str)):
        raise ValueError("Every PRICE ticker must exist in the runtime security master.")
    price_dates = set(prices["date"].dropna())
    benchmark_dates = set(benchmark["date"].dropna())
    if price_dates != benchmark_dates:
        raise ValueError("BENCHMARK dates must exactly match PRICE trading dates.")

    resolved = workspace.resolve()
    project_outputs = (ROOT / "outputs").resolve()
    if project_outputs not in resolved.parents:
        raise ValueError(f"Workspace must stay under {project_outputs}: {resolved}")
    if workspace.exists() and replace:
        shutil.rmtree(workspace)
    paths = Paths(workspace)
    for directory in [paths.normalized, paths.curated, paths.reports, paths.raw]:
        directory.mkdir(parents=True, exist_ok=True)

    prices.to_parquet(paths.normalized / "prices.parquet", index=False)
    benchmark.to_parquet(paths.normalized / "benchmark.parquet", index=False)
    runtime_master.to_parquet(paths.normalized / "security_master.parquet", index=False)
    full_master.to_parquet(paths.normalized / "security_master_full.parquet", index=False)
    actions.to_parquet(paths.normalized / "corporate_actions.parquet", index=False)

    price_hash = sha256_file(paths.normalized / "prices.parquet")
    contract = {
        "dataset": "Data 17/8 - complete CSV import",
        "adjustment_policy": "verified_vendor_total_return_adjusted",
        "source": "cafef_raw_kbs_adjusted_crosscheck_packaged_csv",
        "source_url": "local-upload://ai_quantum_complete_dataset.csv",
        "methodology": (
            "The CSV preserves the previously verified CafeF raw OHLC and KBS adjusted "
            "close observations with row-level provenance; no prices are recomputed on import."
        ),
        "certified_by": "colab-complete-csv-schema-and-hash-audit",
        "certified_at": datetime.now(timezone.utc).isoformat(),
        "input_csv_sha256": sha256_file(csv_path),
        "output_price_dataset_sha256": price_hash,
    }
    (paths.normalized / "price_adjustment_contract.json").write_text(
        json.dumps(contract, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    crosscheck = {
        "dataset": contract["dataset"],
        "status": "packaged_verified_panel",
        "requested_tickers": int(prices["ticker"].nunique()),
        "cafef_series_collected": int(prices["ticker"].nunique()),
        "cross_source_verified_tickers": int(prices["ticker"].nunique()),
        "rows": len(prices),
        "dates": int(prices["date"].nunique()),
        "failures": [],
        "raw_price_source": "CafeF public PriceHistory endpoint (preserved package)",
        "adjusted_price_source": "KBS public endpoint through vnstock (preserved package)",
        "adjustment_policy": "verified_vendor_total_return_adjusted",
        "sha256": price_hash,
    }
    (paths.reports / "cafef_price_crosscheck_audit.json").write_text(
        json.dumps(crosscheck, indent=2, ensure_ascii=False), encoding="utf-8"
    )

    quality, coverage = validate_data(paths)
    universe = build_universe(
        paths,
        rebalance="monthly",
        definition="hose_all_listed",
        max_assets=300,
        liquidity_lookback_days=60,
        minimum_observations=40,
    )
    leak = leakage_audit(paths)
    exploratory_permitted = bool(
        quality["status"] == "pass"
        and leak["status"] in {"pass", "pass_with_limitations"}
        and not prices.empty
        and not benchmark.empty
    )
    packaged_audit = {
        "dataset": "Data 17/8 - complete CSV import",
        "status": "blocked",
        "research_ready": False,
        "exploratory_run_permitted": exploratory_permitted,
        "checks": {
            "price_panel": {"passed": quality["status"] == "pass", "rows": len(prices)},
            "official_total_return_benchmark": {"passed": not benchmark.empty, "rows": len(benchmark)},
            "corporate_action_ledger": {"passed": not actions.empty, "rows": len(actions)},
            "company_document_repository": {"passed": False, "reason": "not_embedded_in_single_csv"},
        },
        "blockers": ["company_document_repository_not_embedded_in_single_csv"],
        "interpretation": (
            "The single CSV is complete for the exploratory model runtime. Raw disclosure "
            "binaries and optional PIT financial features are not embedded, so it is not a "
            "confirmatory full-HOSE research package."
        ),
    }
    (paths.reports / "DATA_17_8_AUDIT.json").write_text(
        json.dumps(packaged_audit, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    result = {
        "input_csv": str(csv_path),
        "input_csv_sha256": sha256_file(csv_path),
        "workspace": str(workspace),
        "record_counts": {str(key): int(value) for key, value in counts.items()},
        "price_rows": len(prices),
        "runtime_tickers": int(prices["ticker"].nunique()),
        "full_master_tickers": int(full_master["ticker"].nunique()),
        "benchmark_rows": len(benchmark),
        "corporate_action_rows": len(actions),
        "universe_rows": len(universe),
        "quality_status": quality["status"],
        "leakage_status": leak["status"],
        "exploratory_run_permitted": exploratory_permitted,
        "confirmatory_audit_status": packaged_audit["status"],
        "coverage_rows": len(coverage),
    }
    (paths.reports / "COLAB_CSV_IMPORT_REPORT.json").write_text(
        json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    return result


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("csv", type=Path)
    parser.add_argument("--workspace", type=Path, default=DEFAULT_WORKSPACE)
    parser.add_argument("--no-replace", action="store_true")
    args = parser.parse_args()
    print(json.dumps(
        import_dataset(args.csv.resolve(), args.workspace.resolve(), not args.no_replace),
        indent=2,
        ensure_ascii=False,
    ))


if __name__ == "__main__":
    main()


In [ ]:
# Materialize the documented standalone module boundaries.
from pathlib import Path
FACADE_SOURCES = {
  "config.py": "\"\"\"Central standalone configuration contract.\"\"\"\nfrom dataclasses import dataclass, asdict\n\n@dataclass(frozen=True)\nclass ExperimentConfig:\n    random_seed: int = 42\n    start_date: str = \"2020-01-01\"\n    end_date: str = \"2025-12-31\"\n    training_months: int = 24\n    validation_months: int = 3\n    testing_months: int = 1\n    rebalance_frequency: str = \"monthly\"\n    maximum_universe_size: int = 300\n    candidate_count: int = 8\n    cardinality: int = 4\n    minimum_history: int = 40\n    maximum_weight: float = 0.40\n    minimum_weight: float = 0.05\n    risk_aversion: float = 1.25\n    transaction_cost_bps: float = 10.0\n    slippage_bps: float = 5.0\n    turnover_penalty: float = 0.01\n    qaoa_depth: int = 2\n    shots: int = 1024\n    optimizer_iterations: int = 45\n    simulated_annealing_iterations: int = 800\n    bootstrap_iterations: int = 500\n    significance_level: float = 0.05\n    execution_profile: str = \"FULL\"\n\n    def to_dict(self) -> dict:\n        return asdict(self)\n",
  "data_contracts.py": "\"\"\"Required record types and schema for the uploaded research CSV.\"\"\"\nREQUIRED_RECORD_TYPES = {\"METADATA\", \"PRICE\", \"BENCHMARK\", \"SECURITY\", \"CORPORATE_ACTION\"}\nPRICE_REQUIRED = {\"ticker\", \"date\", \"open\", \"high\", \"low\", \"close\", \"volume\", \"available_at\"}\nBENCHMARK_REQUIRED = {\"date\", \"total_return_index\", \"available_at\"}\nSECURITY_REQUIRED = {\"ticker\", \"company_name\", \"exchange\", \"listing_date\", \"available_at\"}\nACTION_REQUIRED = {\"ticker\", \"event_type\", \"ex_date\", \"available_at\"}\n",
  "data_loader.py": "\"\"\"Public data-loading facade for the standalone notebook.\"\"\"\nfrom pathlib import Path\nfrom scripts.import_colab_complete_csv import import_dataset\n\ndef load_complete_csv(csv_path: Path, workspace: Path) -> dict:\n    return import_dataset(csv_path, workspace, replace=True)\n",
  "data_quality.py": "\"\"\"Data quality facade.\"\"\"\nfrom src.data_pipeline import validate_data\n__all__ = [\"validate_data\"]\n",
  "leakage_audit.py": "\"\"\"Point-in-time leakage audit facade.\"\"\"\nfrom src.data_pipeline import leakage_audit\n__all__ = [\"leakage_audit\"]\n",
  "features.py": "\"\"\"Point-in-time feature engineering facade.\"\"\"\nfrom src.research import build_features, attach_point_in_time_features\n__all__ = [\"build_features\", \"attach_point_in_time_features\"]\n",
  "walk_forward.py": "\"\"\"Walk-forward split and purge facade.\"\"\"\nfrom src.research import make_folds, purged_fold_frames\n__all__ = [\"make_folds\", \"purged_fold_frames\"]\n",
  "signals.py": "\"\"\"XGBoost ranking and prediction facade.\"\"\"\nfrom src.research import fit_ranker, predict, calibrate_rank_signal_to_returns\n__all__ = [\"fit_ranker\", \"predict\", \"calibrate_rank_signal_to_returns\"]\n",
  "adaptive_universe.py": "\"\"\"Adaptive Universe Reduction facade.\"\"\"\nfrom src.research import adaptive_reduce\n__all__ = [\"adaptive_reduce\"]\n",
  "covariance.py": "\"\"\"Multivariate EWMA covariance facade.\"\"\"\nfrom src.research import ewma_mean_cov\n__all__ = [\"ewma_mean_cov\"]\n",
  "qubo.py": "\"\"\"Cardinality-constrained QUBO facade.\"\"\"\nfrom src.research import qubo_instance, energy, feasible_states\n__all__ = [\"qubo_instance\", \"energy\", \"feasible_states\"]\n",
  "exact_solver.py": "\"\"\"Exact combinatorial solver facade.\"\"\"\nfrom src.research import exact_solver\n__all__ = [\"exact_solver\"]\n",
  "simulated_annealing.py": "\"\"\"Simulated annealing solver facade.\"\"\"\nfrom src.research import simulated_annealing\n__all__ = [\"simulated_annealing\"]\n",
  "penalty_qaoa.py": "\"\"\"Full-Hilbert-space Penalty-QAOA facade.\"\"\"\nfrom src.research import penalty_qaoa_statevector\n__all__ = [\"penalty_qaoa_statevector\"]\n",
  "xy_qaoa.py": "\"\"\"Dicke-state feasible-subspace XY-QAOA facade.\"\"\"\nfrom src.research import xy_qaoa_statevector\n__all__ = [\"xy_qaoa_statevector\"]\n",
  "weight_optimizer.py": "\"\"\"Constrained classical weight optimizer facade.\"\"\"\nfrom src.research import optimize_weights\n__all__ = [\"optimize_weights\"]\n",
  "backtest.py": "\"\"\"Out-of-sample accounting facade.\"\"\"\nfrom src.research import record_rebalanced_strategy, financial_metrics, transaction_cost_breakdown\n__all__ = [\"record_rebalanced_strategy\", \"financial_metrics\", \"transaction_cost_breakdown\"]\n",
  "statistics.py": "\"\"\"Block bootstrap and Holm correction facade.\"\"\"\nfrom src.research import paired_block_bootstrap_test, holm_adjust\n__all__ = [\"paired_block_bootstrap_test\", \"holm_adjust\"]\n",
  "reporting.py": "\"\"\"Research reporting facade.\"\"\"\nfrom src.research import create_report\n__all__ = [\"create_report\"]\n",
  "pipeline.py": "\"\"\"End-to-end research pipeline facade.\"\"\"\nfrom src.research import run_experiment\n__all__ = [\"run_experiment\"]\n"
}
for name, source in FACADE_SOURCES.items():
    target = STANDALONE_ROOT / 'ai_quantum_system' / name
    target.write_text(source, encoding='utf-8')
print('Facade modules written:', len(FACADE_SOURCES))


In [ ]:
%%writefile /content/ai_quantum_standalone/configs/standalone_full.yaml
mode: exploratory
label: "DATA 17/8 - OFFICIAL DISCLOSURE ENRICHED, FAIL-CLOSED EXPLORATORY HOSE PIPELINE"
seed: 42

data:
  source: data_17_8_hose_disclosure_enriched
  start: "2020-01-01"
  end: "2025-12-31"
  rebalance: monthly

universe:
  definition: hose_all_listed
  index_code: null
  max_assets: 300
  liquidity_lookback_days: 60
  minimum_observations: 40

walk_forward:
  train_months: 24
  validation_months: 3
  test_months: 1
  max_folds: null
  embargo_days: 20
  selection: all
  continuous_monthly: true
  final_holdout_months: 12
  holdout_policy: freeze_before_single_evaluation

target:
  horizon_days: 20

model:
  kind: xgboost
  signal_mode: validation_blend
  xgboost_weight_grid: [0.25, 0.50, 0.75, 1.00]
  blend_stability_penalty: 0.25
  n_estimators: 180
  max_depth: 4
  learning_rate: 0.035
  min_feature_coverage: 0.05
  tuning_scope: development_period_only

reduction:
  qubit_budget: 8
  candidate_size: 8
  min_candidate_size: 8
  max_candidate_size: 8
  cardinality: 4
  signal_weight: 0.40
  liquidity_weight: 0.35
  risk_weight: 0.15
  correlation_penalty: 0.10
  stability_weight: 0.20
  minimum_candidate_retention: 0.50
  low_signal_dispersion_ratio: 0.0
  high_correlation_threshold: 0.65
  liquidity_floor_quantile: 0.20
  risk_ceiling_quantile: 0.90
  correlation_cluster_threshold: 0.70
  max_candidates_per_cluster: 2
  max_candidates_per_sector: 3
  minimum_expected_return: 0.0
  expected_return_column: optimization_expected_return
  require_minimum_expected_return: true
  insufficient_positive_policy: defensive_topk

qubo:
  risk_aversion: 0.55
  expected_return_source: validation_blend_calibrated

covariance:
  method: ewma
  span: 60
  horizon_days: 20
  minimum_coverage: 0.95

solver:
  qaoa_depth: 2
  shots: 1024
  seeds: [11, 23, 47]
  parameter_trials: 45
  optimizer: COBYLA
  solution_selection: best_observed_feasible
  declared_depth_grid: [1, 2, 3]
  declared_shots_grid: [256, 1024, 2048]

weights:
  lower: 0.05
  upper: 0.40
  risk_aversion: 1.25
  turnover_penalty: 0.01

exposure:
  mode: market_trend_volatility
  fast_days: 63
  slow_days: 126
  minimum_exposure: 0.25
  neutral_exposure: 0.60
  target_market_volatility: 0.22

constraints:
  # VND 50m is the executable base case. The larger declared scenarios remain
  # capacity stress tests and are not used to tune portfolio performance.
  portfolio_notional_vnd: 50000000
  capacity_scenarios_vnd: [50000000, 100000000, 1000000000]
  max_adv_participation: 0.05
  max_one_way_turnover: null
  sector_cap: 0.50
  board_lot: 100

backtest:
  transaction_cost_bps: 10
  commission_bps: 10
  sell_tax_bps: 10
  slippage_bps: 5
  impact_coefficient: 0.0005
  delisting_return: -1.0
  maximum_unexplained_gap_days: 5
  risk_free_annual: 0.03

benchmark:
  name: VNALLSHARETRI
  required: true

risk_free:
  mode: fixed_annual
  annual_rate: 0.03

sensitivity:
  design: preregistered_partial_factorial
  representative_folds: 3
  seeds: [11, 23]

exploratory:
  allowed_leakage_blockers:
    - price_adjustment_policy_verified
    - price_adjustment_contract_matches_dataset


In [ ]:
%%writefile /content/ai_quantum_standalone/configs/standalone_smoke.yaml
mode: exploratory
label: "STANDALONE COLAB SMOKE - REAL UPLOADED DATA"
seed: 42

data:
  source: data_17_8_hose_disclosure_enriched
  start: "2020-01-01"
  end: "2025-12-31"
  rebalance: monthly

universe:
  definition: hose_all_listed
  index_code: null
  max_assets: 300
  liquidity_lookback_days: 60
  minimum_observations: 40

walk_forward:
  train_months: 24
  validation_months: 3
  test_months: 1
  max_folds: 4
  embargo_days: 20
  selection: evenly_spaced
  continuous_monthly: false
  final_holdout_months: 0
  holdout_policy: freeze_before_single_evaluation

target:
  horizon_days: 20

model:
  kind: xgboost
  signal_mode: validation_blend
  xgboost_weight_grid: [0.25, 0.50, 0.75, 1.00]
  blend_stability_penalty: 0.25
  n_estimators: 40
  max_depth: 3
  learning_rate: 0.035
  min_feature_coverage: 0.05
  tuning_scope: development_period_only

reduction:
  qubit_budget: 8
  candidate_size: 8
  min_candidate_size: 8
  max_candidate_size: 8
  cardinality: 4
  signal_weight: 0.40
  liquidity_weight: 0.35
  risk_weight: 0.15
  correlation_penalty: 0.10
  stability_weight: 0.20
  minimum_candidate_retention: 0.50
  low_signal_dispersion_ratio: 0.0
  high_correlation_threshold: 0.65
  liquidity_floor_quantile: 0.20
  risk_ceiling_quantile: 0.90
  correlation_cluster_threshold: 0.70
  max_candidates_per_cluster: 2
  max_candidates_per_sector: 3
  minimum_expected_return: 0.0
  expected_return_column: optimization_expected_return
  require_minimum_expected_return: true
  insufficient_positive_policy: defensive_topk

qubo:
  risk_aversion: 0.55
  expected_return_source: validation_blend_calibrated

covariance:
  method: ewma
  span: 60
  horizon_days: 20
  minimum_coverage: 0.95

solver:
  qaoa_depth: 1
  shots: 256
  seeds: [11]
  parameter_trials: 24
  optimizer: COBYLA
  solution_selection: best_observed_feasible
  declared_depth_grid: [1, 2, 3]
  declared_shots_grid: [256, 1024, 2048]

weights:
  lower: 0.05
  upper: 0.40
  risk_aversion: 1.25
  turnover_penalty: 0.01

exposure:
  mode: market_trend_volatility
  fast_days: 63
  slow_days: 126
  minimum_exposure: 0.25
  neutral_exposure: 0.60
  target_market_volatility: 0.22

constraints:
  # VND 50m is the executable base case. The larger declared scenarios remain
  # capacity stress tests and are not used to tune portfolio performance.
  portfolio_notional_vnd: 50000000
  capacity_scenarios_vnd: [50000000, 100000000, 1000000000]
  max_adv_participation: 0.05
  max_one_way_turnover: null
  sector_cap: 0.50
  board_lot: 100

backtest:
  transaction_cost_bps: 10
  commission_bps: 10
  sell_tax_bps: 10
  slippage_bps: 5
  impact_coefficient: 0.0005
  delisting_return: -1.0
  maximum_unexplained_gap_days: 5
  risk_free_annual: 0.03

benchmark:
  name: VNALLSHARETRI
  required: true

risk_free:
  mode: fixed_annual
  annual_rate: 0.03

sensitivity:
  design: preregistered_partial_factorial
  representative_folds: 1
  seeds: [11]

exploratory:
  allowed_leakage_blockers:
    - price_adjustment_policy_verified
    - price_adjustment_contract_matches_dataset


In [ ]:
%%writefile /content/ai_quantum_standalone/tests/test_standalone.py
"""Validation tests embedded in the standalone Google Colab notebook."""

from __future__ import annotations

import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import pytest

from scripts.import_colab_complete_csv import bool_series, parse_dates, require_columns
from src.data_pipeline import sha256_file
from src.research import (
    build_features,
    energy,
    ewma_mean_cov,
    exact_solver,
    feasible_states,
    financial_metrics,
    holm_adjust,
    make_folds,
    optimize_weights,
    paired_block_bootstrap_test,
    penalty_qaoa_statevector,
    purged_fold_frames,
    simulated_annealing,
    transaction_cost_breakdown,
    xy_qaoa_statevector,
)


def _prices(tickers: int = 4, periods: int = 190) -> pd.DataFrame:
    dates = pd.bdate_range("2020-01-01", periods=periods)
    rows: list[dict] = []
    for j in range(tickers):
        close = 20 + j + np.cumsum(0.02 + 0.1 * np.sin(np.arange(periods) / 11 + j))
        for i, date in enumerate(dates):
            rows.append({
                "ticker": f"T{j}", "date": date, "open": close[i] * 0.998,
                "high": close[i] * 1.01, "low": close[i] * 0.99,
                "close": close[i], "adjusted_close": close[i],
                "volume": 1_000_000 + 1000 * i,
                "trading_value": close[i] * (1_000_000 + 1000 * i),
                "available_at": date + pd.Timedelta(hours=8),
            })
    return pd.DataFrame(rows)


def test_01_sha256_file(tmp_path: Path) -> None:
    path = tmp_path / "sample.bin"
    path.write_bytes(b"standalone-colab")
    assert sha256_file(path) == hashlib.sha256(b"standalone-colab").hexdigest()


def test_02_schema_rejects_missing_columns() -> None:
    with pytest.raises(ValueError, match="missing required columns"):
        require_columns(pd.DataFrame({"ticker": ["AAA"]}), ["ticker", "date"], "PRICE")


def test_03_parse_dates_is_explicit() -> None:
    result = parse_dates(pd.DataFrame({"date": ["2025-01-02"]}), ["date"])
    assert pd.api.types.is_datetime64_any_dtype(result["date"])


def test_04_boolean_parser_is_conservative() -> None:
    parsed = bool_series(pd.Series(["true", "1", "yes", "false", "0", ""])).tolist()
    assert parsed == [True, True, True, False, False, False]


def test_05_price_fixture_has_valid_ohlc() -> None:
    frame = _prices()
    assert (frame.high >= frame[["open", "close"]].max(axis=1)).all()
    assert (frame.low <= frame[["open", "close"]].min(axis=1)).all()
    assert (frame.volume >= 0).all()


def test_06_feature_builder_is_point_in_time() -> None:
    features = build_features(_prices(), target_horizon_days=20)
    assert (features["feature_available_at"] >= features["date"]).all()
    assert "target_return_20d" in features
    assert "label_end_time" in features


def test_07_walk_forward_ordering() -> None:
    folds = make_folds(pd.Series(pd.bdate_range("2020-01-01", "2025-12-31")), 24, 3, 1, 4)
    assert len(folds) == 4
    assert all(f["train_start"] < f["train_end"] <= f["validation_start"] for f in folds)
    assert all(f["validation_start"] < f["validation_end"] <= f["test_start"] for f in folds)


def test_08_purge_prevents_label_overlap() -> None:
    features = build_features(_prices(periods=500), target_horizon_days=20)
    fold = make_folds(features.date, 12, 2, 1, 1, embargo_days=5)[0]
    train, validation, _, _ = purged_fold_frames(features, fold)
    assert train["label_end_time"].max() < fold["validation_start"] - pd.Timedelta(days=5)
    assert validation["label_end_time"].max() < fold["test_start"] - pd.Timedelta(days=5)


def test_09_feasible_states_preserve_cardinality() -> None:
    states = feasible_states(8, 4)
    assert states.shape == (70, 8)
    assert np.all(states.sum(axis=1) == 4)


def test_10_qubo_energy_matches_definition() -> None:
    q = np.array([[1.0, -0.25], [-0.25, 0.5]])
    bits = np.array([1, 1])
    assert energy(bits, q) == pytest.approx(float(bits @ q @ bits))


def test_11_exact_solver_is_reference_optimum() -> None:
    q = np.diag([0.4, -0.5, 0.1, -0.2])
    result = exact_solver(q, 2)
    all_energies = [energy(s, q) for s in feasible_states(4, 2)]
    assert result["energy"] == pytest.approx(min(all_energies))
    assert result["bits"].sum() == 2


def test_12_simulated_annealing_keeps_cardinality() -> None:
    result = simulated_annealing(np.eye(6), 3, seed=42, steps=50)
    assert result["bits"].sum() == 3
    assert result["feasibility_rate"] == 1.0


def test_13_xy_qaoa_preserves_feasible_subspace() -> None:
    q = np.diag(np.linspace(-0.4, 0.3, 6))
    result = xy_qaoa_statevector(q, 3, p=1, trials=8, shots=128, seed=11)
    assert result["bits"].sum() == 3
    assert result["feasibility_rate"] == 1.0
    assert result["backend"] == "internal_ideal_statevector_fixed_weight"


def test_14_xy_qaoa_is_seed_reproducible() -> None:
    q = np.diag(np.linspace(-0.4, 0.3, 5))
    first = xy_qaoa_statevector(q, 2, p=1, trials=8, shots=64, seed=23)
    second = xy_qaoa_statevector(q, 2, p=1, trials=8, shots=64, seed=23)
    assert np.array_equal(first["bits"], second["bits"])
    assert first["energy"] == pytest.approx(second["energy"])


def test_15_penalty_qaoa_reports_measured_feasibility() -> None:
    q = np.diag(np.linspace(-0.2, 0.2, 5))
    result = penalty_qaoa_statevector(q, 2, p=1, trials=8, shots=128, seed=47)
    assert 0.0 <= result["feasibility_rate"] <= 1.0
    assert result["backend"] == "internal_ideal_statevector_full_hilbert"


def test_16_ewma_covariance_is_symmetric_psd() -> None:
    rng = np.random.default_rng(42)
    returns = pd.DataFrame(rng.normal(0, 0.01, size=(120, 4)), columns=list("ABCD"))
    mean, cov = ewma_mean_cov(returns, span=30, horizon=20)
    assert len(mean) == 4
    assert np.allclose(cov, cov.T)
    assert np.linalg.eigvalsh(cov).min() >= -1e-9


def test_17_weight_optimizer_enforces_long_only_budget() -> None:
    mu = np.array([0.02, 0.01, 0.015, 0.005])
    cov = np.eye(4) * 0.02
    weights = optimize_weights(mu, cov, 0.05, 0.5, 1.0, None, 0.01)
    assert weights.sum() == pytest.approx(1.0, abs=1e-7)
    assert np.all(weights >= 0.05 - 1e-8)
    assert np.all(weights <= 0.5 + 1e-8)


def test_18_transaction_cost_reduces_net_return() -> None:
    total, details = transaction_cost_breakdown(
        {"AAA": 0.6, "BBB": -0.4}, commission_bps=10.0,
        sell_tax_bps=10.0, slippage_bps=5.0, impact_coefficient=0.0005,
        adv_capacity_weights={"AAA": 5.0, "BBB": 4.0},
    )
    assert total > 0
    assert details["BBB"]["sell_tax_cost"] > 0


def test_19_holm_adjustment_controls_familywise_error() -> None:
    adjusted = holm_adjust([0.01, 0.04, 0.20])
    assert adjusted[0] == pytest.approx(0.03)
    assert all(0 <= value <= 1 for value in adjusted)


def test_20_block_bootstrap_returns_valid_statistics() -> None:
    index = pd.bdate_range("2024-01-01", periods=80)
    a = pd.Series(np.linspace(-0.01, 0.02, 80), index=index)
    b = pd.Series(np.linspace(-0.012, 0.015, 80), index=index)
    result = paired_block_bootstrap_test(a, b, seed=42, samples=50, block=5)
    assert result["ci_low"] <= result["ci_high"]
    assert 0 <= result["p_value"] <= 1


def test_21_financial_metrics_include_drawdown() -> None:
    returns = pd.Series([0.01, -0.02, 0.015, -0.005] * 20)
    metrics = financial_metrics(returns, 0.03)
    assert metrics["observations"] == 80
    assert metrics["max_drawdown"] <= 0


## 3. Cài thư viện trong môi trường cô lập tương thích Colab Python 3.12

In [ ]:
if VENV_DIR.exists():
    shutil.rmtree(VENV_DIR)
# Colab may omit ensurepip/python3-venv. PyPA virtualenv supplies isolated seed wheels.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "virtualenv==20.35.4"], check=True)
subprocess.run([sys.executable, "-m", "virtualenv", str(VENV_DIR)], check=True)
ENV_PYTHON = VENV_DIR / "bin" / "python"
PIPELINE_ENV = os.environ.copy()
PIPELINE_ENV["MPLBACKEND"] = "Agg"
subprocess.run([str(ENV_PYTHON), "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run([str(ENV_PYTHON), "-m", "pip", "install", "-e", f"{STANDALONE_ROOT}[dev]"], check=True)
subprocess.run([str(ENV_PYTHON), "-m", "pip", "check"], check=True)
health = "import sys,numpy,pandas,pyarrow,scipy,sklearn,xgboost,matplotlib,yaml; print({'python':sys.version,'executable':sys.executable,'numpy':numpy.__version__,'pandas':pandas.__version__,'pyarrow':pyarrow.__version__,'scipy':scipy.__version__,'sklearn':sklearn.__version__,'xgboost':xgboost.__version__,'matplotlib':matplotlib.__version__,'backend':matplotlib.get_backend()})"
subprocess.run([str(ENV_PYTHON), "-c", health], cwd=STANDALONE_ROOT, env=PIPELINE_ENV, check=True)
print("Isolated environment ready:", ENV_PYTHON)

## 4. Upload dữ liệu

Chọn đúng một file `ai_quantum_complete_dataset.csv` hoặc `ai_quantum_complete_dataset.zip`. Nếu dùng một dataset mới hợp lệ, đặt `EXPECTED_CSV_SHA256 = ""` hoặc thay bằng hash đã kiểm toán của file đó.

In [ ]:
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Hãy upload đúng một file CSV hoặc ZIP.")
UPLOAD_PATH = Path("/content") / next(iter(uploaded))
print("Uploaded:", UPLOAD_PATH, f"{UPLOAD_PATH.stat().st_size / 1e6:.2f} MB")

## 5. Giải nén an toàn, tính SHA-256 và kiểm tra hợp đồng file

In [ ]:
def sha256_stream(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

if UPLOAD_PATH.suffix.lower() == ".zip":
    extract_dir = Path("/content/standalone_uploaded_csv")
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir()
    with zipfile.ZipFile(UPLOAD_PATH) as archive:
        corrupt = archive.testzip()
        if corrupt:
            raise ValueError(f"ZIP bị lỗi tại: {corrupt}")
        root = extract_dir.resolve()
        for member in archive.infolist():
            target = (extract_dir / member.filename).resolve()
            if target != root and root not in target.parents:
                raise ValueError(f"ZIP chứa đường dẫn không an toàn: {member.filename}")
        archive.extractall(extract_dir)
    candidates = list(extract_dir.rglob("*.csv"))
    if len(candidates) != 1:
        raise ValueError(f"ZIP phải chứa đúng một CSV; tìm thấy {len(candidates)}")
    CSV_PATH = candidates[0]
elif UPLOAD_PATH.suffix.lower() == ".csv":
    CSV_PATH = UPLOAD_PATH
else:
    raise ValueError("Chỉ chấp nhận CSV hoặc ZIP.")

CSV_SHA256 = sha256_stream(CSV_PATH)
if EXPECTED_CSV_SHA256 and CSV_SHA256.lower() != EXPECTED_CSV_SHA256.lower():
    raise ValueError(f"SHA-256 không khớp: {CSV_SHA256}")
print("CSV:", CSV_PATH)
print("SHA-256:", CSV_SHA256)

## 6. Import, schema validation, data-quality audit và leakage audit

In [ ]:
import_command = [
    str(ENV_PYTHON), "scripts/import_colab_complete_csv.py", str(CSV_PATH),
    "--workspace", str(WORKSPACE),
]
subprocess.run(import_command, cwd=STANDALONE_ROOT, env=PIPELINE_ENV, check=True)
IMPORT_REPORT = json.loads((WORKSPACE / "outputs/reports/COLAB_CSV_IMPORT_REPORT.json").read_text(encoding="utf-8"))
assert IMPORT_REPORT["input_csv_sha256"] == CSV_SHA256
assert IMPORT_REPORT["quality_status"] == "pass"
assert IMPORT_REPORT["leakage_status"] in {"pass", "pass_with_limitations"}
assert IMPORT_REPORT["exploratory_run_permitted"]
print(json.dumps(IMPORT_REPORT, indent=2, ensure_ascii=False))

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, Image, display

normalized = WORKSPACE / "outputs/normalized"
prices = pd.read_parquet(normalized / "prices.parquet")
benchmark = pd.read_parquet(normalized / "benchmark.parquet")
master = pd.read_parquet(normalized / "security_master_full.parquet")
actions = pd.read_parquet(normalized / "corporate_actions.parquet")
quality = json.loads((WORKSPACE / "outputs/reports/data_quality.json").read_text(encoding="utf-8"))
leakage = json.loads((WORKSPACE / "outputs/reports/leakage_audit.json").read_text(encoding="utf-8"))
data_summary = pd.DataFrame({
    "Chỉ tiêu": ["SHA-256", "PRICE rows", "Runtime tickers", "Security master", "Start", "End", "Benchmark rows", "Corporate actions", "Data quality", "Leakage audit", "Research classification"],
    "Kết quả": [CSV_SHA256, len(prices), prices.ticker.nunique(), master.ticker.nunique(), str(prices.date.min().date()), str(prices.date.max().date()), len(benchmark), len(actions), quality["status"], leakage["status"], "EXPLORATORY"],
})
display(data_summary)
display(prices.head())
display(Markdown("**Phạm vi diễn giải:** runtime đủ cho nghiên cứu exploratory; không tự động nâng cấp thành confirmatory full-HOSE nếu thiếu PIT financial statements, disclosure binaries hoặc lịch sử membership đầy đủ."))

## 7. Chạy validation tests trước thực nghiệm

In [ ]:
test_result = subprocess.run(
    [str(ENV_PYTHON), "-m", "pytest", "-q"], cwd=STANDALONE_ROOT,
    env=PIPELINE_ENV, text=True, capture_output=True,
)
print(test_result.stdout)
if test_result.stderr:
    print(test_result.stderr)
test_result.check_returncode()
print("Validation gate passed.")

## 8. Chạy pipeline SMOKE hoặc FULL trên chính dữ liệu upload

In [ ]:
CONFIG_PATH = STANDALONE_ROOT / "configs" / (
    "standalone_smoke.yaml" if EXECUTION_PROFILE == "SMOKE" else "standalone_full.yaml"
)
experiments = WORKSPACE / "outputs/experiments"
experiments.mkdir(parents=True, exist_ok=True)
before = {path.resolve() for path in experiments.iterdir() if path.is_dir()}
runner = (
    "from pathlib import Path; from src.research import run_experiment; "
    f"print(run_experiment(Path(r'{WORKSPACE}'), Path(r'{CONFIG_PATH}')))"
)
subprocess.run([str(ENV_PYTHON), "-c", runner], cwd=STANDALONE_ROOT, env=PIPELINE_ENV, check=True)
after = {path.resolve() for path in experiments.iterdir() if path.is_dir()}
created = sorted(after - before, key=lambda path: path.stat().st_mtime)
if not created:
    raise RuntimeError("Pipeline không tạo experiment mới.")
ACTIVE = created[-1]
manifest = json.loads((ACTIVE / "manifest.json").read_text(encoding="utf-8"))
if manifest.get("status") != "success":
    raise RuntimeError(f"Experiment không thành công: {manifest}")
expected_folds = 4 if EXECUTION_PROFILE == "SMOKE" else manifest["folds_requested"]
assert manifest["folds_completed"] == expected_folds
print("ACTIVE:", ACTIVE)
print(json.dumps({key: manifest.get(key) for key in ["status", "experiment_id", "folds_requested", "folds_completed", "actual_oos_start", "actual_oos_end", "data_class"]}, indent=2, ensure_ascii=False))

## 9. Chuẩn hóa artifact, tạo biểu đồ bổ sung và báo cáo tiếng Việt

In [ ]:
import hashlib
import platform

aliases = {
    "data_quality.json": "data_quality_report.json",
    "fold_manifest.csv": "folds.csv",
    "feature_coverage_by_fold.csv": "features_summary.csv",
    "selected_universe.csv": "adaptive_universe.csv",
    "optimization_instances.json": "qubo_instances.json",
}
for source, target in aliases.items():
    source_path = ACTIVE / source
    if source_path.exists():
        shutil.copy2(source_path, ACTIVE / target)

environment = {
    "kernel_python": sys.version,
    "pipeline_python": str(ENV_PYTHON),
    "platform": platform.platform(),
    "execution_profile": EXECUTION_PROFILE,
    "mplbackend": PIPELINE_ENV["MPLBACKEND"],
}
(ACTIVE / "environment.json").write_text(json.dumps(environment, indent=2, ensure_ascii=False), encoding="utf-8")
(ACTIVE / "dataset_hash.json").write_text(json.dumps({"csv_sha256": CSV_SHA256, "csv_path": str(CSV_PATH)}, indent=2), encoding="utf-8")
config_freeze_path = ACTIVE / "config_freeze.json"
if not config_freeze_path.exists():
    config_freeze_path.write_text(json.dumps({
        "config_sha256": hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest(),
        "execution_profile": EXECUTION_PROFILE,
        "folds_completed": manifest["folds_completed"],
        "notebook_config": NOTEBOOK_CONFIG,
        "freeze_scope": "smoke_execution_config_without_final_holdout",
        "policy": "all_parameters_fixed_before_the_smoke_run; not_a_confirmatory_holdout_freeze",
    }, indent=2, ensure_ascii=False), encoding="utf-8")

rankings = pd.read_csv(ACTIVE / "rankings.csv")
comparisons = pd.read_csv(ACTIVE / "comparisons.csv")
metrics = pd.read_csv(ACTIVE / "strategy_metrics_summary.csv")
tests = pd.read_csv(ACTIVE / "statistical_tests.csv")
ablations = pd.read_csv(ACTIVE / "ablation_results.csv")
sensitivity = pd.read_csv(ACTIVE / "sensitivity_results.csv")
latest = pd.read_csv(ACTIVE / "latest_selected_portfolio.csv")
latest_summary = json.loads((ACTIVE / "latest_portfolio_summary.json").read_text(encoding="utf-8"))
constraints = pd.read_csv(ACTIVE / "constraint_diagnostics.csv")
weights = pd.read_csv(ACTIVE / "weights.csv")
exposure = pd.read_csv(ACTIVE / "exposure_by_fold.csv") if (ACTIVE / "exposure_by_fold.csv").exists() else pd.DataFrame()

figures = ACTIVE / "figures"
figures.mkdir(exist_ok=True)
import matplotlib.pyplot as plt
plt.switch_backend("Agg")
full_weights = weights[weights.strategy.eq("full_pipeline_xy_qaoa")].copy()
if not full_weights.empty:
    pivot = full_weights.pivot_table(index="decision_time", columns="ticker", values="weight", aggfunc="sum", fill_value=0)
    top = pivot.mean().nlargest(min(10, len(pivot.columns))).index
    ax = pivot[top].plot.area(figsize=(12, 6), colormap="tab20")
    ax.set(title="Tỷ trọng các tài sản chính theo thời gian", xlabel="Ngày tái cân bằng", ylabel="Tỷ trọng")
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
    plt.tight_layout(); plt.savefig(figures / "weights_over_time.png", dpi=160); plt.close()
if not exposure.empty and "cash_weight" in exposure:
    ax = exposure.plot(x="decision_time", y="cash_weight", figsize=(11, 4), legend=False)
    ax.set(title="Tỷ trọng tiền mặt theo fold", xlabel="Ngày tái cân bằng", ylabel="Cash weight")
    plt.tight_layout(); plt.savefig(figures / "cash_exposure.png", dpi=160); plt.close()
if "sector" in latest and latest["sector"].fillna("").str.len().gt(0).any():
    sector = latest.assign(sector=latest.sector.fillna("Unknown")).groupby("sector").target_weight.sum().sort_values()
    ax = sector.plot.barh(figsize=(9, 5)); ax.set(title="Phân bổ ngành của danh mục cuối", xlabel="Target weight", ylabel="Ngành")
    plt.tight_layout(); plt.savefig(figures / "sector_allocation.png", dpi=160); plt.close()

def row(test_name: str):
    found = tests.loc[tests.test.eq(test_name)]
    return found.iloc[0] if len(found) else None

h1 = row("xgboost_rank_ic_vs_ewma_rank_ic")
h2_return = row("adaptive_universe_forward_return_vs_fixed_topm")
h2_div = row("adaptive_universe_diversification_vs_fixed_topm")
h3 = row("xy_feasibility_vs_penalty_qaoa")
h4 = row("xy_optimality_gap_vs_penalty_qaoa")
h5 = tests.loc[tests.hypothesis.eq("H5")]
alpha = NOTEBOOK_CONFIG["significance_level"]

def significant(test_row) -> bool:
    return test_row is not None and pd.notna(test_row.p_value_holm) and float(test_row.p_value_holm) < alpha

hypothesis_text = []
if h1 is not None:
    h1_label = "được hỗ trợ" if significant(h1) and h1.mean_difference > 0 else "không được hỗ trợ"
    hypothesis_text.append(f"### H1 — {h1_label}\nChênh lệch Rank IC XGBoost–EWMA là {h1.mean_difference:.4f}, CI 95% [{h1.ci_low:.4f}, {h1.ci_high:.4f}], p-Holm={h1.p_value_holm:.4f}. Kết luận chỉ dựa trên dự báo ngoài mẫu của run hiện tại.")
h2_count = int(significant(h2_return)) + int(significant(h2_div))
h2_label = "được hỗ trợ đầy đủ" if h2_count == 2 else "được hỗ trợ một phần" if h2_count == 1 else "không được hỗ trợ"
hypothesis_text.append(f"### H2 — {h2_label}\nAUR được đánh giá đồng thời về forward return và đa dạng hóa; p-Holm tương ứng là {getattr(h2_return, 'p_value_holm', np.nan):.4f} và {getattr(h2_div, 'p_value_holm', np.nan):.4f}.")
if h3 is not None:
    h3_label = "được hỗ trợ trong ideal simulator" if significant(h3) and h3.mean_difference > 0 else "không được hỗ trợ"
    hypothesis_text.append(f"### H3 — {h3_label}\nChênh lệch feasibility rate là {h3.mean_difference:.4f}, p-Holm={h3.p_value_holm:.4f}. Kết quả không được ngoại suy thành quantum advantage trên phần cứng thật.")
if h4 is not None:
    h4_label = "được hỗ trợ có điều kiện so với Penalty-QAOA" if significant(h4) and h4.mean_difference > 0 else "không được hỗ trợ"
    hypothesis_text.append(f"### H4 — {h4_label}\nCải thiện optimality gap so với Penalty-QAOA là {h4.mean_difference:.4f}, p-Holm={h4.p_value_holm:.4f}; Exact vẫn là nghiệm tham chiếu và Simulated Annealing vẫn phải được đọc riêng.")
h5_sig = int((h5.p_value_holm < alpha).sum()) if len(h5) else 0
h5_label = "được hỗ trợ" if h5_sig and (h5.loc[h5.p_value_holm < alpha, "mean_difference"] > 0).all() else "không được hỗ trợ"
hypothesis_text.append(f"### H5 — {h5_label}\nCó {h5_sig}/{len(h5)} so sánh hiệu quả tài chính đạt ý nghĩa sau hiệu chỉnh Holm. Lợi nhuận dương, nếu xuất hiện, không tự thân chứng minh alpha thống kê.")
hypothesis_text.append(f"### H6 — phân tích độ nhạy đã hoàn thành\nCó {len(sensitivity)} quan sát sensitivity trong lưới cấu hình đã khai báo. Kết luận chỉ có giá trị trong lưới depth, shots, seed, cardinality, noise và chi phí được chạy.")
HYPOTHESIS_MARKDOWN = "\n\n".join(hypothesis_text)

pipeline_metrics = metrics.loc[metrics.strategy.eq("full_pipeline_xy_qaoa")]
pipeline_line = "Không có metric full pipeline."
if len(pipeline_metrics):
    value = pipeline_metrics.iloc[0]
    pipeline_line = f"Full pipeline đạt cumulative return {value.cumulative_return:.2%}, annualized return {value.annualized_return:.2%}, Sharpe {value.sharpe:.3f} và maximum drawdown {value.max_drawdown:.2%} sau chi phí trong cửa sổ ngoài mẫu."
report_vi = f"""# BÁO CÁO KẾT QUẢ HỆ THỐNG AI–QUANTUM STANDALONE

## 1. Tóm tắt hệ thống
Hệ thống kết hợp XGBoost, EWMA đa biến, Adaptive Universe Reduction, cardinality-constrained QUBO, Exact Solver, Simulated Annealing, Penalty-QAOA, feasible-subspace XY-QAOA và tối ưu tỷ trọng cổ điển trong thiết kế walk-forward point-in-time.

## 2. Dữ liệu và kiểm toán
Run sử dụng {len(prices):,} quan sát giá của {prices.ticker.nunique()} mã, giai đoạn {prices.date.min().date()}–{prices.date.max().date()}. Data quality là `{quality['status']}`, leakage audit là `{leakage['status']}` và phân loại nghiên cứu là exploratory.

## 3. Thiết kế thực nghiệm
Profile `{EXECUTION_PROFILE}` hoàn thành {manifest['folds_completed']}/{manifest['folds_requested']} folds. Mọi feature, scaler, mô hình và covariance được ước lượng trong training/validation window trước khi đánh giá test window.

## 4. Kết quả tín hiệu, AUR và solver
XGBoost được đánh giá bằng Rank IC ngoài mẫu; EWMA là đối chứng và bộ ước lượng covariance. AUR kết hợp tín hiệu, thanh khoản, rủi ro và tương quan để tạo tập 8 ứng viên. Solver comparison được lưu trong `comparisons.csv`; không sử dụng runtime simulator để tuyên bố quantum speedup.

## 5. Hiệu quả danh mục
{pipeline_line}

## 6. Kết luận giả thuyết
{HYPOTHESIS_MARKDOWN}

## 7. Rổ cuối và giới hạn
Rổ cuối tại {latest.decision_time.iloc[0]} gồm {', '.join(latest.ticker.astype(str))}. Đây là output nghiên cứu lịch sử, không phải khuyến nghị đầu tư hiện tại. Hạn chế chính gồm phân loại exploratory, simulator lý tưởng, dữ liệu ngành còn thiếu ở một số mã và chưa có bằng chứng quantum advantage.

## 8. Hướng nghiên cứu tiếp theo
Cần bổ sung PIT financial statements, lịch sử membership HOSE, kiểm chứng corporate actions đa nguồn và thử nghiệm noise/hardware trước khi đưa ra kết luận confirmatory.
"""
(ACTIVE / "research_report_vi.md").write_text(report_vi, encoding="utf-8")
print("Artifact normalization and Vietnamese report completed.")

## 10. Kết quả dữ liệu, mô hình, solver, danh mục và giả thuyết

In [ ]:
display(Markdown(f"### Experiment `{manifest['experiment_id']}` — {manifest['folds_completed']}/{manifest['folds_requested']} folds"))
display(data_summary)
coverage_path = WORKSPACE / "outputs/reports/coverage_report.csv"
if coverage_path.exists():
    display(pd.read_csv(coverage_path).head(20))
fold_rank = rankings.groupby("fold")[["xgboost_rank_ic", "ewma_rank_ic"]].first()
display(pd.DataFrame({
    "Model": ["XGBoost", "EWMA"],
    "Mean Rank IC": [fold_rank.xgboost_rank_ic.mean(), fold_rank.ewma_rank_ic.mean()],
    "Median Rank IC": [fold_rank.xgboost_rank_ic.median(), fold_rank.ewma_rank_ic.median()],
    "IC hit rate": [(fold_rank.xgboost_rank_ic > 0).mean(), (fold_rank.ewma_rank_ic > 0).mean()],
}))
aur = pd.read_csv(ACTIVE / "aur_diagnostics.csv")
display(aur.head(20))
display(comparisons)
display(metrics.sort_values("sharpe", ascending=False))
display(ablations)
display(sensitivity.head(40))
display(tests)
display(latest)
display(constraints.tail(1).T.rename(columns={constraints.tail(1).index[0]: "Fold cuối"}))
display(Markdown("**Rổ cổ phiếu cuối là output lịch sử của backtest, không phải khuyến nghị đầu tư hiện tại.**"))

In [ ]:
for figure_name in [
    "equity_curve.png", "drawdown.png", "risk_return.png", "rank_ic_by_fold.png",
    "feasibility_rate.png", "optimality_gap.png", "turnover_and_cost.png",
    "sensitivity_analysis.png", "weights_over_time.png", "cash_exposure.png",
    "sector_allocation.png",
]:
    figure_path = ACTIVE / "figures" / figure_name
    if figure_path.exists():
        display(Markdown(f"### {figure_name.replace('_', ' ').replace('.png', '').title()}"))
        display(Image(filename=str(figure_path)))

In [ ]:
display(Markdown("# Kết luận kiểm định giả thuyết"))
display(Markdown(HYPOTHESIS_MARKDOWN))
display(Markdown((ACTIVE / "research_report_vi.md").read_text(encoding="utf-8")))

## 11. Audit artifact và tải kết quả

In [ ]:
required_artifacts = [
    "config_freeze.json", "manifest.json", "environment.json", "dataset_hash.json",
    "data_quality_report.json", "leakage_audit.json", "folds.csv", "features_summary.csv",
    "rankings.csv", "adaptive_universe.csv", "qubo_instances.json", "solver_runs.csv",
    "comparisons.csv", "weights.csv", "trades.csv", "portfolio_returns.csv",
    "strategy_metrics_summary.csv", "statistical_tests.csv", "ablation_results.csv",
    "sensitivity_results.csv", "constraint_diagnostics.csv", "latest_selected_portfolio.csv",
    "latest_portfolio_summary.json", "research_report_vi.md",
]
missing = [name for name in required_artifacts if not (ACTIVE / name).exists()]
if missing:
    raise FileNotFoundError(f"Thiếu artifact bắt buộc: {missing}")
artifact_count = len([path for path in ACTIVE.rglob("*") if path.is_file()])
result_target = RESULTS_ROOT / manifest["experiment_id"]
if result_target.exists():
    shutil.rmtree(result_target)
shutil.copytree(ACTIVE, result_target)
archive = shutil.make_archive(str(RESULTS_ROOT / f"ai_quantum_{manifest['experiment_id']}"), "zip", root_dir=result_target)
print({"status": manifest["status"], "folds": manifest["folds_completed"], "artifacts": artifact_count, "zip": archive, "size_mb": round(Path(archive).stat().st_size / 1e6, 2)})
from google.colab import files
display(Markdown(f"Chạy `files.download(r'{archive}')` để tải toàn bộ kết quả."))